<a href="https://colab.research.google.com/github/Mrsayem/MVSegnet-Paper/blob/main/MVSegNetfixed_final_modified.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import zipfile
import requests
import shutil
from pathlib import Path


def download_dataset():
    base_dir      = Path("fetal_vm_segmentation")
    data_dir      = base_dir / "data"
    raw_dir       = data_dir / "raw"
    processed_dir = data_dir / "processed"

    for dir_path in [base_dir, data_dir, raw_dir, processed_dir]:
        dir_path.mkdir(exist_ok=True, parents=True)

    print("✓ Directory structure created")

    zenodo_record = "8265464"
    zenodo_url    = f"https://zenodo.org/records/{zenodo_record}"

    files_to_download = {
        "Trans-ventricular.zip": f"https://zenodo.org/records/{zenodo_record}/files/Trans-ventricular.zip?download=1",
        "Readme.txt":            f"https://zenodo.org/records/{zenodo_record}/files/Readme.txt?download=1"
    }

    print(f"\n📥 Downloading Lateral Ventricle Dataset from Zenodo...")
    print(f"Dataset: {zenodo_url}")

    for filename, url in files_to_download.items():
        output_path = raw_dir / filename

        if output_path.exists():
            print(f"✓ {filename} already exists, skipping...")
            continue

        print(f"Downloading {filename}...")
        try:
            response   = requests.get(url, stream=True)
            total_size = int(response.headers.get('content-length', 0))

            with open(output_path, 'wb') as f:
                downloaded = 0
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        if total_size > 0:
                            percent = (downloaded / total_size) * 100
                            print(f"\r  Progress: {percent:.1f}%", end='')

            print(f"\n✓ Downloaded {filename}")

        except Exception as e:
            print(f"\n⚠️  Auto-download failed: {e}")
            print(f"\nManual download instructions:")
            print(f"1. Visit: {zenodo_url}")
            print(f"2. Download: {filename}")
            print(f"3. Save to: {raw_dir}")
            return raw_dir, processed_dir, False

    zip_path = raw_dir / "Trans-ventricular.zip"
    if zip_path.exists():
        print(f"\n📦 Extracting {zip_path.name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(raw_dir)
        print("✓ Extraction complete")

    return raw_dir, processed_dir, True


def organize_lv_data(raw_dir, processed_dir):
    print("\n📁 Organizing LV data...")

    images_dir = processed_dir / "images"
    masks_dir  = processed_dir / "masks"
    images_dir.mkdir(exist_ok=True)
    masks_dir.mkdir(exist_ok=True)

    trans_vent_dir = raw_dir / "Trans-ventricular"

    if trans_vent_dir.exists():
        print(f"✓ Found Trans-ventricular directory")

        possible_image_dirs = [
            trans_vent_dir / "images",
            trans_vent_dir / "JPEGImages",
            trans_vent_dir / "img",
            trans_vent_dir
        ]
        possible_mask_dirs = [
            trans_vent_dir / "masks",
            trans_vent_dir / "SegmentationClass",
            trans_vent_dir / "labels",
            trans_vent_dir / "annotations"
        ]

        image_source = next(
            (d for d in possible_image_dirs
             if d.exists() and (list(d.glob("*.png")) or list(d.glob("*.jpg")))),
            None
        )
        mask_source = next(
            (d for d in possible_mask_dirs
             if d.exists() and list(d.glob("*.png"))),
            None
        )

        if image_source and mask_source:
            for img in image_source.glob("*.png"):
                shutil.copy(img, images_dir)
            for img in image_source.glob("*.jpg"):
                shutil.copy(img, images_dir)
            for mask in mask_source.glob("*.png"):
                shutil.copy(mask, masks_dir)
            print(f"✓ Copied images from: {image_source}")
            print(f"✓ Copied masks from:  {mask_source}")
        else:
            print("⚠️  Could not auto-organize images/masks.")
        csv_filename = "Trans-ventricular-Pixel-Size.csv"
        csv_dest     = processed_dir / "pixel_size.csv"

        csv_source = trans_vent_dir / csv_filename
        if csv_source.exists():
            shutil.copy(csv_source, csv_dest)
            print(f"✓ Copied pixel size CSV → {csv_dest}")
        else:
            found_csvs = list(trans_vent_dir.rglob(csv_filename))
            if found_csvs:
                shutil.copy(found_csvs[0], csv_dest)
                print(f"✓ Found and copied CSV from: {found_csvs[0]}")
            else:
                print(f"❌ CSV not found: {csv_filename}")
                print(f"   Searched in: {trans_vent_dir}")
                print(f"   Please manually copy it to: {csv_dest}")
    print(f"\n✓ Output structure:")
    print(f"  Images:    {images_dir}")
    print(f"  Masks:     {masks_dir}")
    print(f"  CSV:       {processed_dir / 'pixel_size.csv'}")

    return images_dir, masks_dir


def verify_dataset(images_dir, masks_dir, processed_dir):
    print("\n🔍 Verifying dataset...")

    num_images = len(list(images_dir.glob("*.png"))) + len(list(images_dir.glob("*.jpg")))
    num_masks  = len(list(masks_dir.glob("*.png")))
    csv_exists = (processed_dir / "pixel_size.csv").exists()

    print(f"  Images:         {num_images}")
    print(f"  Masks:          {num_masks}")
    print(f"  pixel_size.csv: {'✅ Found' if csv_exists else '❌ Missing'}")

    if num_images > 0 and num_masks > 0:
        print("✓ Dataset verified successfully!")
        return True
    else:
        print("⚠️  No images/masks found.")
        return False


if __name__ == "__main__":
    print("=" * 70)
    print("FETAL LATERAL VENTRICLE SEGMENTATION - Dataset Setup")
    print("Dataset: Zenodo 8265464 - Trans-ventricular subset")
    print("=" * 70)

    raw_dir, processed_dir, success = download_dataset()

    if success:
        images_dir, masks_dir = organize_lv_data(raw_dir, processed_dir)
        dataset_ready = verify_dataset(images_dir, masks_dir, processed_dir)

        if dataset_ready:
            print("\n" + "=" * 70)
            print("✓ Setup complete! Next steps:")
            print("1. Run: python organize_dataset.py")
            print("2. Run: python preprocess_data.py")
            print("=" * 70)
        else:
            print("\n⚠️  Manual intervention required")
            print("Check the Trans-ventricular folder structure")

FETAL LATERAL VENTRICLE SEGMENTATION - Dataset Setup
Dataset: Zenodo 8265464 - Trans-ventricular subset
✓ Directory structure created

📥 Downloading Lateral Ventricle Dataset from Zenodo...
Dataset: https://zenodo.org/records/8265464
  Progress: 100.0%
✓ Downloaded Trans-ventricular.zip

✓ Downloaded Readme.txt

📦 Extracting Trans-ventricular.zip...
✓ Extraction complete

📁 Organizing LV data...
✓ Found Trans-ventricular directory
⚠️  Could not auto-organize images/masks.
✓ Copied pixel size CSV → fetal_vm_segmentation/data/processed/pixel_size.csv

✓ Output structure:
  Images:    fetal_vm_segmentation/data/processed/images
  Masks:     fetal_vm_segmentation/data/processed/masks
  CSV:       fetal_vm_segmentation/data/processed/pixel_size.csv

🔍 Verifying dataset...
  Images:         0
  Masks:          0
  pixel_size.csv: ✅ Found
⚠️  No images/masks found.

⚠️  Manual intervention required
Check the Trans-ventricular folder structure


In [ ]:

import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm


def organize_dataset():
    base_dir      = Path("fetal_vm_segmentation")
    raw_dir       = base_dir / "data" / "raw"
    processed_dir = base_dir / "data" / "processed"

    images_out = processed_dir / "images"
    masks_out  = processed_dir / "masks"
    images_out.mkdir(parents=True, exist_ok=True)
    masks_out.mkdir(parents=True, exist_ok=True)

    print("=" * 70)
    print("DATASET ORGANIZATION - FIXED VERSION")
    print("=" * 70)

    segmask_zip = raw_dir / "Trans-ventricular" / "Trans-ventricular-segmentation-mask.zip"

    if not segmask_zip.exists():
        print(f"Segmentation mask ZIP not found: {segmask_zip}")
        return

    print(f"\n📦 Extracting segmentation masks...")
    extract_dir = raw_dir / "Trans-ventricular" / "segmentation-mask-extracted"
    extract_dir.mkdir(exist_ok=True)

    try:
        with zipfile.ZipFile(segmask_zip, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"✓ Extracted to: {extract_dir}")
    except Exception as e:
        print(f"Extraction failed: {e}")
        return
    trans_vent_inner = raw_dir / "Trans-ventricular" / "Trans-ventricular"

    if not trans_vent_inner.exists():
        print(f"Image folder not found: {trans_vent_inner}")
        return

    all_images = list(trans_vent_inner.glob("*.png")) + list(trans_vent_inner.glob("*.jpg"))
    print(f"\n📊 Found {len(all_images)} images in: {trans_vent_inner.name}/")
    possible_mask_dirs = [
        extract_dir / "masks",
        extract_dir / "SegmentationClass",
        extract_dir / "labels",
        extract_dir / "Trans-ventricular" / "masks",
        extract_dir / "Trans-ventricular" / "SegmentationClass",
    ]

    mask_dir = None
    for dir_path in possible_mask_dirs:
        if dir_path.exists() and list(dir_path.glob("*.png")):
            mask_dir = dir_path
            print(f"✓ Found masks in: {mask_dir.relative_to(raw_dir)}")
            break

    if mask_dir is None:
        print(f"⚠️  Searching for masks recursively...")
        all_extracted_pngs = list(extract_dir.rglob("*.png"))
        if all_extracted_pngs:
            mask_dir = all_extracted_pngs[0].parent
            print(f"✓ Found masks in: {mask_dir.relative_to(raw_dir)}")
        else:
            print(f"No masks found in extracted folder!")
            return

    all_masks = list(mask_dir.glob("*.png"))
    print(f"📊 Found {len(all_masks)} masks in: {mask_dir.name}/")


    print(f"\n📋 Copying {len(all_images)} images...")
    for img_file in tqdm(all_images):
        dest = images_out / img_file.name
        if not dest.exists():
            shutil.copy2(img_file, dest)

    print(f"\n📋 Copying {len(all_masks)} masks...")
    for mask_file in tqdm(all_masks):
        dest = masks_out / mask_file.name
        if not dest.exists():
            shutil.copy2(mask_file, dest)

    print(f"\n📋 Locating pixel size CSV...")

    csv_filename = "Trans-ventricular-Pixel-Size.csv"
    csv_dest     = processed_dir / "pixel_size.csv"

    csv_search_paths = [
        raw_dir / "Trans-ventricular" / csv_filename,
        raw_dir / csv_filename,
        *list((raw_dir / "Trans-ventricular").rglob(csv_filename))
    ]

    csv_found = False
    for csv_path in csv_search_paths:
        if Path(csv_path).exists():
            shutil.copy2(csv_path, csv_dest)
            print(f"✓ Pixel size CSV copied → {csv_dest}")
            csv_found = True
            break

    if not csv_found:
        print(f"'{csv_filename}' not found anywhere under {raw_dir}")
        print(f"   Manually copy it to: {csv_dest}")


    print(f"\n🔍 Verifying image-mask pairs...")

    image_names = set(f.stem for f in images_out.glob("*.png"))
    mask_names  = set(f.stem for f in masks_out.glob("*.png"))

    matched     = image_names & mask_names
    only_images = image_names - mask_names
    only_masks  = mask_names  - image_names

    print(f"\n📊 Matching results:")
    print(f"  ✓ Matched pairs:           {len(matched)}")
    print(f"  ⚠️  Images without masks:  {len(only_images)}")
    print(f"  ⚠️  Masks without images:  {len(only_masks)}")

    if only_images:
        print(f"\n  Sample images without masks:")
        for name in list(only_images)[:5]:
            print(f"    - {name}")

    if only_masks:
        print(f"\n  Sample masks without images:")
        for name in list(only_masks)[:5]:
            print(f"    - {name}")

    print(f"\n" + "=" * 70)
    print("ORGANIZATION COMPLETE")
    print("=" * 70)
    print(f"📁 Output:")
    print(f"  Images:    {images_out}  ({len(list(images_out.glob('*.png')))})")
    print(f"  Masks:     {masks_out}  ({len(list(masks_out.glob('*.png')))})")
    print(f"  CSV:       {csv_dest}  ({'✅' if csv_dest.exists() else '❌'})")
    print(f"  Matched:   {len(matched)} pairs")

    if len(matched) > 0:
        print(f"\n✅ Ready to run: python preprocess_data.py")
    else:
        print(f"\n⚠️  No matched pairs. Check naming conventions.")


if __name__ == "__main__":
    organize_dataset()

DATASET ORGANIZATION - FIXED VERSION

📦 Extracting segmentation masks...
✓ Extracted to: fetal_vm_segmentation/data/raw/Trans-ventricular/segmentation-mask-extracted

📊 Found 584 images in: Trans-ventricular/
✓ Found masks in: Trans-ventricular/segmentation-mask-extracted/SegmentationClass
📊 Found 584 masks in: SegmentationClass/

📋 Copying 584 images...


100%|██████████| 584/584 [00:00<00:00, 1790.41it/s]



📋 Copying 584 masks...


100%|██████████| 584/584 [00:00<00:00, 5995.02it/s]


📋 Locating pixel size CSV...
✓ Pixel size CSV copied → fetal_vm_segmentation/data/processed/pixel_size.csv

🔍 Verifying image-mask pairs...

📊 Matching results:
  ✓ Matched pairs:           584
  ⚠️  Images without masks:  0
  ⚠️  Masks without images:  0

ORGANIZATION COMPLETE
📁 Output:
  Images:    fetal_vm_segmentation/data/processed/images  (584)
  Masks:     fetal_vm_segmentation/data/processed/masks  (584)
  CSV:       fetal_vm_segmentation/data/processed/pixel_size.csv  (✅)
  Matched:   584 pairs

✅ Ready to run: python preprocess_data.py


In [ ]:
"""
Full Preprocessing Pipeline
Generates segmentation, edge, and auxiliary-mask labels.
"""

import cv2
import json
import numpy as np
import albumentations as A
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm


# ── 1. Edge map generation (for 'edge' output head) ──────────────────────────

def generate_edge_map(mask_binary, kernel_size=3):
    """
    Morphological boundary: dilated - eroded.
    Returns uint8 HxW, values 0 or 255.
    """
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    dilated = cv2.dilate(mask_binary, kernel, iterations=1)
    eroded = cv2.erode(mask_binary, kernel, iterations=1)
    edge = np.clip(
        dilated.astype(np.int16) - eroded.astype(np.int16), 0, 255
    ).astype(np.uint8)
    return edge


def generate_aux_masks(mask_binary, sizes=[(256, 256), (128, 128)]):
    """
    Downsample mask for auxiliary supervision.
    Returns [aux_d2 (256x256), aux_d3 (128x128)].
    Uses INTER_NEAREST to preserve binary values exactly.
    """
    return [
        cv2.resize(mask_binary, (w, h), interpolation=cv2.INTER_NEAREST)
        for (h, w) in sizes
    ]

class FetalVMPreprocessor:

    def __init__(
        self,
        images_dir,
        masks_dir,
        output_dir,
        target_size=(512, 512),
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.output_dir = Path(output_dir)
        self.target_size = target_size

        for split in ["train", "val", "test"]:
            for subdir in ["images", "masks", "edges", "aux_d2", "aux_d3"]:
                (self.output_dir / split / subdir).mkdir(
                    parents=True,
                    exist_ok=True,
                )

    def get_augmentation_pipeline(self, is_train=True):
        if is_train:
            return A.Compose([A.Resize(*self.target_size)])
            # return A.Compose([
            #     A.Resize(*self.target_size),
            #     A.HorizontalFlip(p=0.5),
            #     A.ShiftScaleRotate(
            #         shift_limit=0.1,
            #         scale_limit=0.1,
            #         rotate_limit=15,
            #         p=0.5,
            #     ),
            #     A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            #     A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            #     A.Blur(blur_limit=3, p=0.2),
            # ])
        return A.Compose([A.Resize(*self.target_size)])

    def process_dataset(
        self,
        train_ratio=0.70,
        val_ratio=0.15,
        test_ratio=0.15,
    ):
        # Collect valid image-mask pairs
        image_files = sorted(
            list(self.images_dir.glob("*.png"))
            + list(self.images_dir.glob("*.jpg"))
        )

        valid_pairs = []
        for img_path in image_files:
            mask_path = self.masks_dir / img_path.name

            if not mask_path.exists():
                mask_path = self.masks_dir / f"{img_path.stem}.png"

            if mask_path.exists():
                valid_pairs.append((img_path, mask_path))

        print(f"✓ {len(valid_pairs)} valid image-mask pairs")

        # Split
        train_val, test = train_test_split(
            valid_pairs,
            test_size=test_ratio,
            random_state=42,
        )
        train, val = train_test_split(
            train_val,
            test_size=val_ratio / (train_ratio + val_ratio),
            random_state=42,
        )

        print("\n📂 Split:")
        print(
            f"  Train: {len(train)} | "
            f"Val: {len(val)} | "
            f"Test: {len(test)}"
        )

        splits = {
            "train": (train, True),
            "val": (val, False),
            "test": (test, False),
        }
        metadata = {"train": [], "val": [], "test": []}

        for split_name, (pairs, is_train) in splits.items():
            print(f"\n🔄 Processing {split_name} ({len(pairs)} samples)...")
            transform = self.get_augmentation_pipeline(is_train)

            for idx, (img_path, mask_path) in enumerate(tqdm(pairs)):
                # Load
                image = cv2.imread(str(img_path))
                if image is None:
                    continue

                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                if mask is None:
                    continue

                mask = (mask > 127).astype(np.uint8) * 255

                # Augment
                transformed = transform(image=image, mask=mask)
                image_aug = transformed["image"]
                mask_aug = transformed["mask"]

                # Skip empty or trivial masks
                if (mask_aug > 127).sum() < 100:
                    continue

                # Derived labels
                edge_map = generate_edge_map(mask_aug)
                aux_d2, aux_d3 = generate_aux_masks(
                    mask_aug,
                    sizes=[(256, 256), (128, 128)],
                )

                # Save
                fname = f"{idx:04d}.png"
                out = self.output_dir / split_name

                cv2.imwrite(
                    str(out / "images" / fname),
                    cv2.cvtColor(image_aug, cv2.COLOR_RGB2BGR),
                )
                cv2.imwrite(str(out / "masks" / fname), mask_aug)
                cv2.imwrite(str(out / "edges" / fname), edge_map)
                cv2.imwrite(str(out / "aux_d2" / fname), aux_d2)
                cv2.imwrite(str(out / "aux_d3" / fname), aux_d3)

                # Metadata
                metadata[split_name].append(
                    {
                        "id": idx,
                        "file": img_path.name,
                    }
                )

        # Save metadata
        meta_path = self.output_dir / "metadata.json"
        with open(meta_path, "w", encoding="utf-8") as file:
            json.dump(metadata, file, indent=2)

        print("\n Done.")
        print(f"   Preprocessed data → {self.output_dir}")
        print(f"   Metadata          → {meta_path}")


# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    BASE_DIR = Path("fetal_vm_segmentation")

    print("=" * 70)
    print("FETAL VM SEGMENTATION - Data Preprocessing")
    print("=" * 70)

    preprocessor = FetalVMPreprocessor(
        images_dir=BASE_DIR / "data" / "processed" / "images",
        masks_dir=BASE_DIR / "data" / "processed" / "masks",
        output_dir=BASE_DIR / "data" / "preprocessed",
        target_size=(512, 512),
    )

    preprocessor.process_dataset(
        train_ratio=0.70,
        val_ratio=0.15,
        test_ratio=0.15,
    )

    print("\n✅ Ready for training!")
    print("Next step: python train_mvsegnet.py")

FETAL VM SEGMENTATION - Data Preprocessing
✓ 584 valid image-mask pairs

📂 Split:
  Train: 408 | Val: 88 | Test: 88

🔄 Processing train (408 samples)...


100%|██████████| 408/408 [00:14<00:00, 27.73it/s]



🔄 Processing val (88 samples)...


100%|██████████| 88/88 [00:02<00:00, 29.90it/s]



🔄 Processing test (88 samples)...


100%|██████████| 88/88 [00:02<00:00, 30.23it/s]


✅ Done.
   Preprocessed data → fetal_vm_segmentation/data/preprocessed
   Metadata          → fetal_vm_segmentation/data/preprocessed/metadata.json

✅ Ready for training!
Next step: python train_mvsegnet.py


**Run this for better result**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def model_size_mb(model: nn.Module) -> float:
    return count_parameters(model) * 4 / 1024 / 1024




class FiLMLayer(nn.Module):

    def __init__(self, feature_channels: int, condition_dim: int = 1):
        super().__init__()
        self.gamma = nn.Linear(condition_dim, feature_channels)
        self.beta  = nn.Linear(condition_dim, feature_channels)

        # FIX 1: true identity initialization
        nn.init.zeros_(self.gamma.weight)
        nn.init.ones_(self.gamma.bias)
        nn.init.zeros_(self.beta.weight)
        nn.init.zeros_(self.beta.bias)

    def forward(self, x: torch.Tensor, condition: torch.Tensor) -> torch.Tensor:
        """
        x         : (B, C, H, W)
        condition : (B, condition_dim)
        """
        gamma = self.gamma(condition).unsqueeze(-1).unsqueeze(-1)  # (B,C,1,1)
        beta  = self.beta(condition).unsqueeze(-1).unsqueeze(-1)
        return gamma * x + beta

class AttentionGate(nn.Module):

    def __init__(self, F_g: int, F_l: int, F_int: int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.shape[2:] != x1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=False)
        psi = F.relu(g1 + x1, inplace=True)
        return x * self.psi(psi)



class MultiScaleVentricleAttention(nn.Module):

    def __init__(self, in_channels: int):
        super().__init__()
        mid = in_channels // 4

        self.branch_1x1 = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True)
        )
        self.branch_3x3 = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1, groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            nn.Conv2d(mid, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True)
        )
        self.branch_5x5 = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 5, padding=2, groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            nn.Conv2d(mid, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True)
        )
        self.branch_pool = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True)
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, in_channels // 8, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 8, in_channels, 1),
            nn.Sigmoid()
        )
        self.spatial_attn = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        ms = torch.cat([
            self.branch_1x1(x),
            self.branch_3x3(x),
            self.branch_5x5(x),
            self.branch_pool(x)
        ], dim=1)

        ms = ms * self.channel_attn(ms)
        sa = self.spatial_attn(torch.cat([
            ms.mean(dim=1, keepdim=True),
            ms.max(dim=1, keepdim=True)[0]
        ], dim=1))
        return ms * sa


class AdaptiveBoundaryRefinement(nn.Module):

    def __init__(self, in_channels: int):
        super().__init__()
        mid = in_channels // 4

        self.edge_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, 1, 1)
        )
        self.boundary_conv = nn.Sequential(
            nn.Conv2d(in_channels + 1, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1, groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True)
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(in_channels + mid, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor):
        edge_logits = self.edge_conv(x)
        edge_prob   = torch.sigmoid(edge_logits)
        refined     = self.boundary_conv(torch.cat([x, edge_prob], dim=1))
        out         = self.fusion(torch.cat([x, refined], dim=1))
        return out, edge_logits


class DecoderBlock(nn.Module):

    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, dropout: float = 0.1):
        super().__init__()
        self.upconv = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(in_ch, in_ch // 2, 1, bias=False),
            nn.BatchNorm2d(in_ch // 2),
            nn.ReLU(inplace=True)
        )
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch // 2 + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.upconv(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        out = self.conv(torch.cat([x, skip], dim=1))
        return self.drop(out)



#  MVSegNet


class MVSegNet(nn.Module):

    def __init__(self, num_classes: int = 1, num_severity_classes: int = 4):
        super().__init__()


        try:
            mobilenet = models.mobilenet_v3_small(
                weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
            )
        except Exception:
            mobilenet = models.mobilenet_v3_small(pretrained=True)

        self.encoder0 = mobilenet.features[0:2]
        self.encoder1 = mobilenet.features[2:4]
        self.encoder2 = mobilenet.features[4:9]
        self.encoder3 = mobilenet.features[9:12]
        self.encoder4 = mobilenet.features[12:]

        dummy = torch.randn(1, 3, 224, 224)
        was_training = mobilenet.training
        mobilenet.eval()
        with torch.no_grad():
            e0 = self.encoder0(dummy)
            e1 = self.encoder1(e0)
            e2 = self.encoder2(e1)
            e3 = self.encoder3(e2)
            e4 = self.encoder4(e3)
        mobilenet.train(was_training)

        self.c0, self.c1, self.c2, self.c3, self.c4 = (
            e0.shape[1], e1.shape[1], e2.shape[1], e3.shape[1], e4.shape[1]
        )
        print(f"[MVSegNet] Encoder channels: {self.c0}, {self.c1}, {self.c2}, {self.c3}, {self.c4}")


        self.msam = MultiScaleVentricleAttention(self.c4)
        self.abrb_d2 = AdaptiveBoundaryRefinement(self.c2)
        self.film = FiLMLayer(feature_channels=self.c4, condition_dim=1)
        self.ag1 = AttentionGate(F_g=self.c4, F_l=self.c3, F_int=self.c3 // 2)
        self.ag2 = AttentionGate(F_g=self.c3, F_l=self.c2, F_int=self.c2 // 2)
        self.ag3 = AttentionGate(F_g=self.c2, F_l=self.c1, F_int=max(self.c1 // 2, 4))
        self.ag4 = AttentionGate(F_g=self.c1, F_l=self.c0, F_int=max(self.c0 // 2, 4))
        self.decoder1 = DecoderBlock(self.c4, self.c3, self.c3, dropout=0.2)
        self.decoder2 = DecoderBlock(self.c3, self.c2, self.c2, dropout=0.15)
        self.decoder3 = DecoderBlock(self.c2, self.c1, self.c1, dropout=0.1)
        self.decoder4 = DecoderBlock(self.c1, self.c0, self.c0, dropout=0.0)
        self.msam_d1 = MultiScaleVentricleAttention(self.c3)
        self.msam_d2 = MultiScaleVentricleAttention(self.c2)

        self.seg_head = nn.Sequential(
            nn.Conv2d(self.c0, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1)
        )

        self.aux_head_d3 = nn.Conv2d(self.c1, num_classes, 1)
        self.aux_head_d2 = nn.Conv2d(self.c2, num_classes, 1)

        _ = num_severity_classes

    def forward(self, x: torch.Tensor, ga_weeks: torch.Tensor = None) -> dict:
        input_size = x.shape[2:]

        e0 = self.encoder0(x)
        e1 = self.encoder1(e0)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)


        e4 = self.msam(e4)


        if ga_weeks is not None:
            e4 = self.film(e4, ga_weeks)


        e3_att = self.ag1(e4, e3)
        d1 = self.decoder1(e4, e3_att)
        d1 = self.msam_d1(d1)

        e2_att = self.ag2(d1, e2)
        d2 = self.decoder2(d1, e2_att)
        d2 = self.msam_d2(d2)

        d2, edge_logits = self.abrb_d2(d2)

        e1_att = self.ag3(d2, e1)
        d3 = self.decoder3(d2, e1_att)

        e0_att = self.ag4(d3, e0)
        d4 = self.decoder4(d3, e0_att)

        edge_logits_up = F.interpolate(
            edge_logits, size=input_size, mode='bilinear', align_corners=False
        )

        seg_out = F.interpolate(
            self.seg_head(d4), size=input_size, mode='bilinear', align_corners=False
        )

        aux_d3 = F.interpolate(
            self.aux_head_d3(d3), size=input_size, mode='bilinear', align_corners=False
        )
        aux_d2 = F.interpolate(
            self.aux_head_d2(d2), size=input_size, mode='bilinear', align_corners=False
        )

        return {
            "seg": seg_out,
            "edge": edge_logits_up,
            "aux": [aux_d3, aux_d2],
        }

MODEL_REGISTRY = {
    'mvsegnet': MVSegNet,
}

def build_model(name: str, **kwargs) -> nn.Module:
    name = name.lower()
    if name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model '{name}'. Choose from {list(MODEL_REGISTRY)}")
    model = MODEL_REGISTRY[name](**kwargs)
    params = count_parameters(model)
    size = model_size_mb(model)
    print(f"[{name}] Parameters: {params:,}  |  Size: {size:.1f} MB")
    return model


if __name__ == '__main__':
    print("=" * 60)
    m = build_model('mvsegnet', num_classes=1, num_severity_classes=4)
    x = torch.randn(2, 3, 512, 512)
    ga = torch.rand(2, 1)
    with torch.no_grad():
        out = m(x, ga)
    print(f"  seg         : {tuple(out['seg'].shape)}")
    print(f"  edge        : {tuple(out['edge'].shape)}")
    print(f"  aux[0]      : {tuple(out['aux'][0].shape)}")
    print(f"  aux[1]      : {tuple(out['aux'][1].shape)}")
    print("=" * 60)
    print("MVSegNet OK.")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 192MB/s]


[MVSegNet] Encoder channels: 16, 24, 48, 96, 576
[mvsegnet] Parameters: 2,153,492  |  Size: 8.2 MB
  seg         : (2, 1, 512, 512)
  edge        : (2, 1, 512, 512)
  aux[0]      : (2, 1, 512, 512)
  aux[1]      : (2, 1, 512, 512)
MVSegNet OK.


**Mobilenet+ attention unet. This is the final because of ablation problem**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def model_size_mb(model: nn.Module) -> float:
    return count_parameters(model) * 4 / 1024 / 1024




class AttentionGate(nn.Module):
    """
    Additive attention gate from Attention U-Net.
    """
    def __init__(self, F_g: int, F_l: int, F_int: int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.shape[2:] != x1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=False)
        psi = F.relu(g1 + x1, inplace=True)
        return x * self.psi(psi)



class DecoderBlock(nn.Module):
    """
    Bilinear upsample + Conv decoder block.
    """
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, dropout: float = 0.1):
        super().__init__()
        self.upconv = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(in_ch, in_ch // 2, 1, bias=False),
            nn.BatchNorm2d(in_ch // 2),
            nn.ReLU(inplace=True)
        )
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch // 2 + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.upconv(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        out = self.conv(torch.cat([x, skip], dim=1))
        return self.drop(out)




class MVSegNet(nn.Module):
    """
    Simplified MVSegNet — MobileNetV3-Small encoder + Attention-gated
    U-Net decoder only. FiLM, MSAM, and ABRB have been removed.

    Return type is unchanged: a dict with keys "seg", "edge", "aux",
    where "edge" now comes from a lightweight conv head (no boundary
    refinement module) and "aux" is a list of two auxiliary logits.
    """
    def __init__(self, num_classes: int = 1, num_severity_classes: int = 4):
        super().__init__()

        # ── Encoder
        try:
            mobilenet = models.mobilenet_v3_small(
                weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
            )
        except Exception:
            mobilenet = models.mobilenet_v3_small(pretrained=True)

        self.encoder0 = mobilenet.features[0:2]
        self.encoder1 = mobilenet.features[2:4]
        self.encoder2 = mobilenet.features[4:9]
        self.encoder3 = mobilenet.features[9:12]
        self.encoder4 = mobilenet.features[12:]


        dummy = torch.randn(1, 3, 224, 224)
        was_training = mobilenet.training
        mobilenet.eval()
        with torch.no_grad():
            e0 = self.encoder0(dummy)
            e1 = self.encoder1(e0)
            e2 = self.encoder2(e1)
            e3 = self.encoder3(e2)
            e4 = self.encoder4(e3)
        mobilenet.train(was_training)

        self.c0, self.c1, self.c2, self.c3, self.c4 = (
            e0.shape[1], e1.shape[1], e2.shape[1], e3.shape[1], e4.shape[1]
        )
        print(f"[MVSegNet] Encoder channels: {self.c0}, {self.c1}, {self.c2}, {self.c3}, {self.c4}")

        # ── Attention gates ──────────────────────────────────
        self.ag1 = AttentionGate(F_g=self.c4, F_l=self.c3, F_int=self.c3 // 2)
        self.ag2 = AttentionGate(F_g=self.c3, F_l=self.c2, F_int=self.c2 // 2)
        self.ag3 = AttentionGate(F_g=self.c2, F_l=self.c1, F_int=max(self.c1 // 2, 4))
        self.ag4 = AttentionGate(F_g=self.c1, F_l=self.c0, F_int=max(self.c0 // 2, 4))

        # ── Decoder ──────────────────────────────────────────
        self.decoder1 = DecoderBlock(self.c4, self.c3, self.c3, dropout=0.2)
        self.decoder2 = DecoderBlock(self.c3, self.c2, self.c2, dropout=0.15)
        self.decoder3 = DecoderBlock(self.c2, self.c1, self.c1, dropout=0.1)
        self.decoder4 = DecoderBlock(self.c1, self.c0, self.c0, dropout=0.0)

        # ── Segmentation head ────────────────────────────────
        self.seg_head = nn.Sequential(
            nn.Conv2d(self.c0, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1)
        )

        self.edge_head = nn.Sequential(
            nn.Conv2d(self.c2, self.c2 // 4, 1, bias=False),
            nn.BatchNorm2d(self.c2 // 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(self.c2 // 4, 1, 1)
        )


        self.aux_head_d3 = nn.Conv2d(self.c1, num_classes, 1)
        self.aux_head_d2 = nn.Conv2d(self.c2, num_classes, 1)


        _ = num_severity_classes

    def forward(self, x: torch.Tensor, ga_weeks: torch.Tensor = None) -> dict:
        input_size = x.shape[2:]

        _ = ga_weeks


        e0 = self.encoder0(x)
        e1 = self.encoder1(e0)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)


        e3_att = self.ag1(e4, e3)
        d1 = self.decoder1(e4, e3_att)

        e2_att = self.ag2(d1, e2)
        d2 = self.decoder2(d1, e2_att)

        edge_logits = self.edge_head(d2)

        e1_att = self.ag3(d2, e1)
        d3 = self.decoder3(d2, e1_att)

        e0_att = self.ag4(d3, e0)
        d4 = self.decoder4(d3, e0_att)

        edge_logits_up = F.interpolate(
            edge_logits, size=input_size, mode='bilinear', align_corners=False
        )

        seg_out = F.interpolate(
            self.seg_head(d4), size=input_size, mode='bilinear', align_corners=False
        )

        aux_d3 = F.interpolate(
            self.aux_head_d3(d3), size=input_size, mode='bilinear', align_corners=False
        )
        aux_d2 = F.interpolate(
            self.aux_head_d2(d2), size=input_size, mode='bilinear', align_corners=False
        )

        return {
            "seg": seg_out,
            "edge": edge_logits_up,
            "aux": [aux_d3, aux_d2],
        }



MODEL_REGISTRY = {
    'mvsegnet': MVSegNet,
}

def build_model(name: str, **kwargs) -> nn.Module:
    name = name.lower()
    if name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model '{name}'. Choose from {list(MODEL_REGISTRY)}")
    model = MODEL_REGISTRY[name](**kwargs)
    params = count_parameters(model)
    size = model_size_mb(model)
    print(f"[{name}] Parameters: {params:,}  |  Size: {size:.1f} MB")
    return model


if __name__ == '__main__':
    print("=" * 60)
    m = build_model('mvsegnet', num_classes=1, num_severity_classes=4)
    x = torch.randn(2, 3, 512, 512)
    with torch.no_grad():
        out = m(x)
    print(f"  seg         : {tuple(out['seg'].shape)}")
    print(f"  edge        : {tuple(out['edge'].shape)}")
    print(f"  aux[0]      : {tuple(out['aux'][0].shape)}")
    print(f"  aux[1]      : {tuple(out['aux'][1].shape)}")
    print("=" * 60)
    print("MVSegNet OK.")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 94.0MB/s]


[MVSegNet] Encoder channels: 16, 24, 48, 96, 576
[mvsegnet] Parameters: 1,665,164  |  Size: 6.4 MB
  seg         : (2, 1, 512, 512)
  edge        : (2, 1, 512, 512)
  aux[0]      : (2, 1, 512, 512)
  aux[1]      : (2, 1, 512, 512)
MVSegNet OK.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models



class TwoWayCrossAttention(nn.Module):
    """Simplified SAM-style two-way attention block."""
    def __init__(self, embed_dim: int = 256, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.token_self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.token_to_img    = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.img_to_token    = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)

        self.norm_t1 = nn.LayerNorm(embed_dim)
        self.norm_t2 = nn.LayerNorm(embed_dim)
        self.norm_t3 = nn.LayerNorm(embed_dim)
        self.norm_f1 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.norm_ffn = nn.LayerNorm(embed_dim)

    def forward(self, tokens: torch.Tensor, image_feats: torch.Tensor):
        t2 = self.norm_t1(tokens)
        t2, _ = self.token_self_attn(t2, t2, t2)
        tokens = tokens + t2

        t2 = self.norm_t2(tokens)
        t2, _ = self.token_to_img(t2, image_feats, image_feats)
        tokens = tokens + t2

        f2 = self.norm_f1(image_feats)
        f2, _ = self.img_to_token(f2, tokens, tokens)
        image_feats = image_feats + f2

        tokens = tokens + self.ffn(self.norm_t3(tokens))
        return tokens, image_feats


class FetSAM(nn.Module):

    def __init__(self, num_classes: int = 1, num_severity_classes: int = 4,
                 embed_dim: int = 256):
        super().__init__()

        try:
            efficientnet = models.efficientnet_b0(
                weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
            )
        except Exception:
            efficientnet = models.efficientnet_b0(pretrained=True)

        self.enc0 = efficientnet.features[0:2]
        self.enc1 = efficientnet.features[2:3]
        self.enc2 = efficientnet.features[3:4]
        self.enc3 = efficientnet.features[4:6]
        self.enc4 = efficientnet.features[6:]

        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            _e0 = self.enc0(dummy)
            _e1 = self.enc1(_e0)
            _e2 = self.enc2(_e1)
            _e3 = self.enc3(_e2)
            _e4 = self.enc4(_e3)
        self.c0, self.c1, self.c2, self.c3, self.c4 = (
            _e0.shape[1], _e1.shape[1], _e2.shape[1], _e3.shape[1], _e4.shape[1]
        )
        print(f"[FetSAM]   Encoder channels: {self.c0}, {self.c1}, {self.c2}, {self.c3}, {self.c4}")

        self.img_proj = nn.Sequential(
            nn.Conv2d(self.c4, embed_dim, 1, bias=False),
            nn.BatchNorm2d(embed_dim), nn.ReLU(inplace=True)
        )
        self.pos_embed  = nn.Parameter(torch.randn(1, embed_dim, 16, 16) * 0.02)
        self.mask_token = nn.Embedding(1, embed_dim)

        self.cross_attn1 = TwoWayCrossAttention(embed_dim, num_heads=8)
        self.cross_attn2 = TwoWayCrossAttention(embed_dim, num_heads=8)

        E = embed_dim
        self.skip3 = nn.Conv2d(self.c3, E // 2,  1, bias=False)
        self.skip2 = nn.Conv2d(self.c2, E // 4,  1, bias=False)
        self.skip1 = nn.Conv2d(self.c1, E // 8,  1, bias=False)
        self.skip0 = nn.Conv2d(self.c0, E // 16, 1, bias=False)

        def up_block(in_ch, out_ch):
            return nn.Sequential(
                nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2),
                nn.BatchNorm2d(out_ch), nn.GELU()
            )

        def merge_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.GELU(),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.GELU()
            )

        self.up1    = up_block(E,      E // 2)
        self.merge1 = merge_block(E,   E // 2)

        self.up2    = up_block(E // 2,  E // 4)
        self.merge2 = merge_block(E // 2, E // 4)

        self.up3    = up_block(E // 4,  E // 8)
        self.merge3 = merge_block(E // 4, E // 8)

        self.up4    = up_block(E // 8,  E // 16)
        self.merge4 = merge_block(E // 8, E // 16)

        self.seg_head = nn.Sequential(
            nn.Conv2d(E // 16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, num_classes, 1)
        )
        self.edge_head = nn.Sequential(
            nn.Conv2d(E, 64, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1)
        )

        self.uncertainty_head = nn.Sequential(
            nn.Conv2d(E // 16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1),
            nn.Sigmoid()
        )

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Sequential(
            nn.Linear(E, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_severity_classes)
        )

    @staticmethod
    def _align(src: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
        if src.shape[2:] != ref.shape[2:]:
            src = F.interpolate(src, size=ref.shape[2:], mode='bilinear', align_corners=True)
        return src

    def forward(self, x: torch.Tensor,
                ga_weeks: torch.Tensor = None) -> dict:
        input_size = x.shape[2:]

        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        feat = self.img_proj(e4)
        pos  = F.interpolate(self.pos_embed, size=feat.shape[2:],
                             mode='bilinear', align_corners=True)
        feat = feat + pos

        B, C, H, W = feat.shape
        feat_flat = feat.permute(0, 2, 3, 1).reshape(B, H * W, C)

        tokens = self.mask_token.weight.unsqueeze(0).expand(B, -1, -1)

        tokens, feat_flat = self.cross_attn1(tokens, feat_flat)
        tokens, feat_flat = self.cross_attn2(tokens, feat_flat)

        feat_spatial = feat_flat.reshape(B, H, W, C).permute(0, 3, 1, 2)

        edge_map = torch.sigmoid(self.edge_head(feat_spatial))

        d1 = self.up1(feat_spatial)
        d1 = self._align(d1, e3)
        d1 = self.merge1(torch.cat([d1, self.skip3(e3)], dim=1))

        d2 = self.up2(d1)
        d2 = self._align(d2, e2)
        d2 = self.merge2(torch.cat([d2, self.skip2(e2)], dim=1))

        d3 = self.up3(d2)
        d3 = self._align(d3, e1)
        d3 = self.merge3(torch.cat([d3, self.skip1(e1)], dim=1))

        d4 = self.up4(d3)
        d4 = self._align(d4, e0)
        d4 = self.merge4(torch.cat([d4, self.skip0(e0)], dim=1))

        seg_out = F.interpolate(
            self.seg_head(d4), size=input_size, mode='bilinear', align_corners=True
        )
        uncertainty = F.interpolate(
            self.uncertainty_head(d4), size=input_size, mode='bilinear', align_corners=True
        )
        class_out = self.classifier(self.global_pool(feat_spatial).flatten(1))

        return {
            'seg':         seg_out,
            'class':       class_out,
            'edge':        edge_map,
            'uncertainty': uncertainty,
        }


if __name__ == '__main__':
    m = FetSAM(num_classes=1, num_severity_classes=4)
    x = torch.randn(2, 3, 512, 512)
    with torch.no_grad():
        out = m(x)
    print(f"  seg         : {tuple(out['seg'].shape)}")
    print(f"  class       : {tuple(out['class'].shape)}")
    print(f"  edge        : {tuple(out['edge'].shape)}")
    print(f"  uncertainty : {tuple(out['uncertainty'].shape)}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 147MB/s]


[FetSAM]   Encoder channels: 16, 24, 40, 112, 1280
  seg         : (2, 1, 512, 512)
  class       : (2, 4)
  edge        : (2, 1, 16, 16)
  uncertainty : (2, 1, 512, 512)


In [ ]:
MODEL_REGISTRY = {
    'mvsegnet': MVSegNet,
    'fetsam':   FetSAM,
}

def build_model(name: str, **kwargs) -> nn.Module:
    name = name.lower()
    if name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model '{name}'. Choose from {list(MODEL_REGISTRY)}")
    model  = MODEL_REGISTRY[name](**kwargs)
    params = count_parameters(model)
    size   = model_size_mb(model)
    print(f"[{name}] Parameters: {params:,}  |  Size: {size:.1f} MB")
    return model

In [ ]:
# ─────────────────────────────────────────────────────────────
#  STANDARD U-Net  (Generalization Baseline)
# ─────────────────────────────────────────────────────────────

class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.drop(self.block(x))


class UNet(nn.Module):


    def __init__(self, num_classes: int = 1,
                 num_severity_classes: int = 4,
                 base_ch: int = 32):
        super().__init__()
        b = base_ch  # 64 → 128 → 256 → 512 → 1024

        # ── Encoder ──────────────────────────────────────────
        self.enc1 = DoubleConv(3,   b,   dropout=0.0)
        self.enc2 = DoubleConv(b,   b*2, dropout=0.0)
        self.enc3 = DoubleConv(b*2, b*4, dropout=0.1)
        self.enc4 = DoubleConv(b*4, b*8, dropout=0.1)
        self.pool = nn.MaxPool2d(2)

        # ── Bottleneck ────────────────────────────────────────
        self.bottleneck = DoubleConv(b*8, b*16, dropout=0.2)

        # ── Decoder ───────────────────────────────────────────
        self.up4  = self._up_block(b*16, b*8)
        self.dec4 = DoubleConv(b*16, b*8,  dropout=0.1)

        self.up3  = self._up_block(b*8,  b*4)
        self.dec3 = DoubleConv(b*8,  b*4,  dropout=0.1)

        self.up2  = self._up_block(b*4,  b*2)
        self.dec2 = DoubleConv(b*4,  b*2,  dropout=0.0)

        self.up1  = self._up_block(b*2,  b)
        self.dec1 = DoubleConv(b*2,  b,    dropout=0.0)

        # ── Segmentation head ─────────────────────────────────
        self.seg_head = nn.Sequential(
            nn.Conv2d(b, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1)
        )

        # ── Edge head — RAW LOGITS (F1 contract) ──────────────
        self.edge_head = nn.Sequential(
            nn.Conv2d(b*16, 64, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1)
            # No sigmoid — EdgeLoss uses BCEWithLogits
        )

        # ── Uncertainty head — boundary targeted (F7 contract) ─
        self.uncertainty_head = nn.Sequential(
            nn.Conv2d(b, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1)
        )

        # ── Aux heads — deep supervision (F6 contract) ────────
        self.aux_head_d3 = nn.Conv2d(b*4, num_classes, 1)
        self.aux_head_d2 = nn.Conv2d(b*2, num_classes, 1)

        # ── Classification head ───────────────────────────────
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Sequential(
            nn.Linear(b*16, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_severity_classes)
        )

        print(f"[UNet] base_ch={base_ch} | "
              f"Params: {count_parameters(self):,} | "
              f"Size: {model_size_mb(self):.1f} MB")

    @staticmethod
    def _up_block(in_ch: int, out_ch: int) -> nn.Sequential:
        """Bilinear upsample + pointwise conv — no checkerboard."""
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )

    @staticmethod
    def _align(src: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
        if src.shape[2:] != ref.shape[2:]:
            src = F.interpolate(src, size=ref.shape[2:],
                                mode='bilinear', align_corners=True)
        return src

    def forward(self, x: torch.Tensor,
                ga_weeks: torch.Tensor = None) -> dict:
        # ga_weeks accepted but not used — keeps trainer/loss untouched
        input_size = x.shape[2:]

        # ── Encoder ──────────────────────────────────────────
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # ── Bottleneck ────────────────────────────────────────
        b  = self.bottleneck(self.pool(e4))

        # ── Edge logits from bottleneck ───────────────────────
        edge_logits = self.edge_head(b)

        # ── Decoder ───────────────────────────────────────────
        d4 = self.dec4(torch.cat([self._align(self.up4(b),  e4), e4], dim=1))
        d3 = self.dec3(torch.cat([self._align(self.up3(d4), e3), e3], dim=1))
        d2 = self.dec2(torch.cat([self._align(self.up2(d3), e2), e2], dim=1))
        d1 = self.dec1(torch.cat([self._align(self.up1(d2), e1), e1], dim=1))

        # ── All outputs ───────────────────────────────────────
        seg_out = F.interpolate(
            self.seg_head(d1),
            size=input_size, mode='bilinear', align_corners=True
        )
        uncertainty = F.interpolate(
            self.uncertainty_head(d1),
            size=input_size, mode='bilinear', align_corners=True
        )
        aux_d3 = F.interpolate(
            self.aux_head_d3(d3),
            size=input_size, mode='bilinear', align_corners=True
        )
        aux_d2 = F.interpolate(
            self.aux_head_d2(d2),
            size=input_size, mode='bilinear', align_corners=True
        )
        class_out = self.classifier(self.global_pool(b).flatten(1))

        return {
            'seg':         seg_out,
            'class':       class_out,
            'edge':        edge_logits,
            'uncertainty': uncertainty,
            'aux':         [aux_d3, aux_d2],
        }



MODEL_REGISTRY['unet'] = UNet



if __name__ == '__main__':
    m  = build_model('unet', num_classes=1, num_severity_classes=4)
    x  = torch.randn(2, 3, 512, 512)
    ga = torch.rand(2, 1)
    with torch.no_grad():
        out = m(x, ga)
    print(f"  seg         : {tuple(out['seg'].shape)}")
    print(f"  class       : {tuple(out['class'].shape)}")
    print(f"  edge        : {tuple(out['edge'].shape)}")
    print(f"  uncertainty : {tuple(out['uncertainty'].shape)}")
    print(f"  aux[0]      : {tuple(out['aux'][0].shape)}")
    print(f"  aux[1]      : {tuple(out['aux'][1].shape)}")

[UNet] base_ch=32 | Params: 7,425,257 | Size: 28.3 MB
[unet] Parameters: 7,425,257  |  Size: 28.3 MB
  seg         : (2, 1, 512, 512)
  class       : (2, 4)
  edge        : (2, 1, 32, 32)
  uncertainty : (2, 1, 512, 512)
  aux[0]      : (2, 1, 512, 512)
  aux[1]      : (2, 1, 512, 512)


In [ ]:


class AttentionUNet(nn.Module):
    def __init__(self, num_classes: int = 1,
                 num_severity_classes: int = 4,
                 base_ch: int = 32):
        super().__init__()
        b = base_ch


        self.enc1 = DoubleConv(3,   b,   dropout=0.0)
        self.enc2 = DoubleConv(b,   b*2, dropout=0.0)
        self.enc3 = DoubleConv(b*2, b*4, dropout=0.1)
        self.enc4 = DoubleConv(b*4, b*8, dropout=0.1)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(b*8, b*16, dropout=0.2)

        self.ag4 = AttentionGate(F_g=b*16, F_l=b*8,  F_int=b*4)
        self.ag3 = AttentionGate(F_g=b*8,  F_l=b*4,  F_int=b*2)
        self.ag2 = AttentionGate(F_g=b*4,  F_l=b*2,  F_int=b)
        self.ag1 = AttentionGate(F_g=b*2,  F_l=b,    F_int=max(b//2, 4))

        # ── Decoder (bilinear, no checkerboard) ───────────────
        self.up4  = self._up_block(b*16, b*8)
        self.dec4 = DoubleConv(b*16, b*8, dropout=0.1)

        self.up3  = self._up_block(b*8,  b*4)
        self.dec3 = DoubleConv(b*8,  b*4, dropout=0.1)

        self.up2  = self._up_block(b*4,  b*2)
        self.dec2 = DoubleConv(b*4,  b*2, dropout=0.0)

        self.up1  = self._up_block(b*2,  b)
        self.dec1 = DoubleConv(b*2,  b,   dropout=0.0)

        # ── Heads ─────────────────────────────────────────────
        self.seg_head = nn.Sequential(
            nn.Conv2d(b, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1)
        )
        self.edge_head = nn.Sequential(      # raw logits — no sigmoid
            nn.Conv2d(b*16, 64, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1)
        )
        self.uncertainty_head = nn.Sequential(
            nn.Conv2d(b, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1)
        )
        self.aux_head_d3 = nn.Conv2d(b*4, num_classes, 1)
        self.aux_head_d2 = nn.Conv2d(b*2, num_classes, 1)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Sequential(
            nn.Linear(b*16, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_severity_classes)
        )

        print(f"[AttentionUNet] base_ch={base_ch} | "
              f"Params: {count_parameters(self):,} | "
              f"Size: {model_size_mb(self):.1f} MB")

    @staticmethod
    def _up_block(in_ch: int, out_ch: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )

    @staticmethod
    def _align(src: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
        if src.shape[2:] != ref.shape[2:]:
            src = F.interpolate(src, size=ref.shape[2:],
                                mode='bilinear', align_corners=True)
        return src

    def forward(self, x: torch.Tensor,
                ga_weeks: torch.Tensor = None) -> dict:
        input_size = x.shape[2:]

        # ── Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))

        edge_logits = self.edge_head(b)

        # ── Attended decoder
        e4a = self.ag4(b,  e4)
        d4  = self.dec4(torch.cat([self._align(self.up4(b),  e4a), e4a], dim=1))

        e3a = self.ag3(d4, e3)
        d3  = self.dec3(torch.cat([self._align(self.up3(d4), e3a), e3a], dim=1))

        e2a = self.ag2(d3, e2)
        d2  = self.dec2(torch.cat([self._align(self.up2(d3), e2a), e2a], dim=1))

        e1a = self.ag1(d2, e1)
        d1  = self.dec1(torch.cat([self._align(self.up1(d2), e1a), e1a], dim=1))

        # ── Outputs
        seg_out = F.interpolate(
            self.seg_head(d1), size=input_size, mode='bilinear', align_corners=True)
        uncertainty = F.interpolate(
            self.uncertainty_head(d1), size=input_size, mode='bilinear', align_corners=True)
        aux_d3 = F.interpolate(
            self.aux_head_d3(d3), size=input_size, mode='bilinear', align_corners=True)
        aux_d2 = F.interpolate(
            self.aux_head_d2(d2), size=input_size, mode='bilinear', align_corners=True)
        class_out = self.classifier(self.global_pool(b).flatten(1))

        return {
            'seg':         seg_out,
            'class':       class_out,
            'edge':        edge_logits,
            'uncertainty': uncertainty,
            'aux':         [aux_d3, aux_d2],
        }


MODEL_REGISTRY['attention_unet'] = AttentionUNet

# ── Sanity check ──────────────────────────────────────────────
if __name__ == '__main__':
    m  = build_model('attention_unet', num_classes=1, num_severity_classes=4)
    x  = torch.randn(2, 3, 512, 512)
    with torch.no_grad():
        out = m(x)
    print(f"  seg         : {tuple(out['seg'].shape)}")
    print(f"  edge        : {tuple(out['edge'].shape)}")
    print(f"  uncertainty : {tuple(out['uncertainty'].shape)}")
    print(f"  aux[0]      : {tuple(out['aux'][0].shape)}")
    print(f"  aux[1]      : {tuple(out['aux'][1].shape)}")

[AttentionUNet] base_ch=32 | Params: 7,557,025 | Size: 28.8 MB
[attention_unet] Parameters: 7,557,025  |  Size: 28.8 MB
  seg         : (2, 1, 512, 512)
  edge        : (2, 1, 32, 32)
  uncertainty : (2, 1, 512, 512)
  aux[0]      : (2, 1, 512, 512)
  aux[1]      : (2, 1, 512, 512)


In [ ]:
class UNetPlusPlus(nn.Module):

    def __init__(self, num_classes: int = 1,
                 num_severity_classes: int = 4,
                 base_ch: int = 16):
        super().__init__()
        b = base_ch

        self.enc1       = DoubleConv(3,   b)
        self.enc2       = DoubleConv(b,   b*2)
        self.enc3       = DoubleConv(b*2, b*4)
        self.enc4       = DoubleConv(b*4, b*8)
        self.pool       = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(b*8, b*16)

        self.up_l3 = self._up(b*16, b*8)
        self.up_l2 = self._up(b*8,  b*4)
        self.up_l1 = self._up(b*4,  b*2)
        self.up_l0 = self._up(b*2,  b)

        self.n31 = DoubleConv(b*16, b*8)
        self.n21 = DoubleConv(b*8,  b*4)
        self.n11 = DoubleConv(b*4,  b*2)
        self.n01 = DoubleConv(b*2,  b)

        self.n22 = DoubleConv(b*12, b*4)
        self.n12 = DoubleConv(b*6,  b*2)
        self.n02 = DoubleConv(b*3,  b)

        self.n13 = DoubleConv(b*8,  b*2)
        self.n03 = DoubleConv(b*4,  b)

        self.n04 = DoubleConv(b*5,  b)

        self.seg_head = nn.Sequential(
            nn.Conv2d(b, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1)
        )

        self.edge_head = nn.Sequential(
            nn.Conv2d(b*16, 64, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1)
        )

        self.uncertainty_head = nn.Sequential(
            nn.Conv2d(b, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1)
        )

        self.aux_head_d3 = nn.Conv2d(b*4, num_classes, 1)
        self.aux_head_d2 = nn.Conv2d(b*2, num_classes, 1)

        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(b*16, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_severity_classes)
        )

        print(f"[UNetPlusPlus] base_ch={base_ch} | "
              f"Params: {count_parameters(self):,} | "
              f"Size: {model_size_mb(self):.1f} MB")

    @staticmethod
    def _up(in_ch: int, out_ch: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Upsample(
                scale_factor=2,
                mode='bilinear',
                align_corners=True
            ),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    @staticmethod
    def _align(src: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
        if src.shape[2:] != ref.shape[2:]:
            src = F.interpolate(
                src,
                size=ref.shape[2:],
                mode='bilinear',
                align_corners=True
            )

        return src

    def forward(self, x: torch.Tensor,
                ga_weeks: torch.Tensor = None) -> dict:

        input_size = x.shape[2:]

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        bn = self.bottleneck(self.pool(e4))

        edge_logits = self.edge_head(bn)
        class_out = self.classifier(
            self.global_pool(bn).flatten(1)
        )

        n31 = self.n31(
            torch.cat([
                e4,
                self._align(self.up_l3(bn), e4)
            ], dim=1)
        )

        n21 = self.n21(
            torch.cat([
                e3,
                self._align(self.up_l2(e4), e3)
            ], dim=1)
        )

        n11 = self.n11(
            torch.cat([
                e2,
                self._align(self.up_l1(e3), e2)
            ], dim=1)
        )

        n01 = self.n01(
            torch.cat([
                e1,
                self._align(self.up_l0(e2), e1)
            ], dim=1)
        )

        n22 = self.n22(
            torch.cat([
                e3,
                n21,
                self._align(self.up_l2(n31), e3)
            ], dim=1)
        )

        n12 = self.n12(
            torch.cat([
                e2,
                n11,
                self._align(self.up_l1(n21), e2)
            ], dim=1)
        )

        n02 = self.n02(
            torch.cat([
                e1,
                n01,
                self._align(self.up_l0(n11), e1)
            ], dim=1)
        )

        n13 = self.n13(
            torch.cat([
                e2,
                n11,
                n12,
                self._align(self.up_l1(n22), e2)
            ], dim=1)
        )

        n03 = self.n03(
            torch.cat([
                e1,
                n01,
                n02,
                self._align(self.up_l0(n12), e1)
            ], dim=1)
        )

        n04 = self.n04(
            torch.cat([
                e1,
                n01,
                n02,
                n03,
                self._align(self.up_l0(n13), e1)
            ], dim=1)
        )

        seg_out = F.interpolate(
            self.seg_head(n04),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        uncertainty = F.interpolate(
            self.uncertainty_head(n04),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        aux_d3 = F.interpolate(
            self.aux_head_d3(n22),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        aux_d2 = F.interpolate(
            self.aux_head_d2(n13),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        return {
            'seg': seg_out,
            'class': class_out,
            'edge': edge_logits,
            'uncertainty': uncertainty,
            'aux': [aux_d3, aux_d2],
        }


MODEL_REGISTRY['unetplusplus'] = UNetPlusPlus


if __name__ == '__main__':
    m = build_model(
        'unetplusplus',
        num_classes=1,
        num_severity_classes=4
    )

    x = torch.randn(2, 3, 512, 512)

    with torch.no_grad():
        out = m(x)

    print(f"  seg         : {tuple(out['seg'].shape)}")
    print(f"  edge        : {tuple(out['edge'].shape)}")
    print(f"  aux[0]      : {tuple(out['aux'][0].shape)}")
    print(f"  aux[1]      : {tuple(out['aux'][1].shape)}")

[UNetPlusPlus] base_ch=16 | Params: 2,170,681 | Size: 8.3 MB
[unetplusplus] Parameters: 2,170,681  |  Size: 8.3 MB
  seg         : (2, 1, 512, 512)
  edge        : (2, 1, 32, 32)
  aux[0]      : (2, 1, 512, 512)
  aux[1]      : (2, 1, 512, 512)


In [ ]:
class MobileNetUNet(nn.Module):

    def __init__(self, num_classes: int = 1,
                 num_severity_classes: int = 4):
        super().__init__()

        try:
            mobilenet = models.mobilenet_v2(
                weights=models.MobileNet_V2_Weights.IMAGENET1K_V1
            )
        except Exception:
            mobilenet = models.mobilenet_v2(pretrained=True)

        self.encoder0 = mobilenet.features[0:2]
        self.encoder1 = mobilenet.features[2:4]
        self.encoder2 = mobilenet.features[4:7]
        self.encoder3 = mobilenet.features[7:14]
        self.encoder4 = mobilenet.features[14:]

        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            e0 = self.encoder0(dummy)
            e1 = self.encoder1(e0)
            e2 = self.encoder2(e1)
            e3 = self.encoder3(e2)
            e4 = self.encoder4(e3)

        c0, c1, c2, c3, c4 = (
            e0.shape[1], e1.shape[1], e2.shape[1],
            e3.shape[1], e4.shape[1]
        )

        print(f"[MobileNetUNet] Encoder channels: {c0}, {c1}, {c2}, {c3}, {c4}")

        self.decoder1 = DecoderBlock(c4, c3, c3, dropout=0.2)
        self.decoder2 = DecoderBlock(c3, c2, c2, dropout=0.1)
        self.decoder3 = DecoderBlock(c2, c1, c1, dropout=0.1)
        self.decoder4 = DecoderBlock(c1, c0, c0, dropout=0.0)

        self.seg_head = nn.Sequential(
            nn.Conv2d(c0, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1)
        )

        self.edge_head = nn.Sequential(
            nn.Conv2d(c4, 64, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1)
        )

        self.uncertainty_head = nn.Sequential(
            nn.Conv2d(c0, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1)
        )

        self.aux_head_d3 = nn.Conv2d(c1, num_classes, 1)
        self.aux_head_d2 = nn.Conv2d(c2, num_classes, 1)

        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(c4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_severity_classes)
        )

        print(f"[MobileNetUNet] Params: {count_parameters(self):,} | "
              f"Size: {model_size_mb(self):.1f} MB")

    def forward(self, x: torch.Tensor,
                ga_weeks: torch.Tensor = None) -> dict:
        input_size = x.shape[2:]

        e0 = self.encoder0(x)
        e1 = self.encoder1(e0)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)

        edge_logits = self.edge_head(e4)
        class_out = self.classifier(self.global_pool(e4).flatten(1))

        d1 = self.decoder1(e4, e3)
        d2 = self.decoder2(d1, e2)
        d3 = self.decoder3(d2, e1)
        d4 = self.decoder4(d3, e0)

        seg_out = F.interpolate(
            self.seg_head(d4),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        uncertainty = F.interpolate(
            self.uncertainty_head(d4),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        aux_d3 = F.interpolate(
            self.aux_head_d3(d3),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        aux_d2 = F.interpolate(
            self.aux_head_d2(d2),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        return {
            'seg': seg_out,
            'class': class_out,
            'edge': edge_logits,
            'uncertainty': uncertainty,
            'aux': [aux_d3, aux_d2],
        }


MODEL_REGISTRY['mobilenet_unet'] = MobileNetUNet


if __name__ == '__main__':
    m = build_model(
        'mobilenet_unet',
        num_classes=1,
        num_severity_classes=4
    )

    x = torch.randn(2, 3, 512, 512)

    with torch.no_grad():
        out = m(x)

    print(f"  seg         : {tuple(out['seg'].shape)}")
    print(f"  edge        : {tuple(out['edge'].shape)}")
    print(f"  aux[0]      : {tuple(out['aux'][0].shape)}")
    print(f"  aux[1]      : {tuple(out['aux'][1].shape)}")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 151MB/s]


[MobileNetUNet] Encoder channels: 16, 24, 32, 96, 1280
[MobileNetUNet] Params: 4,265,433 | Size: 16.3 MB
[mobilenet_unet] Parameters: 4,265,433  |  Size: 16.3 MB
  seg         : (2, 1, 512, 512)
  edge        : (2, 1, 16, 16)
  aux[0]      : (2, 1, 512, 512)
  aux[1]      : (2, 1, 512, 512)


In [ ]:
class ASPP(nn.Module):

    def __init__(self, in_channels: int, out_channels: int = 128,
                 rates: tuple = (6, 12, 18)):
        super().__init__()

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
            )
        ] + [
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3,
                          padding=r, dilation=r, bias=False),
                nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
            ) for r in rates
        ])

        self.global_branch = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )

        n_branches = len(rates) + 2

        self.project = nn.Sequential(
            nn.Conv2d(out_channels * n_branches, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        feats = [b(x) for b in self.branches]

        feats.append(
            F.interpolate(
                self.global_branch(x),
                size=size,
                mode='bilinear',
                align_corners=True
            )
        )

        return self.project(torch.cat(feats, dim=1))


class DeepLabV3Plus(nn.Module):

    def __init__(self, num_classes: int = 1,
                 num_severity_classes: int = 4,
                 aspp_ch: int = 128):
        super().__init__()

        try:
            mobilenet = models.mobilenet_v2(
                weights=models.MobileNet_V2_Weights.IMAGENET1K_V1
            )
        except Exception:
            mobilenet = models.mobilenet_v2(pretrained=True)

        self.backbone_low = mobilenet.features[0:4]
        self.backbone_high = mobilenet.features[4:14]

        dummy = torch.randn(1, 3, 224, 224)

        with torch.no_grad():
            low_f = self.backbone_low(dummy)
            high_f = self.backbone_high(low_f)

        self.low_ch = low_f.shape[1]
        self.high_ch = high_f.shape[1]

        print(f"[DeepLabV3+] low_ch={self.low_ch}, high_ch={self.high_ch}")

        self.aspp = ASPP(self.high_ch, aspp_ch)

        self.low_proj = nn.Sequential(
            nn.Conv2d(self.low_ch, 32, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.decoder_conv = nn.Sequential(
            nn.Conv2d(aspp_ch + 32, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        self.seg_head = nn.Sequential(
            nn.Conv2d(128, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1)
        )

        self.edge_head = nn.Sequential(
            nn.Conv2d(aspp_ch, 64, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1)
        )

        self.uncertainty_head = nn.Sequential(
            nn.Conv2d(128, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1)
        )

        self.aux_head_d3 = nn.Conv2d(aspp_ch, num_classes, 1)
        self.aux_head_d2 = nn.Conv2d(128, num_classes, 1)

        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(self.high_ch, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_severity_classes)
        )

        print(f"[DeepLabV3+] Params: {count_parameters(self):,} | "
              f"Size: {model_size_mb(self):.1f} MB")

    def forward(self, x: torch.Tensor,
                ga_weeks: torch.Tensor = None) -> dict:
        input_size = x.shape[2:]

        low_feat = self.backbone_low(x)
        high_feat = self.backbone_high(low_feat)

        aspp_out = self.aspp(high_feat)

        edge_logits = self.edge_head(aspp_out)

        class_out = self.classifier(
            self.global_pool(high_feat).flatten(1)
        )

        aux_d3 = F.interpolate(
            self.aux_head_d3(aspp_out),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        aspp_up = F.interpolate(
            aspp_out,
            size=low_feat.shape[2:],
            mode='bilinear',
            align_corners=True
        )

        low_proj = self.low_proj(low_feat)

        dec = self.decoder_conv(
            torch.cat([aspp_up, low_proj], dim=1)
        )

        aux_d2 = F.interpolate(
            self.aux_head_d2(dec),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        seg_out = F.interpolate(
            self.seg_head(dec),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        uncertainty = F.interpolate(
            self.uncertainty_head(dec),
            size=input_size,
            mode='bilinear',
            align_corners=True
        )

        return {
            'seg': seg_out,
            'class': class_out,
            'edge': edge_logits,
            'uncertainty': uncertainty,
            'aux': [aux_d3, aux_d2],
        }


MODEL_REGISTRY['deeplabv3plus'] = DeepLabV3Plus


if __name__ == '__main__':
    m = build_model(
        'deeplabv3plus',
        num_classes=1,
        num_severity_classes=4
    )

    x = torch.randn(2, 3, 512, 512)

    with torch.no_grad():
        out = m(x)

    print(f"  seg         : {tuple(out['seg'].shape)}")
    print(f"  edge        : {tuple(out['edge'].shape)}")
    print(f"  aux[0]      : {tuple(out['aux'][0].shape)}")
    print(f"  aux[1]      : {tuple(out['aux'][1].shape)}")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 24.5MB/s]


[DeepLabV3+] low_ch=24, high_ch=96
[DeepLabV3+] Params: 1,423,753 | Size: 5.4 MB
[deeplabv3plus] Parameters: 1,423,753  |  Size: 5.4 MB
  seg         : (2, 1, 512, 512)
  edge        : (2, 1, 32, 32)
  aux[0]      : (2, 1, 512, 512)
  aux[1]      : (2, 1, 512, 512)


In [ ]:
import torch
import sys
sys.path.append('fetal_vm_segmentation')


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def analyze_model():
    print("=" * 60)
    print("MODEL ARCHITECTURE ANALYSIS")
    print("=" * 60)

    model = MVSegNet(num_classes=1, num_severity_classes=4)

    total_params = count_parameters(model)
    print(f"\n📊 Total Parameters: {total_params:,}")
    print(f"📦 Estimated Size:   {total_params * 4 / 1024 / 1024:.2f} MB")

    if total_params > 3_000_000:
        print(f"\n⚠️  WARNING: Model has too many parameters!")
        print(f"   Expected: ~2.3M, Got: {total_params / 1_000_000:.1f}M")

    print("\n" + "=" * 60)
    print("PARAMETER BREAKDOWN BY COMPONENT:")
    print("=" * 60)

    components = {
        'Encoder (MobileNetV3)': ['encoder'],
        'MSAM (bottleneck)':     ['msam'],
        'MSAM (decoder d1/d2)':  ['msam_d1', 'msam_d2'],
        'ABRB Module':           ['abrb'],
        'FiLM Conditioning':     ['film'],
        'Attention Gates':       ['ag1', 'ag2', 'ag3', 'ag4'],
        'Decoder':               ['decoder'],
        'Segmentation Head':     ['seg_head'],
        'Aux Heads (deep sup.)': ['aux_head'],
        'Uncertainty Head':      ['uncertainty_head'],
        'Classification Head':   ['classifier', 'global_pool'],
    }

    accounted = 0
    for name, keywords in components.items():
        params = sum(
            p.numel() for n, p in model.named_parameters()
            if any(k in n for k in keywords) and p.requires_grad
        )
        accounted += params
        print(f"{name:30s}: {params:>10,} params ({params/1e6:.3f}M)")

    unaccounted = total_params - accounted
    if unaccounted:
        print(f"{'(unaccounted)':30s}: {unaccounted:>10,} params")

    print("\n" + "=" * 60)
    print("TESTING FORWARD PASS:")
    print("=" * 60)

    dummy_input = torch.randn(2, 3, 512, 512)
    ga_weeks = torch.rand(2, 1)

    print(f"Input shape:    {dummy_input.shape}")
    print(f"ga_weeks shape: {ga_weeks.shape}")

    try:
        with torch.no_grad():
            out = model(dummy_input, ga_weeks)

        seg_out = out['seg']
        class_out = out['class']
        edge_map = out['edge']
        uncertainty = out['uncertainty']
        aux = out['aux']

        print("✓ Forward pass successful!")
        print(f"  seg         : {tuple(seg_out.shape)}")
        print(f"  class       : {tuple(class_out.shape)}")
        print(f"  edge        : {tuple(edge_map.shape)}")
        print(f"  uncertainty : {tuple(uncertainty.shape)}")
        print(f"  aux[0] (d3) : {tuple(aux[0].shape)}")
        print(f"  aux[1] (d2) : {tuple(aux[1].shape)}")

        print(f"\n  seg range         : [{seg_out.min():.2f}, {seg_out.max():.2f}]")
        print(f"  class range       : [{class_out.min():.2f}, {class_out.max():.2f}]")
        print(f"  uncertainty range : [{uncertainty.min():.2f}, {uncertainty.max():.2f}]")

    except Exception as e:
        print(f"❌ Forward pass failed: {e}")
        raise

    print("\n" + "=" * 60)
    print("ENCODER ANALYSIS:")
    print("=" * 60)

    with torch.no_grad():
        x = dummy_input
        e0 = model.encoder0(x)
        print(f"encoder0 → {e0.shape}")

        e1 = model.encoder1(e0)
        print(f"encoder1 → {e1.shape}")

        e2 = model.encoder2(e1)
        print(f"encoder2 → {e2.shape}")

        e3 = model.encoder3(e2)
        print(f"encoder3 → {e3.shape}")

        e4 = model.encoder4(e3)
        print(f"encoder4 → {e4.shape}")

    expected = [16, 24, 40, 96, 576]
    actual = [
        e0.shape[1],
        e1.shape[1],
        e2.shape[1],
        e3.shape[1],
        e4.shape[1]
    ]

    print(f"\nExpected channels: {expected}")
    print(f"Actual channels:   {actual}")

    if actual != expected:
        print("⚠️  Channel mismatch — possible MobileNetV3 version issue")
    else:
        print("✓ Channels match")


if __name__ == "__main__":
    analyze_model()

MODEL ARCHITECTURE ANALYSIS
[MVSegNet] Encoder channels: 16, 24, 48, 96, 576

📊 Total Parameters: 2,306,937
📦 Estimated Size:   8.80 MB

PARAMETER BREAKDOWN BY COMPONENT:
Encoder (MobileNetV3)         :    927,008 params (0.927M)
MSAM (bottleneck)             :    482,136 params (0.482M)
MSAM (decoder d1/d2)          :     17,998 params (0.018M)
ABRB Module                   :      4,501 params (0.005M)
FiLM Conditioning             :      2,304 params (0.002M)
Attention Gates               :     37,364 params (0.037M)
Decoder                       :    672,232 params (0.672M)
Segmentation Head             :     27,873 params (0.028M)
Aux Heads (deep sup.)         :         74 params (0.000M)
Uncertainty Head              :      4,705 params (0.005M)
Classification Head           :    148,740 params (0.149M)
(unaccounted)                 :    -17,998 params

TESTING FORWARD PASS:
Input shape:    torch.Size([2, 3, 512, 512])
ga_weeks shape: torch.Size([2, 1])
✓ Forward pass successful!


In [ ]:


import cv2
import numpy as np
import torch

from pathlib import Path
from torch.utils.data import Dataset


class FetalVMDataset(Dataset):
    """
    Segmentation-only dataset.

    Returns:
        image: Tensor [3, H, W]
        mask:  Tensor [1, H, W]

    Does not load:
        - Severity
        - Classification labels
        - Uncertainty targets
        - Gestational age
        - metadata.json
    """

    def __init__(
        self,
        data_dir: Path,
        split: str = "test",
        transform=None,
    ):
        super().__init__()

        self.data_dir = Path(data_dir)
        self.split = split
        self.transform = transform

        self.images_dir = (
            self.data_dir
            / split
            / "images"
        )

        self.masks_dir = (
            self.data_dir
            / split
            / "masks"
        )

        if not self.images_dir.exists():
            raise FileNotFoundError(
                f"Image directory not found: {self.images_dir}"
            )

        if not self.masks_dir.exists():
            raise FileNotFoundError(
                f"Mask directory not found: {self.masks_dir}"
            )

        image_extensions = {
            ".png",
            ".jpg",
            ".jpeg",
        }

        image_paths = sorted(
            path
            for path in self.images_dir.iterdir()
            if path.suffix.lower() in image_extensions
        )

        self.samples = []

        for image_path in image_paths:
            # First try the exact same filename.
            mask_path = (
                self.masks_dir
                / image_path.name
            )

            # Otherwise try a PNG mask with the same stem.
            if not mask_path.exists():
                mask_path = (
                    self.masks_dir
                    / f"{image_path.stem}.png"
                )

            if mask_path.exists():
                self.samples.append(
                    (image_path, mask_path)
                )
            else:
                print(
                    f"Warning: no mask found for "
                    f"{image_path.name}"
                )

        if not self.samples:
            raise RuntimeError(
                f"No valid image-mask pairs found "
                f"for split '{split}'."
            )

        print(
            f"[{split}] Loaded "
            f"{len(self.samples)} image-mask pairs"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, mask_path = self.samples[idx]

        # Read RGB image.
        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )

        if image is None:
            raise RuntimeError(
                f"Could not read image: {image_path}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB,
        )

        # Read binary mask.
        mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE,
        )

        if mask is None:
            raise RuntimeError(
                f"Could not read mask: {mask_path}"
            )

        mask = (
            mask > 127
        ).astype(np.float32)

        # Apply Albumentations transforms.
        if self.transform is not None:
            transformed = self.transform(
                image=image,
                mask=mask,
            )

            image = transformed["image"]
            mask = transformed["mask"]

        else:
            image = torch.from_numpy(
                image.transpose(2, 0, 1)
            ).float() / 255.0

            mask = torch.from_numpy(
                mask
            ).float()

        # Ensure image is a float tensor.
        if not torch.is_tensor(image):
            image = torch.from_numpy(
                np.asarray(image).transpose(2, 0, 1)
            ).float() / 255.0
        else:
            image = image.float()

        # Ensure mask is [1, H, W].
        if not torch.is_tensor(mask):
            mask = torch.from_numpy(
                np.asarray(mask)
            ).float()
        else:
            mask = mask.float()

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        elif mask.ndim == 3 and mask.shape[-1] == 1:
            mask = mask.permute(2, 0, 1)

        return image, mask

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class BoundaryLoss(nn.Module):


    def __init__(self):
        super().__init__()

        sobel_x = torch.tensor(
            [
                [-1, 0, 1],
                [-2, 0, 2],
                [-1, 0, 1],
            ],
            dtype=torch.float32,
        ).view(1, 1, 3, 3)

        sobel_y = torch.tensor(
            [
                [-1, -2, -1],
                [0, 0, 0],
                [1, 2, 1],
            ],
            dtype=torch.float32,
        ).view(1, 1, 3, 3)

        self.register_buffer("sobel_x", sobel_x)
        self.register_buffer("sobel_y", sobel_y)

    def _get_edges(self, x: torch.Tensor) -> torch.Tensor:
        x = x.float()

        sobel_x = self.sobel_x.to(
            device=x.device,
            dtype=x.dtype,
        )
        sobel_y = self.sobel_y.to(
            device=x.device,
            dtype=x.dtype,
        )

        gx = F.conv2d(x, sobel_x, padding=1)
        gy = F.conv2d(x, sobel_y, padding=1)

        return torch.sqrt(gx.square() + gy.square() + 1e-6)

    def forward(
        self,
        pred_logits: torch.Tensor,
        target: torch.Tensor,
    ) -> torch.Tensor:
        pred_prob = torch.sigmoid(pred_logits)
        target = target.float()

        pred_edges = self._get_edges(pred_prob)
        target_edges = self._get_edges(target)

        return F.mse_loss(pred_edges, target_edges)


class TverskyLoss(nn.Module):
    def __init__(
        self,
        alpha: float = 0.3,
        beta: float = 0.7,
        smooth: float = 1.0,
    ):
        super().__init__()

        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(
        self,
        logits: torch.Tensor,
        target: torch.Tensor,
    ) -> torch.Tensor:
        target = target.float()
        probability = torch.sigmoid(logits)

        intersection = (probability * target).sum(dim=(1, 2, 3))
        false_positive = (
            probability * (1.0 - target)
        ).sum(dim=(1, 2, 3))
        false_negative = (
            (1.0 - probability) * target
        ).sum(dim=(1, 2, 3))

        tversky = (
            intersection + self.smooth
        ) / (
            intersection
            + self.alpha * false_positive
            + self.beta * false_negative
            + self.smooth
        )

        return 1.0 - tversky.mean()


class CombinedLossLite(nn.Module):

    def __init__(
        self,
        edge_start_epoch: int = 5,
        boundary_start_epoch: int = 10,
        edge_weight: float = 0.03,
        aux_weight: float = 0.10,
        boundary_weight: float = 0.01,
    ):
        super().__init__()

        self.bce = nn.BCEWithLogitsLoss()
        self.seg_overlap = TverskyLoss(
            alpha=0.55,
            beta=0.45,
        )
        self.edge = nn.BCEWithLogitsLoss()
        self.boundary = BoundaryLoss()

        self.edge_start_epoch = edge_start_epoch
        self.boundary_start_epoch = boundary_start_epoch

        self.edge_weight = edge_weight
        self.aux_weight = aux_weight
        self.boundary_weight = boundary_weight

    def forward(
        self,
        seg_pred: torch.Tensor,
        seg_gt: torch.Tensor,
        edge_pred: torch.Tensor,
        edge_gt: torch.Tensor,
        aux_preds=None,
        epoch: int = 0,
    ):
        seg_gt = seg_gt.float()
        edge_gt = edge_gt.float()

        # Main segmentation loss
        seg_loss = (
            0.5 * self.bce(seg_pred, seg_gt)
            + 0.5 * self.seg_overlap(seg_pred, seg_gt)
        )

        # Edge loss
        edge_loss = self.edge(edge_pred, edge_gt)

        # Auxiliary deep-supervision loss
        aux_loss = seg_pred.new_tensor(0.0)

        if aux_preds is not None:
            aux_weights = [0.4, 0.2]

            for aux_pred, weight in zip(aux_preds, aux_weights):
                aux_target = seg_gt

                if aux_pred.shape[-2:] != seg_gt.shape[-2:]:
                    aux_target = F.interpolate(
                        seg_gt,
                        size=aux_pred.shape[-2:],
                        mode="nearest",
                    )

                aux_loss = (
                    aux_loss
                    + weight
                    * self.seg_overlap(aux_pred, aux_target)
                )

        # Boundary alignment loss
        boundary_loss = self.boundary(seg_pred, seg_gt)

        current_edge_weight = (
            0.0
            if epoch < self.edge_start_epoch
            else self.edge_weight
        )
        current_boundary_weight = (
            0.0
            if epoch < self.boundary_start_epoch
            else self.boundary_weight
        )

        total_loss = (
            seg_loss
            + current_edge_weight * edge_loss
            + self.aux_weight * aux_loss
            + current_boundary_weight * boundary_loss
        )

        return (
            total_loss,
            seg_loss,
            edge_loss,
            aux_loss,
            boundary_loss,
        )

In [ ]:


from pathlib import Path
import json
import random

import numpy as np
import torch
from torch import optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_criterion(model_name: str) -> CombinedLossLite:
    return CombinedLossLite()


def unpack_segmentation_batch(batch):

    if len(batch) == 2:
        images, masks = batch
        ga_weeks = None

    elif len(batch) == 3:
        images, masks, ga_weeks = batch

    elif len(batch) == 4:
        images, masks, _, ga_weeks = batch

    else:
        raise ValueError(
            "Expected a batch with 2, 3, or 4 items, "
            f"but received {len(batch)} items."
        )

    return images, masks, ga_weeks


@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device,
    epoch=0,
    thr=0.5,
):
    model.eval()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0

    for batch in loader:
        images, masks, ga_weeks = unpack_segmentation_batch(batch)

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).float()

        if ga_weeks is not None:
            ga_weeks = ga_weeks.to(device, non_blocking=True).float()

        outputs = model(images, ga_weeks)

        seg_pred = outputs["seg"]
        edge_pred = outputs["edge"]
        aux_preds = outputs.get("aux")

        edge_gt = make_edge_targets(
            masks,
            edge_pred.shape[2:],
        )

        loss, _, _, _, _ = criterion(
            seg_pred=seg_pred,
            seg_gt=masks,
            edge_pred=edge_pred,
            edge_gt=edge_gt,
            aux_preds=aux_preds,
            epoch=epoch,
        )

        total_loss += loss.item()
        total_dice += dice_score(
            seg_pred,
            masks,
            thr=thr,
        )
        total_iou += iou_score(
            seg_pred,
            masks,
            thr=thr,
        )

    num_batches = max(len(loader), 1)

    return (
        total_loss / num_batches,
        total_dice / num_batches,
        total_iou / num_batches,
    )


def train_one_run(
    model_name: str,
    run_id: int,
    seed: int,
    cfg: dict,
):
    tag = f"{model_name}_run{run_id}"

    print(f"\n{'=' * 65}")
    print(f"  {tag.upper()}  |  seed={seed}")
    print(f"{'=' * 65}")

    set_seed(seed)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    print(f"  Device : {device}")

    # Paths
    base = Path(cfg["base_dir"])
    data_dir = base / "data" / "preprocessed"
    models_dir = base / "models"
    results_dir = base / "results"

    models_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    if not data_dir.exists():
        print(f"  ERROR: data not found at {data_dir}")
        return None

    # Datasets
    train_tf, val_tf = get_transforms()

    train_ds = FetalVMDataset(
        data_dir,
        "train",
        train_tf,
    )
    val_ds = FetalVMDataset(
        data_dir,
        "val",
        val_tf,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        generator=generator,
        worker_init_fn=seed_worker,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg["batch_size"],
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        worker_init_fn=seed_worker,
    )

    print(
        f"  Train : {len(train_ds)} samples  |  "
        f"Val : {len(val_ds)} samples"
    )


    model = build_model(
        name=model_name,
        num_classes=1,
    ).to(device)

    criterion = get_criterion(model_name).to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
    )

    if model_name == "unet":
        scheduler = OneCycleLR(
            optimizer,
            max_lr=1e-4,
            epochs=cfg["epochs"],
            steps_per_epoch=len(train_loader),
            pct_start=0.1,
            anneal_strategy="cos",
            div_factor=10,
            final_div_factor=1e3,
        )
    else:
        scheduler = OneCycleLR(
            optimizer,
            max_lr=1e-3,
            epochs=cfg["epochs"],
            steps_per_epoch=len(train_loader),
            pct_start=0.1,
            anneal_strategy="cos",
            div_factor=25,
            final_div_factor=1e4,
        )

    scaler = (
        torch.cuda.amp.GradScaler()
        if device.type == "cuda"
        else None
    )

    best_dice = 0.0

    history = {
        key: []
        for key in [
            "train_loss",
            "train_dice",
            "train_iou",
            "val_loss",
            "val_dice",
            "val_iou",
            "lr",
        ]
    }

    for epoch in range(cfg["epochs"]):
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"\n  Epoch {epoch + 1}/{cfg['epochs']}  "
            f"[lr={current_lr:.2e}]"
        )


        train_loss, train_dice, train_iou = train_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            scheduler=scheduler,
            epoch=epoch,
        )

        val_loss, val_dice, val_iou = validate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            epoch=epoch,
        )

        history["train_loss"].append(train_loss)
        history["train_dice"].append(train_dice)
        history["train_iou"].append(train_iou)

        history["val_loss"].append(val_loss)
        history["val_dice"].append(val_dice)
        history["val_iou"].append(val_iou)

        history["lr"].append(
            optimizer.param_groups[0]["lr"]
        )

        print(
            f"  Train  loss={train_loss:.4f}  "
            f"dice={train_dice:.4f}  "
            f"iou={train_iou:.4f}"
        )
        print(
            f"  Val    loss={val_loss:.4f}  "
            f"dice={val_dice:.4f}  "
            f"iou={val_iou:.4f}"
        )

        if val_dice > best_dice:
            best_dice = val_dice

            checkpoint = {
                "epoch": epoch,
                "model_name": model_name,
                "run_id": run_id,
                "seed": seed,
                "model_state_dict": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "best_dice": best_dice,
                "history": history,
            }

            torch.save(
                checkpoint,
                models_dir / f"{tag}_best.pth",
            )

            print(
                f"  ✓ Best model saved "
                f"(dice={best_dice:.4f})"
            )

    history_path = results_dir / f"{tag}_history.json"

    with open(history_path, "w", encoding="utf-8") as file:
        json.dump(history, file, indent=2)

    print(
        f"\n  Run complete. "
        f"Best val Dice = {best_dice:.4f}"
    )

    return best_dice

In [ ]:


import json
import random
from contextlib import nullcontext
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim

from albumentations.pytorch import ToTensorV2
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader
from tqdm import tqdm



SEEDS = {
    0: 42,
    1: 43,
    2: 44,
}

EXPERIMENT_CFG = {
    "base_dir": "fetal_vm_segmentation",
    "batch_size": 8,
    "epochs": 50,
    "lr": 1e-3,
    "num_workers": 4,
    "image_size": 512,
    "weight_decay": 1e-4,
    "amp": True,
    "model_name": "mvsegnet",
}




def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int):
    worker_seed = torch.initial_seed() % (2**32)

    np.random.seed(worker_seed)
    random.seed(worker_seed)



def unpack_segmentation_batch(batch):


    if isinstance(batch, dict):
        images = batch["image"]
        masks = batch["mask"]
        ga_weeks = batch.get("ga_weeks")

        return images, masks, ga_weeks

    if len(batch) == 2:
        images, masks = batch
        ga_weeks = None

    elif len(batch) == 3:
        images, masks, ga_weeks = batch

    elif len(batch) == 4:
        images, masks, _, ga_weeks = batch

    else:
        raise ValueError(
            "Dataset batch must contain 2, 3, or 4 items. "
            f"Received {len(batch)} items."
        )

    return images, masks, ga_weeks


def prepare_batch(batch, device):
    images, masks, ga_weeks = unpack_segmentation_batch(batch)

    images = images.to(
        device,
        non_blocking=True,
    )

    masks = masks.to(
        device,
        non_blocking=True,
    ).float()

    if masks.ndim == 3:
        masks = masks.unsqueeze(1)

    if ga_weeks is not None:
        ga_weeks = ga_weeks.to(
            device,
            non_blocking=True,
        ).float()

        if ga_weeks.ndim == 1:
            ga_weeks = ga_weeks.unsqueeze(1)

    return images, masks, ga_weeks




@torch.no_grad()
def dice_score(
    pred_logits: torch.Tensor,
    target: torch.Tensor,
    thr: float = 0.5,
) -> float:

    target = target.float()

    prediction = (
        torch.sigmoid(pred_logits) > thr
    ).float()

    intersection = (
        prediction * target
    ).sum(dim=(1, 2, 3))

    denominator = (
        prediction.sum(dim=(1, 2, 3))
        + target.sum(dim=(1, 2, 3))
    )

    dice = (
        2.0 * intersection + 1e-8
    ) / (
        denominator + 1e-8
    )

    return dice.mean().item()


@torch.no_grad()
def iou_score(
    pred_logits: torch.Tensor,
    target: torch.Tensor,
    thr: float = 0.5,
) -> float:

    target = target.float()

    prediction = (
        torch.sigmoid(pred_logits) > thr
    ).float()

    intersection = (
        prediction * target
    ).sum(dim=(1, 2, 3))

    union = (
        prediction.sum(dim=(1, 2, 3))
        + target.sum(dim=(1, 2, 3))
        - intersection
    )

    iou = (
        intersection + 1e-8
    ) / (
        union + 1e-8
    )

    return iou.mean().item()



@torch.no_grad()
def make_edge_targets(
    masks: torch.Tensor,
    edge_size=None,
) -> torch.Tensor:
    """
    Creates morphological boundary targets.

    Args:
        masks:
            [B, 1, H, W], values in {0, 1}

        edge_size:
            Optional target spatial size.
    """

    masks = masks.float()

    dilated = F.max_pool2d(
        masks,
        kernel_size=3,
        stride=1,
        padding=1,
    )

    eroded = -F.max_pool2d(
        -masks,
        kernel_size=3,
        stride=1,
        padding=1,
    )

    edges = (
        (dilated - eroded) > 0
    ).float()

    if (
        edge_size is not None
        and edges.shape[-2:] != tuple(edge_size)
    ):
        edges = F.interpolate(
            edges,
            size=edge_size,
            mode="nearest",
        )

    return edges




def get_transforms():

    train_transform = A.Compose(
        [
            A.HorizontalFlip(p=0.5),

            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.10,
                rotate_limit=10,
                border_mode=cv2.BORDER_CONSTANT,
                p=0.5,
            ),

            A.RandomBrightnessContrast(
                brightness_limit=0.15,
                contrast_limit=0.15,
                p=0.3,
            ),

            A.GaussNoise(p=0.15),

            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),

            ToTensorV2(),
        ]
    )

    val_transform = A.Compose(
        [
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),

            ToTensorV2(),
        ]
    )

    return train_transform, val_transform


def get_criterion(model_name: str):
    del model_name

    return CombinedLossLite()



def create_grad_scaler(
    device: torch.device,
    enabled: bool,
):
    use_amp = (
        enabled
        and device.type == "cuda"
    )

    if not use_amp:
        return None

    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=True,
        )

    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(
            enabled=True,
        )


def autocast_context(
    device: torch.device,
    enabled: bool,
):
    use_amp = (
        enabled
        and device.type == "cuda"
    )

    if not use_amp:
        return nullcontext()

    return torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=True,
    )


def train_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    scaler=None,
    scheduler=None,
    epoch: int = 0,
    use_amp: bool = True,
):

    model.train()

    totals = {
        "loss": 0.0,
        "seg_loss": 0.0,
        "edge_loss": 0.0,
        "aux_loss": 0.0,
        "boundary_loss": 0.0,
        "dice": 0.0,
        "iou": 0.0,
    }

    progress_bar = tqdm(
        loader,
        desc="  Train",
        leave=False,
    )

    for batch in progress_bar:

        images, masks, ga_weeks = prepare_batch(
            batch,
            device,
        )

        optimizer.zero_grad(
            set_to_none=True,
        )

        with autocast_context(
            device,
            enabled=use_amp,
        ):

            outputs = model(
                images,
                ga_weeks,
            )

            seg_pred = outputs["seg"]
            edge_pred = outputs["edge"]
            aux_preds = outputs.get("aux")

            edge_gt = make_edge_targets(
                masks,
                edge_pred.shape[-2:],
            )

            (
                loss,
                seg_loss,
                edge_loss,
                aux_loss,
                boundary_loss,
            ) = criterion(
                seg_pred=seg_pred,
                seg_gt=masks,
                edge_pred=edge_pred,
                edge_gt=edge_gt,
                aux_preds=aux_preds,
                epoch=epoch,
            )

        if scaler is not None:

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            scaler.step(optimizer)
            scaler.update()

        else:

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()


        if scheduler is not None:
            scheduler.step()

        batch_dice = dice_score(
            seg_pred.detach(),
            masks,
        )

        batch_iou = iou_score(
            seg_pred.detach(),
            masks,
        )

        totals["loss"] += loss.item()
        totals["seg_loss"] += seg_loss.item()
        totals["edge_loss"] += edge_loss.item()
        totals["aux_loss"] += aux_loss.item()
        totals["boundary_loss"] += boundary_loss.item()
        totals["dice"] += batch_dice
        totals["iou"] += batch_iou

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            dice=f"{batch_dice:.4f}",
        )

    num_batches = max(
        len(loader),
        1,
    )

    return {
        key: value / num_batches
        for key, value in totals.items()
    }



@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device,
    epoch: int = 0,
    thr: float = 0.5,
    use_amp: bool = True,
):

    model.eval()

    totals = {
        "loss": 0.0,
        "seg_loss": 0.0,
        "edge_loss": 0.0,
        "aux_loss": 0.0,
        "boundary_loss": 0.0,
        "dice": 0.0,
        "iou": 0.0,
    }

    for batch in loader:

        images, masks, ga_weeks = prepare_batch(
            batch,
            device,
        )

        with autocast_context(
            device,
            enabled=use_amp,
        ):

            outputs = model(
                images,
                ga_weeks,
            )

            seg_pred = outputs["seg"]
            edge_pred = outputs["edge"]
            aux_preds = outputs.get("aux")

            edge_gt = make_edge_targets(
                masks,
                edge_pred.shape[-2:],
            )

            (
                loss,
                seg_loss,
                edge_loss,
                aux_loss,
                boundary_loss,
            ) = criterion(
                seg_pred=seg_pred,
                seg_gt=masks,
                edge_pred=edge_pred,
                edge_gt=edge_gt,
                aux_preds=aux_preds,
                epoch=epoch,
            )

        totals["loss"] += loss.item()
        totals["seg_loss"] += seg_loss.item()
        totals["edge_loss"] += edge_loss.item()
        totals["aux_loss"] += aux_loss.item()
        totals["boundary_loss"] += boundary_loss.item()

        totals["dice"] += dice_score(
            seg_pred,
            masks,
            thr=thr,
        )

        totals["iou"] += iou_score(
            seg_pred,
            masks,
            thr=thr,
        )

    num_batches = max(
        len(loader),
        1,
    )

    results = {
        key: value / num_batches
        for key, value in totals.items()
    }

    results["thr"] = float(thr)

    return results


@torch.no_grad()
def find_best_threshold(
    model,
    loader,
    device,
    thresholds=None,
    use_amp: bool = True,
):

    if thresholds is None:
        thresholds = np.arange(
            0.30,
            0.71,
            0.05,
        ).tolist()

    model.eval()

    best_threshold = 0.5
    best_dice = -1.0

    for threshold in thresholds:

        total_dice = 0.0
        num_batches = 0

        for batch in loader:

            images, masks, ga_weeks = prepare_batch(
                batch,
                device,
            )

            with autocast_context(
                device,
                enabled=use_amp,
            ):

                outputs = model(
                    images,
                    ga_weeks,
                )

                seg_pred = outputs["seg"]

            total_dice += dice_score(
                seg_pred,
                masks,
                thr=float(threshold),
            )

            num_batches += 1

        average_dice = (
            total_dice
            / max(num_batches, 1)
        )

        if average_dice > best_dice:
            best_dice = average_dice
            best_threshold = float(threshold)

    return best_threshold, best_dice



def train_one_run(
    model_name: str,
    run_id: int,
    seed: int,
    cfg: dict,
):

    tag = f"{model_name}_run{run_id}"

    print(f"\n{'=' * 65}")
    print(f"  {tag.upper()} | seed={seed}")
    print(f"{'=' * 65}")

    set_seed(seed)

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    use_amp = (
        bool(cfg.get("amp", True))
        and device.type == "cuda"
    )

    print(f"  Device : {device}")
    print(f"  AMP    : {use_amp}")



    base_dir = Path(
        cfg["base_dir"]
    )

    data_dir = (
        base_dir
        / "data"
        / "preprocessed"
    )

    models_dir = (
        base_dir
        / "models"
    )

    results_dir = (
        base_dir
        / "results"
    )

    models_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    results_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    if not data_dir.exists():
        raise FileNotFoundError(
            f"Preprocessed data not found: {data_dir}"
        )


    train_transform, val_transform = get_transforms()

    train_dataset = FetalVMDataset(
        data_dir,
        "train",
        train_transform,
    )

    val_dataset = FetalVMDataset(
        data_dir,
        "val",
        val_transform,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    num_workers = cfg.get(
        "num_workers",
        4,
    )

    pin_memory = (
        device.type == "cuda"
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
        generator=generator,
        worker_init_fn=seed_worker,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg["batch_size"],
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
        worker_init_fn=seed_worker,
    )

    print(
        f"  Train : {len(train_dataset)} samples | "
        f"Val : {len(val_dataset)} samples"
    )



    model = build_model(
        name=model_name,
        num_classes=1,
    ).to(device)

    criterion = get_criterion(
        model_name,
    ).to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
    )

    scheduler = OneCycleLR(
        optimizer,
        max_lr=cfg["lr"],
        epochs=cfg["epochs"],
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        anneal_strategy="cos",
        div_factor=25,
        final_div_factor=1e4,
    )

    scaler = create_grad_scaler(
        device,
        enabled=use_amp,
    )


    best_dice = -1.0
    best_threshold = 0.5

    history = {
        "train_loss": [],
        "train_seg_loss": [],
        "train_edge_loss": [],
        "train_aux_loss": [],
        "train_boundary_loss": [],
        "train_dice": [],
        "train_iou": [],

        "val_loss": [],
        "val_seg_loss": [],
        "val_edge_loss": [],
        "val_aux_loss": [],
        "val_boundary_loss": [],
        "val_dice": [],
        "val_iou": [],

        "lr": [],
    }

    checkpoint_path = (
        models_dir
        / f"{tag}_best.pth"
    )


    for epoch in range(
        cfg["epochs"]
    ):

        current_lr = (
            optimizer.param_groups[0]["lr"]
        )

        print(
            f"\n  Epoch {epoch + 1}/{cfg['epochs']} "
            f"[lr={current_lr:.2e}]"
        )

        train_metrics = train_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            scheduler=scheduler,
            epoch=epoch,
            use_amp=use_amp,
        )

        val_metrics = validate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            epoch=epoch,
            thr=0.5,
            use_amp=use_amp,
        )

        for metric_name in [
            "loss",
            "seg_loss",
            "edge_loss",
            "aux_loss",
            "boundary_loss",
            "dice",
            "iou",
        ]:
            history[
                f"train_{metric_name}"
            ].append(
                train_metrics[metric_name]
            )

            history[
                f"val_{metric_name}"
            ].append(
                val_metrics[metric_name]
            )

        history["lr"].append(
            optimizer.param_groups[0]["lr"]
        )

        print(
            f"  Train  "
            f"loss={train_metrics['loss']:.4f}  "
            f"dice={train_metrics['dice']:.4f}  "
            f"iou={train_metrics['iou']:.4f}"
        )

        print(
            f"  Val    "
            f"loss={val_metrics['loss']:.4f}  "
            f"dice={val_metrics['dice']:.4f}  "
            f"iou={val_metrics['iou']:.4f}"
        )

        if val_metrics["dice"] > best_dice:

            best_dice = val_metrics["dice"]

            checkpoint = {
                "epoch": epoch,
                "model_name": model_name,
                "run_id": run_id,
                "seed": seed,

                "model_state_dict": (
                    model.state_dict()
                ),

                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),

                "scheduler_state_dict": (
                    scheduler.state_dict()
                ),

                "best_dice": best_dice,
                "threshold": 0.5,
                "history": history,
                "config": cfg,
            }

            torch.save(
                checkpoint,
                checkpoint_path,
            )

            print(
                f"  ✓ Best model saved "
                f"(dice={best_dice:.4f})"
            )


    if checkpoint_path.exists():

        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
        )

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        (
            best_threshold,
            threshold_dice,
        ) = find_best_threshold(
            model=model,
            loader=val_loader,
            device=device,
            use_amp=use_amp,
        )

        checkpoint["threshold"] = best_threshold
        checkpoint["threshold_dice"] = threshold_dice

        torch.save(
            checkpoint,
            checkpoint_path,
        )

        print(
            f"  Best threshold={best_threshold:.2f} | "
            f"Dice={threshold_dice:.4f}"
        )



    history_path = (
        results_dir
        / f"{tag}_history.json"
    )

    with open(
        history_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            history,
            file,
            indent=2,
        )

    print(
        f"\n  Run complete. "
        f"Best validation Dice={best_dice:.4f}"
    )

    return {
        "best_dice": best_dice,
        "best_threshold": best_threshold,
        "checkpoint": str(checkpoint_path),
        "history": str(history_path),
    }


In [ ]:


import numpy as np
import torch


MODEL_NAME = "mvsegnet"

RESULTS = {}


for run_id in range(3):

    seed = SEEDS[run_id]

    print(
        f"\n🚀 Starting {MODEL_NAME.upper()} "
        f"Run {run_id + 1}/3 "
        f"(seed={seed})"
    )

    run_result = train_one_run(
        model_name=MODEL_NAME,
        run_id=run_id,
        seed=seed,
        cfg=EXPERIMENT_CFG,
    )

    # train_one_run returns a dictionary.
    best_dice = run_result["best_dice"]

    RESULTS[f"run{run_id}"] = best_dice

    print(
        f"✅ Run {run_id + 1} done "
        f"→ Best Dice = {best_dice:.4f}"
    )




print(f"\n{'=' * 50}")
print(f"  {MODEL_NAME.upper()} — 3-Run Summary")
print(f"{'=' * 50}")

dices = list(
    RESULTS.values()
)

for run_name, dice in RESULTS.items():
    print(
        f"  {run_name}: {dice:.4f}"
    )

mean_dice = np.mean(dices)
std_dice = np.std(dices)
best_dice = max(dices)

print(
    f"\n  Mean ± Std : "
    f"{mean_dice:.4f} ± {std_dice:.4f}"
)

print(
    f"  Best       : "
    f"{best_dice:.4f}"
)


🚀 Starting MVSEGNET Run 1/3 (seed=42)

  MVSEGNET_RUN0 | seed=42
  Device : cuda
  AMP    : True
[train] Loaded 235 image-mask pairs
[val] Loaded 57 image-mask pairs
  Train : 235 samples | Val : 57 samples


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[MVSegNet] Encoder channels: 16, 24, 48, 96, 576
[mvsegnet] Parameters: 1,665,164  |  Size: 6.4 MB

  Epoch 1/50 [lr=4.00e-05]


  Train:   0%|          | 0/30 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  Train  loss=0.9264  dice=0.0106  iou=0.0053
  Val    loss=0.8789  dice=0.0119  iou=0.0060
  ✓ Best model saved (dice=0.0119)

  Epoch 2/50 [lr=1.33e-04]


...

  Train  loss=0.1100  dice=0.8264  iou=0.7150
  Val    loss=0.1302  dice=0.7902  iou=0.6750

  Epoch 47/50 [lr=1.91e-05]


  Train  loss=0.1083  dice=0.8303  iou=0.7181
  Val    loss=0.1311  dice=0.7878  iou=0.6726

  Epoch 48/50 [lr=1.07e-05]


  Train  loss=0.1119  dice=0.8232  iou=0.7110
  Val    loss=0.1288  dice=0.7932  iou=0.6783

  Epoch 49/50 [lr=4.71e-06]


  Train  loss=0.1142  dice=0.8195  iou=0.7090
  Val    loss=0.1297  dice=0.7911  iou=0.6765

  Epoch 50/50 [lr=1.14e-06]


  Train  loss=0.1130  dice=0.8217  iou=0.7090
  Val    loss=0.1292  dice=0.7923  iou=0.6776
  Best threshold=0.50 | Dice=0.8120

  Run complete. Best validation Dice=0.8120
✅ Run 3 done → Best Dice = 0.8120

  MVSEGNET — 3-Run Summary
  run0: 0.8089
  run1: 0.7945
  run2: 0.8120

  Mean ± Std : 0.8051 ± 0.0077
  Best       : 0.8120


In [ ]:
print("MODEL_NAME =", MODEL_NAME)
print("MODEL_REGISTRY =", MODEL_REGISTRY)

MODEL_NAME = mvsegnet
MODEL_REGISTRY = {'mvsegnet': <class '__main__.MVSegNet'>}


In [ ]:


import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from scipy.ndimage import binary_erosion, distance_transform_edt
from torch.utils.data import DataLoader
from tqdm import tqdm


def count_parameters(model: nn.Module) -> int:
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


def model_size_mb(model: nn.Module) -> float:
    return count_parameters(model) * 4 / 1024 / 1024


def unpack_evaluation_batch(batch):
    """
    Supported formats:

        (images, masks)
        (images, masks, ga_weeks)

    Legacy compatibility:

        (images, masks, severities, ga_weeks)

    The severity item in the legacy format is discarded.
    """

    if isinstance(batch, dict):
        images = batch["image"]
        masks = batch["mask"]
        ga_weeks = batch.get("ga_weeks")
        return images, masks, ga_weeks

    if len(batch) == 2:
        images, masks = batch
        ga_weeks = None

    elif len(batch) == 3:
        images, masks, ga_weeks = batch

    elif len(batch) == 4:
        images, masks, _, ga_weeks = batch

    else:
        raise ValueError(
            "Expected an evaluation batch with 2, 3, or 4 items, "
            f"but received {len(batch)} items."
        )

    return images, masks, ga_weeks




def _surface_distances(
    pred_np: np.ndarray,
    target_np: np.ndarray,
    px_mm: float,
) -> tuple:


    pred_bool = pred_np.astype(bool)
    target_bool = target_np.astype(bool)

    pred_surface = (
        pred_bool
        ^ binary_erosion(
            pred_bool,
            iterations=1,
            border_value=0,
        )
    )

    target_surface = (
        target_bool
        ^ binary_erosion(
            target_bool,
            iterations=1,
            border_value=0,
        )
    )

    if not pred_surface.any():
        pred_surface = pred_bool

    if not target_surface.any():
        target_surface = target_bool

    # Distance to the opposite surface, not merely to the opposite mask.
    distance_to_target_surface = (
        distance_transform_edt(~target_surface)
        * px_mm
    )

    distance_to_pred_surface = (
        distance_transform_edt(~pred_surface)
        * px_mm
    )

    pred_to_target = distance_to_target_surface[pred_surface]
    target_to_pred = distance_to_pred_surface[target_surface]

    return pred_to_target, target_to_pred


def compute_hd95(
    pred_np: np.ndarray,
    target_np: np.ndarray,
    px_mm: float,
) -> float:
    """
    Symmetric 95th-percentile Hausdorff distance in millimetres.

    Returns NaN when either mask is empty.
    """

    if pred_np.sum() == 0 or target_np.sum() == 0:
        return float("nan")

    distance_1, distance_2 = _surface_distances(
        pred_np,
        target_np,
        px_mm,
    )

    distances = np.concatenate(
        [distance_1, distance_2]
    )

    return float(
        np.percentile(distances, 95)
    )


def compute_asd(
    pred_np: np.ndarray,
    target_np: np.ndarray,
    px_mm: float,
) -> float:
    """
    Average symmetric surface distance in millimetres.

    Returns NaN when either mask is empty.
    """

    if pred_np.sum() == 0 or target_np.sum() == 0:
        return float("nan")

    distance_1, distance_2 = _surface_distances(
        pred_np,
        target_np,
        px_mm,
    )

    return float(
        (
            distance_1.mean()
            + distance_2.mean()
        )
        / 2.0
    )


def compute_ventricle_width_mm(
    mask_np: np.ndarray,
    px_mm: float,
) -> float:
    """
    Computes the maximum horizontal mask extent in millimetres.

    This is a geometric width estimate, not a severity calculation.
    """

    mask_bool = mask_np.astype(bool)

    if mask_bool.sum() == 0:
        return 0.0

    occupied_columns = np.where(
        mask_bool.any(axis=0)
    )[0]

    if len(occupied_columns) == 0:
        return 0.0

    width_pixels = (
        occupied_columns[-1]
        - occupied_columns[0]
        + 1
    )

    return float(
        width_pixels * px_mm
    )


def compute_all_seg_metrics(
    pred_sigmoid: np.ndarray,
    target: np.ndarray,
    px_mm: float,
    threshold: float = 0.5,
) -> dict:
    """
    Computes all segmentation metrics for one sample.
    """

    pred_binary = (
        pred_sigmoid > threshold
    ).astype(np.float32)

    target = (
        target > 0.5
    ).astype(np.float32)

    intersection = (
        pred_binary * target
    ).sum()

    pred_sum = pred_binary.sum()
    target_sum = target.sum()

    dice = (
        2.0 * intersection
    ) / (
        pred_sum + target_sum + 1e-8
    )

    union = (
        pred_sum
        + target_sum
        - intersection
    )

    iou = (
        intersection
    ) / (
        union + 1e-8
    )

    true_positive = intersection

    false_positive = (
        pred_binary * (1.0 - target)
    ).sum()

    false_negative = (
        (1.0 - pred_binary) * target
    ).sum()

    precision = (
        true_positive
    ) / (
        true_positive
        + false_positive
        + 1e-8
    )

    recall = (
        true_positive
    ) / (
        true_positive
        + false_negative
        + 1e-8
    )

    pred_bool = pred_binary.astype(bool)
    target_bool = target.astype(bool)

    hd95 = compute_hd95(
        pred_bool,
        target_bool,
        px_mm,
    )

    asd = compute_asd(
        pred_bool,
        target_bool,
        px_mm,
    )

    pred_width = compute_ventricle_width_mm(
        pred_bool,
        px_mm,
    )

    gt_width = compute_ventricle_width_mm(
        target_bool,
        px_mm,
    )

    width_absolute_error = abs(
        pred_width - gt_width
    )

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "hd95": float(hd95),
        "asd": float(asd),
        "width_ae": float(width_absolute_error),
        "pred_width": float(pred_width),
        "gt_width": float(gt_width),
    }


# ============================================================
# EVALUATE ONE CHECKPOINT
# ============================================================

@torch.no_grad()
def evaluate_checkpoint(
    model: nn.Module,
    loader: DataLoader,
    device,
    px_mm: float,
    threshold: float = 0.5,
    max_visualization_samples: int = 12,
) -> dict:
    """
    Evaluates one segmentation checkpoint.

    Returns:
        - aggregated segmentation metrics
        - per-image inference timing
        - visualization samples
        - per-sample metrics

    No severity or classification metric is calculated.
    """

    model.eval()

    all_metrics = []
    inference_times = []
    visualization_samples = []

    for batch in tqdm(
        loader,
        desc="    Evaluating",
        leave=False,
    ):
        images, masks, ga_weeks = prepare_batch(
              batch,
              device,
          )

        if device.type == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        outputs = model(
            images,
            ga_weeks,
        )

        seg_logits = outputs["seg"]

        if device.type == "cuda":
            torch.cuda.synchronize()

        end_time = time.perf_counter()

        seconds_per_image = (
            end_time - start_time
        ) / images.size(0)

        inference_times.extend(
            [seconds_per_image] * images.size(0)
        )

        seg_probabilities = (
            torch.sigmoid(seg_logits)
            .detach()
            .cpu()
            .numpy()
        )

        masks_np = (
            masks.detach()
            .cpu()
            .numpy()
        )

        images_cpu = (
            images.detach()
            .cpu()
        )

        for sample_index in range(
            images.size(0)
        ):
            sample_metrics = compute_all_seg_metrics(
                pred_sigmoid=seg_probabilities[
                    sample_index,
                    0,
                ],
                target=masks_np[
                    sample_index,
                    0,
                ],
                px_mm=px_mm,
                threshold=threshold,
            )

            all_metrics.append(
                sample_metrics
            )

            if (
                len(visualization_samples)
                < max_visualization_samples
            ):
                visualization_samples.append(
                    {
                        "image": images_cpu[
                            sample_index
                        ],
                        "mask_gt": masks_np[
                            sample_index,
                            0,
                        ],
                        "mask_pred": (
                            seg_probabilities[
                                sample_index,
                                0,
                            ]
                            > threshold
                        ).astype(np.float32),
                    }
                )

    def nanmean_std(metric_name):
        values = np.asarray(
            [
                metric[metric_name]
                for metric in all_metrics
            ],
            dtype=np.float64,
        )

        valid_values = values[
            ~np.isnan(values)
        ]

        if valid_values.size == 0:
            return (
                float("nan"),
                float("nan"),
            )

        return (
            float(valid_values.mean()),
            float(valid_values.std()),
        )

    segmentation_summary = {}

    metric_names = [
        "dice",
        "iou",
        "precision",
        "recall",
        "hd95",
        "asd",
        "width_ae",
    ]

    for metric_name in metric_names:
        mean_value, std_value = nanmean_std(
            metric_name
        )

        segmentation_summary[
            f"{metric_name}_mean"
        ] = mean_value

        segmentation_summary[
            f"{metric_name}_std"
        ] = std_value

    if inference_times:
        mean_ms = (
            float(np.mean(inference_times))
            * 1000.0
        )

        std_ms = (
            float(np.std(inference_times))
            * 1000.0
        )
    else:
        mean_ms = float("nan")
        std_ms = float("nan")

    fps = (
        1000.0 / mean_ms
        if mean_ms > 0
        else 0.0
    )

    return {
        "seg": segmentation_summary,
        "timing": {
            "mean_ms": mean_ms,
            "std_ms": std_ms,
            "fps": fps,
        },
        "samples": visualization_samples,
        "per_sample": all_metrics,
        "threshold": float(threshold),
        "px_mm": float(px_mm),
    }


# ============================================================
# GFLOPS
# ============================================================

def compute_gflops(
    model: nn.Module,
    input_size=(1, 3, 512, 512),
    device="cpu",
) -> float:
    """
    Estimates GFLOPs with forward hooks.

    Uses:
        1 MAC = 2 FLOPs.
    """

    device = torch.device(device)

    flops_total = [0]
    hooks = []

    def conv_hook(module, inputs, output):
        batch_size, out_channels, out_height, out_width = (
            output.shape
        )

        in_channels_per_group = (
            inputs[0].shape[1]
            // module.groups
        )

        kernel_height, kernel_width = (
            module.kernel_size
        )

        flops_total[0] += (
            2
            * batch_size
            * out_channels
            * out_height
            * out_width
            * in_channels_per_group
            * kernel_height
            * kernel_width
        )

    def linear_hook(module, inputs, output):
        input_tensor = inputs[0]

        if input_tensor.dim() == 2:
            item_count = input_tensor.shape[0]
        else:
            item_count = int(
                np.prod(
                    input_tensor.shape[:-1]
                )
            )

        flops_total[0] += (
            2
            * item_count
            * module.in_features
            * module.out_features
        )

    def mha_hook(module, inputs, output):
        try:
            input_tensor = inputs[0]

            if input_tensor.dim() != 3:
                return

            if module.batch_first:
                batch_size, sequence_length, _ = (
                    input_tensor.shape
                )
            else:
                sequence_length, batch_size, _ = (
                    input_tensor.shape
                )

            embed_dim = module.embed_dim

            flops_total[0] += (
                2
                * batch_size
                * (
                    3
                    * sequence_length
                    * embed_dim
                    * embed_dim
                    + sequence_length
                    * sequence_length
                    * embed_dim
                    + sequence_length
                    * embed_dim
                    * embed_dim
                )
            )

        except Exception:
            pass

    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            hooks.append(
                module.register_forward_hook(
                    conv_hook
                )
            )

        elif isinstance(module, nn.Linear):
            hooks.append(
                module.register_forward_hook(
                    linear_hook
                )
            )

        elif isinstance(
            module,
            nn.MultiheadAttention,
        ):
            hooks.append(
                module.register_forward_hook(
                    mha_hook
                )
            )

    dummy_input = torch.zeros(
        input_size,
        device=device,
    )

    model.eval()

    with torch.no_grad():
        model(dummy_input)

    for hook in hooks:
        hook.remove()

    return (
        flops_total[0]
        / 1e9
    )


def aggregate_runs(
    run_results: list,
) -> dict:


    metric_names = [
        "dice",
        "iou",
        "precision",
        "recall",
        "hd95",
        "asd",
        "width_ae",
    ]

    aggregated_segmentation = {}

    for metric_name in metric_names:
        per_run_means = []

        for run_result in run_results:
            value = run_result["seg"].get(
                f"{metric_name}_mean",
                float("nan"),
            )

            if not np.isnan(value):
                per_run_means.append(
                    value
                )

        if per_run_means:
            aggregated_segmentation[
                f"{metric_name}_mean"
            ] = float(
                np.mean(per_run_means)
            )

            aggregated_segmentation[
                f"{metric_name}_std"
            ] = float(
                np.std(per_run_means)
            )

        else:
            aggregated_segmentation[
                f"{metric_name}_mean"
            ] = float("nan")

            aggregated_segmentation[
                f"{metric_name}_std"
            ] = float("nan")

    timing_means = [
        result["timing"]["mean_ms"]
        for result in run_results
        if not np.isnan(
            result["timing"]["mean_ms"]
        )
    ]

    if timing_means:
        timing_ms = float(
            np.mean(timing_means)
        )
        fps = (
            1000.0 / timing_ms
            if timing_ms > 0
            else 0.0
        )
    else:
        timing_ms = float("nan")
        fps = 0.0

    return {
        "seg": aggregated_segmentation,
        "timing": {
            "mean_ms": timing_ms,
            "fps": fps,
        },
        "samples": run_results[0]["samples"],
        "per_run": run_results,
    }




def evaluate_model_all_runs(
    model_name: str,
    loader: DataLoader,
    models_dir: Path,
    device,
    px_mm: float,
    run_ids=(0, 1, 2),
    input_size=(1, 3, 512, 512),
) -> dict:

    models_dir = Path(
        models_dir
    )

    print(
        f"\n  ── {model_name.upper()} "
        f"across {len(run_ids)} runs ──"
    )

    run_results = []
    last_model = None

    for run_id in run_ids:
        checkpoint_path = (
            models_dir
            / f"{model_name}_run{run_id}_best.pth"
        )

        if not checkpoint_path.exists():
            print(
                f"    [run {run_id}] checkpoint not found; "
                f"skipping: {checkpoint_path}"
            )
            continue

        print(
            f"    [run {run_id}] loading "
            f"{checkpoint_path.name}"
        )

        # No severity-class count is passed.
        model = build_model(
            name=model_name,
            num_classes=1,
        ).to(device)

        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
        )

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        # New training code stores this under "threshold".
        # "best_thr" remains as a fallback for older checkpoints.
        best_threshold = checkpoint.get(
            "threshold",
            checkpoint.get(
                "best_thr",
                0.5,
            ),
        )

        print(
            f"    [run {run_id}] using "
            f"threshold={best_threshold:.2f}"
        )

        result = evaluate_checkpoint(
            model=model,
            loader=loader,
            device=device,
            px_mm=px_mm,
            threshold=best_threshold,
        )

        result["epoch"] = checkpoint.get(
            "epoch",
            -1,
        )

        result["seed"] = checkpoint.get(
            "seed",
            -1,
        )

        result["run_id"] = run_id

        run_results.append(
            result
        )

        last_model = model

        print(
            f"    [run {run_id}] "
            f"dice={result['seg']['dice_mean']:.4f}  "
            f"hd95={result['seg']['hd95_mean']:.2f} mm"
        )

    if not run_results:
        print(
            f"    No checkpoints found for {model_name}."
        )
        return None

    print("    Computing GFLOPs...")

    last_model.eval()

    gflops = compute_gflops(
        last_model,
        input_size=input_size,
        device=device,
    )

    parameters = count_parameters(
        last_model
    )

    size_mb = model_size_mb(
        last_model
    )

    print(
        f"    GFLOPs={gflops:.2f}  "
        f"Params={parameters / 1e6:.2f}M  "
        f"Size={size_mb:.1f} MB"
    )

    aggregated = aggregate_runs(
        run_results
    )

    aggregated["gflops"] = gflops
    aggregated["params"] = parameters
    aggregated["size_mb"] = size_mb
    aggregated["px_mm"] = float(px_mm)

    return aggregated


In [ ]:


import argparse
import json
import time
import warnings
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from scipy.ndimage import binary_erosion, distance_transform_edt
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

warnings.filterwarnings("ignore")

RUN_IDS = [0, 1, 2]
MODELS = ["mvsegnet"]

# Used when the cleaned metadata has no pixel-size field.
DEFAULT_PIXEL_SIZE_MM = 0.152

_MEAN_VIS = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32,
)
_STD_VIS = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32,
)


def count_parameters(model: nn.Module) -> int:
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


def model_size_mb(model: nn.Module) -> float:
    return count_parameters(model) * 4 / 1024 / 1024



class FetalVMDataset(Dataset):

    def __init__(
        self,
        data_dir: Path,
        split: str = "test",
        transform=None,
        default_pixel_size_mm: float = DEFAULT_PIXEL_SIZE_MM,
    ):
        super().__init__()

        self.data_dir = Path(data_dir)
        self.split = split
        self.transform = transform
        self.default_pixel_size_mm = float(
            default_pixel_size_mm
        )

        self.images_dir = (
            self.data_dir
            / split
            / "images"
        )
        self.masks_dir = (
            self.data_dir
            / split
            / "masks"
        )

        if not self.images_dir.exists():
            raise FileNotFoundError(
                f"Image directory not found: {self.images_dir}"
            )

        if not self.masks_dir.exists():
            raise FileNotFoundError(
                f"Mask directory not found: {self.masks_dir}"
            )

        image_extensions = {
            ".png",
            ".jpg",
            ".jpeg",
        }

        image_paths = sorted(
            path
            for path in self.images_dir.iterdir()
            if path.suffix.lower() in image_extensions
        )

        self.samples = []

        for image_path in image_paths:
            mask_path = (
                self.masks_dir
                / image_path.name
            )

            if not mask_path.exists():
                mask_path = (
                    self.masks_dir
                    / f"{image_path.stem}.png"
                )

            if mask_path.exists():
                self.samples.append(
                    (image_path, mask_path)
                )

        if not self.samples:
            raise RuntimeError(
                f"No valid image-mask pairs found for split "
                f"'{split}' in {self.data_dir}."
            )

        self.pixel_size_by_id = {}
        metadata_path = (
            self.data_dir
            / "metadata.json"
        )

        if metadata_path.exists():
            try:
                with open(
                    metadata_path,
                    "r",
                    encoding="utf-8",
                ) as file:
                    all_metadata = json.load(file)

                split_metadata = all_metadata.get(
                    split,
                    [],
                )

                if isinstance(split_metadata, list):
                    for entry in split_metadata:
                        if not isinstance(entry, dict):
                            continue

                        sample_id = entry.get("id")
                        pixel_size = entry.get(
                            "pixel_size_mm"
                        )

                        if (
                            sample_id is not None
                            and pixel_size is not None
                        ):
                            self.pixel_size_by_id[
                                str(sample_id)
                            ] = float(pixel_size)

                elif isinstance(split_metadata, dict):
                    for key, entry in split_metadata.items():
                        if not isinstance(entry, dict):
                            continue

                        pixel_size = entry.get(
                            "pixel_size_mm"
                        )

                        if pixel_size is not None:
                            self.pixel_size_by_id[
                                str(key)
                            ] = float(pixel_size)

            except (
                OSError,
                ValueError,
                TypeError,
            ):
                self.pixel_size_by_id = {}

        print(
            f"[{split}] Loaded "
            f"{len(self.samples)} image-mask pairs"
        )

    def __len__(self):
        return len(self.samples)

    def _pixel_size_for_image(
        self,
        image_path: Path,
    ) -> float:
        sample_key = image_path.stem

        try:
            normalized_key = str(
                int(sample_key)
            )
        except ValueError:
            normalized_key = sample_key

        return self.pixel_size_by_id.get(
            normalized_key,
            self.default_pixel_size_mm,
        )

    def __getitem__(self, index):
        image_path, mask_path = self.samples[index]

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )

        if image is None:
            raise RuntimeError(
                f"Could not read image: {image_path}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB,
        )

        mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE,
        )

        if mask is None:
            raise RuntimeError(
                f"Could not read mask: {mask_path}"
            )

        mask = (
            mask > 127
        ).astype(np.float32)

        if self.transform is not None:
            transformed = self.transform(
                image=image,
                mask=mask,
            )

            image = transformed["image"]
            mask = transformed["mask"]

        else:
            image = (
                torch.from_numpy(
                    image.transpose(2, 0, 1)
                )
                .float()
                / 255.0
            )

            mask = torch.from_numpy(
                mask
            ).float()

        if not torch.is_tensor(mask):
            mask = torch.from_numpy(
                np.asarray(mask)
            ).float()
        else:
            mask = mask.float()

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        elif (
            mask.ndim == 3
            and mask.shape[-1] == 1
        ):
            mask = mask.permute(
                2,
                0,
                1,
            )

        pixel_size_mm = self._pixel_size_for_image(
            image_path
        )

        return (
            image,
            mask,
            image_path.name,
            pixel_size_mm,
        )


def _surface_distances(
    pred_np: np.ndarray,
    target_np: np.ndarray,
    pixel_size_mm: float,
) -> tuple:

    pred_bool = pred_np.astype(bool)
    target_bool = target_np.astype(bool)

    pred_surface = (
        pred_bool
        ^ binary_erosion(
            pred_bool,
            iterations=1,
            border_value=0,
        )
    )

    target_surface = (
        target_bool
        ^ binary_erosion(
            target_bool,
            iterations=1,
            border_value=0,
        )
    )

    if not pred_surface.any():
        pred_surface = pred_bool

    if not target_surface.any():
        target_surface = target_bool

    distance_to_target = (
        distance_transform_edt(
            ~target_surface
        )
        * pixel_size_mm
    )

    distance_to_prediction = (
        distance_transform_edt(
            ~pred_surface
        )
        * pixel_size_mm
    )

    return (
        distance_to_target[pred_surface],
        distance_to_prediction[target_surface],
    )


def compute_hd95(
    pred_np: np.ndarray,
    target_np: np.ndarray,
    pixel_size_mm: float,
) -> float:


    if (
        pred_np.sum() == 0
        or target_np.sum() == 0
    ):
        return float("nan")

    distance_1, distance_2 = _surface_distances(
        pred_np,
        target_np,
        pixel_size_mm,
    )

    return float(
        np.percentile(
            np.concatenate(
                [distance_1, distance_2]
            ),
            95,
        )
    )


def compute_asd(
    pred_np: np.ndarray,
    target_np: np.ndarray,
    pixel_size_mm: float,
) -> float:


    if (
        pred_np.sum() == 0
        or target_np.sum() == 0
    ):
        return float("nan")

    distance_1, distance_2 = _surface_distances(
        pred_np,
        target_np,
        pixel_size_mm,
    )

    return float(
        (
            distance_1.mean()
            + distance_2.mean()
        )
        / 2.0
    )


def compute_ventricle_width_mm(
    mask_np: np.ndarray,
    pixel_size_mm: float,
) -> float:


    mask_bool = mask_np.astype(bool)

    if mask_bool.sum() == 0:
        return 0.0

    occupied_columns = np.where(
        mask_bool.any(axis=0)
    )[0]

    if len(occupied_columns) == 0:
        return 0.0

    width_pixels = (
        occupied_columns[-1]
        - occupied_columns[0]
        + 1
    )

    return float(
        width_pixels
        * pixel_size_mm
    )


def compute_all_seg_metrics(
    pred_probability: np.ndarray,
    target: np.ndarray,
    pixel_size_mm: float,
    threshold: float = 0.5,
) -> dict:

    pred_binary = (
        pred_probability > threshold
    ).astype(np.float32)

    target = (
        target > 0.5
    ).astype(np.float32)

    intersection = (
        pred_binary
        * target
    ).sum()

    pred_sum = pred_binary.sum()
    target_sum = target.sum()

    dice = (
        2.0 * intersection
    ) / (
        pred_sum
        + target_sum
        + 1e-8
    )

    union = (
        pred_sum
        + target_sum
        - intersection
    )

    iou = (
        intersection
    ) / (
        union + 1e-8
    )

    false_positive = (
        pred_binary
        * (1.0 - target)
    ).sum()

    false_negative = (
        (1.0 - pred_binary)
        * target
    ).sum()

    precision = (
        intersection
    ) / (
        intersection
        + false_positive
        + 1e-8
    )

    recall = (
        intersection
    ) / (
        intersection
        + false_negative
        + 1e-8
    )

    pred_bool = pred_binary.astype(bool)
    target_bool = target.astype(bool)

    hd95 = compute_hd95(
        pred_bool,
        target_bool,
        pixel_size_mm,
    )

    asd = compute_asd(
        pred_bool,
        target_bool,
        pixel_size_mm,
    )

    predicted_width = compute_ventricle_width_mm(
        pred_bool,
        pixel_size_mm,
    )

    ground_truth_width = compute_ventricle_width_mm(
        target_bool,
        pixel_size_mm,
    )

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "hd95": float(hd95),
        "asd": float(asd),
        "width_ae": float(
            abs(
                predicted_width
                - ground_truth_width
            )
        ),
        "pred_width": float(
            predicted_width
        ),
        "gt_width": float(
            ground_truth_width
        ),
    }


def compute_gflops(
    model: nn.Module,
    input_size=(1, 3, 512, 512),
    device="cpu",
) -> float:

    device = torch.device(device)

    flops_total = [0]
    hooks = []

    def conv_hook(
        module,
        inputs,
        output,
    ):
        (
            batch_size,
            out_channels,
            out_height,
            out_width,
        ) = output.shape

        in_channels_per_group = (
            inputs[0].shape[1]
            // module.groups
        )

        kernel_height, kernel_width = (
            module.kernel_size
        )

        flops_total[0] += (
            2
            * batch_size
            * out_channels
            * out_height
            * out_width
            * in_channels_per_group
            * kernel_height
            * kernel_width
        )

    def linear_hook(
        module,
        inputs,
        output,
    ):
        input_tensor = inputs[0]

        if input_tensor.dim() == 2:
            item_count = input_tensor.shape[0]
        else:
            item_count = int(
                np.prod(
                    input_tensor.shape[:-1]
                )
            )

        flops_total[0] += (
            2
            * item_count
            * module.in_features
            * module.out_features
        )

    for module in model.modules():
        if isinstance(
            module,
            nn.Conv2d,
        ):
            hooks.append(
                module.register_forward_hook(
                    conv_hook
                )
            )

        elif isinstance(
            module,
            nn.Linear,
        ):
            hooks.append(
                module.register_forward_hook(
                    linear_hook
                )
            )

    dummy_input = torch.zeros(
        input_size,
        device=device,
    )

    model.eval()

    with torch.no_grad():
        model(dummy_input)

    for hook in hooks:
        hook.remove()

    return (
        flops_total[0]
        / 1e9
    )


@torch.no_grad()
def evaluate_checkpoint(
    model: nn.Module,
    loader: DataLoader,
    device,
    threshold: float = 0.5,
    max_visualization_samples: int = 50,
) -> dict:

    model.eval()

    all_metrics = []
    inference_times = []
    visualization_samples = []

    progress_bar = tqdm(
        loader,
        desc="    Evaluating",
        leave=False,
    )

    for (
        images,
        masks,
        filenames,
        pixel_sizes_mm,
    ) in progress_bar:

        images = images.to(
            device,
            non_blocking=True,
        ).float()

        masks = masks.to(
            device,
            non_blocking=True,
        ).float()

        if device.type == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        outputs = model(images)
        seg_logits = outputs["seg"]

        if device.type == "cuda":
            torch.cuda.synchronize()

        end_time = time.perf_counter()

        seconds_per_image = (
            end_time - start_time
        ) / images.size(0)

        inference_times.extend(
            [seconds_per_image]
            * images.size(0)
        )

        seg_probabilities = (
            torch.sigmoid(seg_logits)
            .cpu()
            .numpy()
        )

        masks_np = (
            masks.cpu()
            .numpy()
        )

        images_cpu = (
            images.cpu()
        )

        for sample_index in range(
            images.size(0)
        ):
            pixel_size_mm = float(
                pixel_sizes_mm[
                    sample_index
                ]
            )

            sample_metrics = compute_all_seg_metrics(
                pred_probability=seg_probabilities[
                    sample_index,
                    0,
                ],
                target=masks_np[
                    sample_index,
                    0,
                ],
                pixel_size_mm=pixel_size_mm,
                threshold=threshold,
            )

            sample_metrics[
                "filename"
            ] = filenames[
                sample_index
            ]

            sample_metrics[
                "pixel_size_mm"
            ] = pixel_size_mm

            all_metrics.append(
                sample_metrics
            )

            if (
                len(visualization_samples)
                < max_visualization_samples
            ):
                visualization_samples.append(
                    {
                        "image": images_cpu[
                            sample_index
                        ],
                        "mask_gt": masks_np[
                            sample_index,
                            0,
                        ],
                        "mask_pred": (
                            seg_probabilities[
                                sample_index,
                                0,
                            ]
                            > threshold
                        ).astype(np.float32),
                        "filename": filenames[
                            sample_index
                        ],
                        "dice": sample_metrics[
                            "dice"
                        ],
                        "pixel_size_mm": pixel_size_mm,
                    }
                )

    def nanmean_std(metric_name):
        values = np.asarray(
            [
                metric[metric_name]
                for metric in all_metrics
            ],
            dtype=np.float64,
        )

        values = values[
            ~np.isnan(values)
        ]

        if values.size == 0:
            return (
                float("nan"),
                float("nan"),
            )

        return (
            float(values.mean()),
            float(values.std()),
        )

    segmentation_summary = {}

    for metric_name in [
        "dice",
        "iou",
        "precision",
        "recall",
        "hd95",
        "asd",
        "width_ae",
    ]:
        mean_value, std_value = nanmean_std(
            metric_name
        )

        segmentation_summary[
            f"{metric_name}_mean"
        ] = mean_value

        segmentation_summary[
            f"{metric_name}_std"
        ] = std_value

    if inference_times:
        mean_ms = (
            float(
                np.mean(
                    inference_times
                )
            )
            * 1000.0
        )

        std_ms = (
            float(
                np.std(
                    inference_times
                )
            )
            * 1000.0
        )
    else:
        mean_ms = float("nan")
        std_ms = float("nan")

    fps = (
        1000.0 / mean_ms
        if mean_ms > 0
        else 0.0
    )

    return {
        "seg": segmentation_summary,
        "timing": {
            "mean_ms": mean_ms,
            "std_ms": std_ms,
            "fps": fps,
        },
        "samples": visualization_samples,
        "per_sample": all_metrics,
        "threshold": float(
            threshold
        ),
    }

def aggregate_runs(
    run_results: list,
) -> dict:

    segmentation_summary = {}

    for metric_name in [
        "dice",
        "iou",
        "precision",
        "recall",
        "hd95",
        "asd",
        "width_ae",
    ]:
        per_run_values = [
            result["seg"][
                f"{metric_name}_mean"
            ]
            for result in run_results
            if not np.isnan(
                result["seg"][
                    f"{metric_name}_mean"
                ]
            )
        ]

        if per_run_values:
            segmentation_summary[
                f"{metric_name}_mean"
            ] = float(
                np.mean(
                    per_run_values
                )
            )

            segmentation_summary[
                f"{metric_name}_std"
            ] = float(
                np.std(
                    per_run_values
                )
            )

        else:
            segmentation_summary[
                f"{metric_name}_mean"
            ] = float("nan")

            segmentation_summary[
                f"{metric_name}_std"
            ] = float("nan")

    timing_values = [
        result["timing"]["mean_ms"]
        for result in run_results
        if not np.isnan(
            result["timing"]["mean_ms"]
        )
    ]

    if timing_values:
        mean_timing_ms = float(
            np.mean(
                timing_values
            )
        )

        mean_fps = (
            1000.0
            / mean_timing_ms
            if mean_timing_ms > 0
            else 0.0
        )
    else:
        mean_timing_ms = float("nan")
        mean_fps = 0.0

    return {
        "seg": segmentation_summary,
        "timing": {
            "mean_ms": mean_timing_ms,
            "fps": mean_fps,
        },
        "samples": run_results[0][
            "samples"
        ],
        "per_sample": run_results[0][
            "per_sample"
        ],
        "per_run": run_results,
    }


def evaluate_model_all_runs(
    model_name: str,
    loader: DataLoader,
    models_dir: Path,
    device,
    run_ids=RUN_IDS,
    input_size=(1, 3, 512, 512),
) -> dict:

    models_dir = Path(
        models_dir
    )

    print(
        f"\n  ── {model_name.upper()} "
        f"across {len(run_ids)} runs ──"
    )

    run_results = []
    final_model = None

    for run_id in run_ids:
        checkpoint_path = (
            models_dir
            / f"{model_name}_run{run_id}_best.pth"
        )

        if not checkpoint_path.exists():
            print(
                f"    [run {run_id}] "
                f"checkpoint not found; skipping: "
                f"{checkpoint_path}"
            )
            continue

        print(
            f"    [run {run_id}] loading "
            f"{checkpoint_path.name}"
        )

        model = build_model(
            name=model_name,
            num_classes=1,
        ).to(device)

        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
        )

        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        threshold = checkpoint.get(
            "threshold",
            checkpoint.get(
                "best_thr",
                0.5,
            ),
        )

        print(
            f"    [run {run_id}] "
            f"using threshold="
            f"{threshold:.2f}"
        )

        result = evaluate_checkpoint(
            model=model,
            loader=loader,
            device=device,
            threshold=threshold,
        )

        result["epoch"] = checkpoint.get(
            "epoch",
            -1,
        )

        result["seed"] = checkpoint.get(
            "seed",
            -1,
        )

        result["run_id"] = run_id

        run_results.append(
            result
        )

        final_model = model

        print(
            f"    [run {run_id}] "
            f"dice="
            f"{result['seg']['dice_mean']:.4f}  "
            f"hd95="
            f"{result['seg']['hd95_mean']:.2f} mm"
        )

    if not run_results:
        print(
            f"    No checkpoints found "
            f"for {model_name}."
        )
        return None

    print(
        "    Computing GFLOPs..."
    )

    gflops = compute_gflops(
        final_model,
        input_size=input_size,
        device=device,
    )

    parameters = count_parameters(
        final_model
    )

    size_mb = model_size_mb(
        final_model
    )

    print(
        f"    GFLOPs={gflops:.2f}  "
        f"Params={parameters / 1e6:.2f}M  "
        f"Size={size_mb:.1f} MB"
    )

    aggregated = aggregate_runs(
        run_results
    )

    aggregated["gflops"] = gflops
    aggregated["params"] = parameters
    aggregated["size_mb"] = size_mb

    return aggregated


def denormalize_image(
    tensor: torch.Tensor,
) -> np.ndarray:
    image = (
        tensor.permute(
            1,
            2,
            0,
        ).numpy()
        * _STD_VIS
        + _MEAN_VIS
    )

    return np.clip(
        image,
        0,
        1,
    )


def overlay_mask(
    image,
    mask,
    color=(0, 1, 0),
    alpha=0.4,
):
    output = image.copy()

    for channel, value in enumerate(
        color
    ):
        output[:, :, channel] = np.where(
            mask > 0,
            (
                image[:, :, channel]
                * (1.0 - alpha)
                + value * alpha
            ),
            image[:, :, channel],
        )

    return output


def error_map(
    pred_binary,
    gt_binary,
    image,
):
    output = image.copy() * 0.5

    true_positive = (
        (pred_binary > 0)
        & (gt_binary > 0)
    )

    false_positive = (
        (pred_binary > 0)
        & (gt_binary == 0)
    )

    false_negative = (
        (pred_binary == 0)
        & (gt_binary > 0)
    )

    output[true_positive] = [
        0.0,
        0.8,
        0.0,
    ]

    output[false_positive] = [
        0.9,
        0.1,
        0.1,
    ]

    output[false_negative] = [
        0.1,
        0.3,
        0.9,
    ]

    return np.clip(
        output,
        0,
        1,
    )


def plot_qualitative(
    results,
    model_name,
    results_dir,
    n=6,
):
    samples = results[
        "samples"
    ][:n]

    if not samples:
        return

    figure, axes = plt.subplots(
        len(samples),
        4,
        figsize=(
            16,
            len(samples) * 3.5,
        ),
    )

    if len(samples) == 1:
        axes = axes[
            np.newaxis,
            :,
        ]

    titles = [
        "Input Image",
        "Ground Truth",
        "Prediction Overlay",
        "Error Map",
    ]

    for axis, title in zip(
        axes[0],
        titles,
    ):
        axis.set_title(
            title,
            fontsize=11,
            fontweight="bold",
        )

    for row_index, sample in enumerate(
        samples
    ):
        image = denormalize_image(
            sample["image"]
        )

        ground_truth = sample[
            "mask_gt"
        ]

        prediction = sample[
            "mask_pred"
        ]

        height, width = image.shape[:2]

        resized_ground_truth = cv2.resize(
            ground_truth,
            (width, height),
            interpolation=cv2.INTER_NEAREST,
        )

        resized_prediction = cv2.resize(
            prediction,
            (width, height),
            interpolation=cv2.INTER_NEAREST,
        )

        axes[row_index, 0].imshow(
            image
        )

        axes[row_index, 0].set_ylabel(
            f"{sample['filename']}\n"
            f"Dice={sample['dice']:.3f}",
            fontsize=8,
        )

        axes[row_index, 1].imshow(
            resized_ground_truth,
            cmap="gray",
        )

        axes[row_index, 2].imshow(
            overlay_mask(
                image,
                resized_prediction,
            )
        )

        axes[row_index, 3].imshow(
            error_map(
                resized_prediction,
                resized_ground_truth,
                image,
            )
        )

        for axis in axes[row_index]:
            axis.axis("off")

    figure.legend(
        handles=[
            mpatches.Patch(
                color=(0.0, 0.8, 0.0),
                label="True Positive",
            ),
            mpatches.Patch(
                color=(0.9, 0.1, 0.1),
                label="False Positive",
            ),
            mpatches.Patch(
                color=(0.1, 0.3, 0.9),
                label="False Negative",
            ),
        ],
        loc="lower center",
        ncol=3,
        fontsize=10,
    )

    plt.suptitle(
        f"{model_name.upper()} "
        f"— Qualitative Results",
        fontsize=13,
        y=1.01,
    )

    plt.tight_layout()

    output_path = (
        results_dir
        / f"{model_name}_qualitative.png"
    )

    plt.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close()

    print(
        f"    ✓ {output_path.name}"
    )


def plot_worst_cases(
    results,
    model_name,
    results_dir,
    n=10,
):
    samples = sorted(
        results["samples"],
        key=lambda sample: sample[
            "dice"
        ],
    )[:n]

    if not samples:
        return

    figure, axes = plt.subplots(
        len(samples),
        4,
        figsize=(
            16,
            len(samples) * 3.5,
        ),
    )

    if len(samples) == 1:
        axes = axes[
            np.newaxis,
            :,
        ]

    for axis, title in zip(
        axes[0],
        [
            "Input Image",
            "Ground Truth",
            "Prediction Overlay",
            "Error Map",
        ],
    ):
        axis.set_title(
            title,
            fontsize=11,
            fontweight="bold",
        )

    for row_index, sample in enumerate(
        samples
    ):
        image = denormalize_image(
            sample["image"]
        )

        ground_truth = sample[
            "mask_gt"
        ]

        prediction = sample[
            "mask_pred"
        ]

        height, width = image.shape[:2]

        resized_ground_truth = cv2.resize(
            ground_truth,
            (width, height),
            interpolation=cv2.INTER_NEAREST,
        )

        resized_prediction = cv2.resize(
            prediction,
            (width, height),
            interpolation=cv2.INTER_NEAREST,
        )

        axes[row_index, 0].imshow(
            image
        )

        axes[row_index, 0].set_ylabel(
            f"{sample['filename']}\n"
            f"Dice={sample['dice']:.3f}",
            fontsize=8,
        )

        axes[row_index, 1].imshow(
            resized_ground_truth,
            cmap="gray",
        )

        axes[row_index, 2].imshow(
            overlay_mask(
                image,
                resized_prediction,
            )
        )

        axes[row_index, 3].imshow(
            error_map(
                resized_prediction,
                resized_ground_truth,
                image,
            )
        )

        for axis in axes[row_index]:
            axis.axis("off")

    figure.legend(
        handles=[
            mpatches.Patch(
                color=(0.0, 0.8, 0.0),
                label="True Positive",
            ),
            mpatches.Patch(
                color=(0.9, 0.1, 0.1),
                label="False Positive",
            ),
            mpatches.Patch(
                color=(0.1, 0.3, 0.9),
                label="False Negative",
            ),
        ],
        loc="lower center",
        ncol=3,
        fontsize=10,
    )

    plt.suptitle(
        f"{model_name.upper()} "
        f"— {len(samples)} Worst Dice Cases",
        fontsize=13,
        y=1.01,
    )

    plt.tight_layout()

    output_path = (
        results_dir
        / f"{model_name}_worst_cases.png"
    )

    plt.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close()

    print(
        f"    ✓ {output_path.name}"
    )


def plot_training_curves(
    results_dir,
):
    figure, axes = plt.subplots(
        1,
        2,
        figsize=(14, 5),
    )

    colors = [
        "#1f77b4",
        "#aec7e8",
        "#08519c",
    ]

    plotted = False

    for model_name in MODELS:
        for run_id in RUN_IDS:
            history_path = (
                results_dir
                / f"{model_name}_run"
                f"{run_id}_history.json"
            )

            if not history_path.exists():
                continue

            with open(
                history_path,
                "r",
                encoding="utf-8",
            ) as file:
                history = json.load(file)

            color = colors[
                run_id % len(colors)
            ]

            axes[0].plot(
                history["val_dice"],
                color=color,
                label=(
                    f"{model_name} "
                    f"run{run_id}"
                ),
                alpha=0.8,
            )

            axes[1].plot(
                history["val_loss"],
                color=color,
                label=(
                    f"{model_name} "
                    f"run{run_id}"
                ),
                alpha=0.8,
            )

            plotted = True

    for axis, title, ylabel in zip(
        axes,
        [
            "Validation Dice per Run",
            "Validation Loss per Run",
        ],
        [
            "Dice",
            "Loss",
        ],
    ):
        axis.set_title(title)
        axis.set_ylabel(ylabel)
        axis.set_xlabel("Epoch")
        axis.grid(
            True,
            alpha=0.3,
        )

        if plotted:
            axis.legend(
                fontsize=7,
                ncol=2,
            )

    plt.tight_layout()

    output_path = (
        results_dir
        / "training_curves.png"
    )

    plt.savefig(
        output_path,
        dpi=200,
    )

    plt.close()

    print(
        f"  ✓ {output_path.name}"
    )


def build_width_analysis(
    all_results,
    results_dir,
):
    valid_results = {
        model_name: result
        for model_name, result in all_results.items()
        if (
            result is not None
            and result.get(
                "per_sample"
            )
        )
    }

    if not valid_results:
        return

    figure, axes = plt.subplots(
        1,
        len(valid_results),
        figsize=(
            7 * len(valid_results),
            6,
        ),
    )

    if len(valid_results) == 1:
        axes = [axes]

    for axis, (
        model_name,
        result,
    ) in zip(
        axes,
        valid_results.items(),
    ):
        ground_truth_widths = [
            sample["gt_width"]
            for sample in result[
                "per_sample"
            ]
        ]

        predicted_widths = [
            sample["pred_width"]
            for sample in result[
                "per_sample"
            ]
        ]

        axis.scatter(
            ground_truth_widths,
            predicted_widths,
            alpha=0.6,
        )

        if ground_truth_widths:
            lower = min(
                min(ground_truth_widths),
                min(predicted_widths),
            )

            upper = max(
                max(ground_truth_widths),
                max(predicted_widths),
            )

            axis.plot(
                [lower, upper],
                [lower, upper],
                linestyle="--",
            )

        axis.set_title(
            f"{model_name.upper()} "
            f"— Width Agreement"
        )

        axis.set_xlabel(
            "Ground-truth width (mm)"
        )

        axis.set_ylabel(
            "Predicted width (mm)"
        )

        axis.grid(
            True,
            alpha=0.3,
        )

    plt.tight_layout()

    output_path = (
        results_dir
        / "width_analysis.png"
    )

    plt.savefig(
        output_path,
        dpi=200,
    )

    plt.close()

    print(
        f"  ✓ {output_path.name}"
    )




def format_mean_std(
    mean,
    std,
    percentage=False,
    decimals=2,
):
    if np.isnan(mean):
        return "N/A"

    scale = (
        100
        if percentage
        else 1
    )

    return (
        f"{mean * scale:.{decimals}f} "
        f"± "
        f"{std * scale:.{decimals}f}"
    )


def build_comparison_table(
    all_results,
):
    rows = []

    for model_name, result in all_results.items():
        if result is None:
            continue

        segmentation = result[
            "seg"
        ]

        rows.append(
            {
                "Model": model_name.upper(),
                "Params (M)": (
                    f"{result['params'] / 1e6:.2f}"
                ),
                "Size (MB)": (
                    f"{result['size_mb']:.1f}"
                ),
                "GFLOPs": (
                    f"{result['gflops']:.2f}"
                ),
                "Dice (%)": format_mean_std(
                    segmentation[
                        "dice_mean"
                    ],
                    segmentation[
                        "dice_std"
                    ],
                    percentage=True,
                ),
                "IoU (%)": format_mean_std(
                    segmentation[
                        "iou_mean"
                    ],
                    segmentation[
                        "iou_std"
                    ],
                    percentage=True,
                ),
                "Precision (%)": format_mean_std(
                    segmentation[
                        "precision_mean"
                    ],
                    segmentation[
                        "precision_std"
                    ],
                    percentage=True,
                ),
                "Recall (%)": format_mean_std(
                    segmentation[
                        "recall_mean"
                    ],
                    segmentation[
                        "recall_std"
                    ],
                    percentage=True,
                ),
                "HD95 (mm)": format_mean_std(
                    segmentation[
                        "hd95_mean"
                    ],
                    segmentation[
                        "hd95_std"
                    ],
                ),
                "ASD (mm)": format_mean_std(
                    segmentation[
                        "asd_mean"
                    ],
                    segmentation[
                        "asd_std"
                    ],
                ),
                "Width MAE (mm)": format_mean_std(
                    segmentation[
                        "width_ae_mean"
                    ],
                    segmentation[
                        "width_ae_std"
                    ],
                ),
                "FPS": (
                    f"{result['timing']['fps']:.1f}"
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


def build_ablation_table():
    rows = [
        {
            "Variant": (
                "Baseline "
                "(MobileNetV3 U-Net)"
            ),
            "MSAM": "✗",
            "ABRB": "✗",
            "Edge Loss": "✗",
            "Dice (%)": "— fill —",
            "IoU (%)": "— fill —",
            "HD95 (mm)": "— fill —",
            "ASD (mm)": "— fill —",
        },
        {
            "Variant": "+ MSAM",
            "MSAM": "✓",
            "ABRB": "✗",
            "Edge Loss": "✗",
            "Dice (%)": "— fill —",
            "IoU (%)": "— fill —",
            "HD95 (mm)": "— fill —",
            "ASD (mm)": "— fill —",
        },
        {
            "Variant": "+ ABRB",
            "MSAM": "✓",
            "ABRB": "✓",
            "Edge Loss": "✗",
            "Dice (%)": "— fill —",
            "IoU (%)": "— fill —",
            "HD95 (mm)": "— fill —",
            "ASD (mm)": "— fill —",
        },
        {
            "Variant": (
                "Full MVSegNet "
                "(+ Edge Loss)"
            ),
            "MSAM": "✓",
            "ABRB": "✓",
            "Edge Loss": "✓",
            "Dice (%)": "— fill —",
            "IoU (%)": "— fill —",
            "HD95 (mm)": "— fill —",
            "ASD (mm)": "— fill —",
        },
    ]

    return pd.DataFrame(
        rows
    )


def save_worst_cases_csv(
    results,
    model_name,
    results_dir,
    n=20,
):
    per_sample = results.get(
        "per_sample",
        [],
    )

    if not per_sample:
        return None

    dataframe = pd.DataFrame(
        per_sample
    )

    dataframe = dataframe.sort_values(
        "dice",
        ascending=True,
    ).head(n)

    output_path = (
        results_dir
        / f"{model_name}_worst_cases.csv"
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )

    print(
        f"    ✓ {output_path.name}"
    )

    return dataframe


def main():
    parser = argparse.ArgumentParser(
        description=(
            "MVSegNet segmentation-only "
            "evaluation"
        )
    )

    parser.add_argument(
        "--batch",
        type=int,
        default=8,
    )

    parser.add_argument(
        "--pixel-size-mm",
        type=float,
        default=DEFAULT_PIXEL_SIZE_MM,
        help=(
            "Fallback millimetres per pixel when "
            "metadata has no pixel_size_mm field."
        ),
    )

    arguments, unknown = (
        parser.parse_known_args()
    )

    if unknown:
        print(
            f"Ignoring unknown arguments: "
            f"{unknown}"
        )

    base_dir = Path(
        "fetal_vm_segmentation"
    )

    data_dir = (
        base_dir
        / "data"
        / "preprocessed"
    )

    models_dir = (
        base_dir
        / "models"
    )

    results_dir = (
        base_dir
        / "results"
    )

    results_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        "=" * 72
    )

    print(
        "  MVSegNet — "
        "SEGMENTATION-ONLY EVALUATION"
    )

    print(
        "=" * 72
    )

    print(
        f"  Device         : {device}"
    )

    print(
        f"  Fallback px/mm : "
        f"{arguments.pixel_size_mm}"
    )

    print(
        f"  Data dir       : {data_dir}"
    )

    print(
        f"  Models dir     : {models_dir}"
    )

    print(
        f"  Results dir    : {results_dir}"
    )

    print(
        f"  Runs evaluated : "
        f"{len(RUN_IDS)}"
    )

    print(
        "=" * 72
    )

    if not data_dir.exists():
        print(
            f"\nERROR: {data_dir} not found. "
            f"Run preprocessing first."
        )
        return

    test_transform = A.Compose(
        [
            A.Normalize(
                mean=[
                    0.485,
                    0.456,
                    0.406,
                ],
                std=[
                    0.229,
                    0.224,
                    0.225,
                ],
            ),
            ToTensorV2(),
        ]
    )

    try:
        test_dataset = FetalVMDataset(
            data_dir=data_dir,
            split="test",
            transform=test_transform,
            default_pixel_size_mm=(
                arguments.pixel_size_mm
            ),
        )

    except Exception as error:
        print(
            f"ERROR loading test dataset: "
            f"{error}"
        )
        return

    test_loader = DataLoader(
        test_dataset,
        batch_size=arguments.batch,
        shuffle=False,
        num_workers=4,
        pin_memory=(
            device.type == "cuda"
        ),
    )

    print(
        f"\n  Test set: "
        f"{len(test_dataset)} samples"
    )

    all_results = {}

    for model_name in MODELS:
        all_results[
            model_name
        ] = evaluate_model_all_runs(
            model_name=model_name,
            loader=test_loader,
            models_dir=models_dir,
            device=device,
            input_size=(
                1,
                3,
                512,
                512,
            ),
        )


    print(
        "\n"
        + "=" * 72
    )

    print(
        "  TABLE 1 — Main Comparison"
    )

    print(
        "=" * 72
    )

    comparison_table = build_comparison_table(
        all_results
    )

    print(
        comparison_table.to_string(
            index=False
        )
    )

    comparison_path = (
        results_dir
        / "table1_comparison.csv"
    )

    comparison_table.to_csv(
        comparison_path,
        index=False,
    )

    print(
        f"\n  ✓ {comparison_path.name}"
    )

    print(
        "\n"
        + "=" * 72
    )

    print(
        "  TABLE 2 — Ablation Study"
    )

    print(
        "=" * 72
    )

    ablation_table = build_ablation_table()

    print(
        ablation_table.to_string(
            index=False
        )
    )

    ablation_path = (
        results_dir
        / "table2_ablation.csv"
    )

    ablation_table.to_csv(
        ablation_path,
        index=False,
    )

    print(
        f"\n  ✓ {ablation_path.name}"
    )


    print(
        "\n"
        + "=" * 72
    )

    print(
        "  Generating Figures..."
    )

    print(
        "=" * 72
    )

    for model_name, result in all_results.items():
        if result is None:
            continue

        plot_qualitative(
            result,
            model_name,
            results_dir,
            n=6,
        )

        plot_worst_cases(
            result,
            model_name,
            results_dir,
            n=10,
        )

    plot_training_curves(
        results_dir
    )

    build_width_analysis(
        all_results,
        results_dir,
    )



    complete_results = {}

    for model_name, result in all_results.items():
        if result is None:
            continue

        worst_cases = save_worst_cases_csv(
            result,
            model_name,
            results_dir,
            n=20,
        )

        if worst_cases is not None:
            available_columns = [
                column
                for column in [
                    "filename",
                    "dice",
                    "iou",
                    "hd95",
                    "asd",
                    "width_ae",
                    "pixel_size_mm",
                ]
                if column in worst_cases.columns
            ]

            print(
                worst_cases[
                    available_columns
                ].to_string(
                    index=False
                )
            )

        complete_results[
            model_name
        ] = {
            "segmentation": result[
                "seg"
            ],
            "timing": result[
                "timing"
            ],
            "complexity": {
                "gflops": result[
                    "gflops"
                ],
                "params": result[
                    "params"
                ],
                "size_mb": result[
                    "size_mb"
                ],
            },
        }

    complete_path = (
        results_dir
        / "complete_results.json"
    )

    with open(
        complete_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            complete_results,
            file,
            indent=2,
            default=str,
        )

    print(
        f"  ✓ {complete_path.name}"
    )


    print(
        "\n"
        + "=" * 72
    )

    print(
        "  SUMMARY"
    )

    print(
        "=" * 72
    )

    for model_name, result in all_results.items():
        if result is None:
            print(
                f"  {model_name.upper():12s}: "
                f"no checkpoints found"
            )
            continue

        segmentation = result[
            "seg"
        ]

        print(
            f"\n  {model_name.upper()}"
        )

        print(
            f"    Dice      : "
            f"{segmentation['dice_mean'] * 100:.2f} "
            f"± "
            f"{segmentation['dice_std'] * 100:.2f} %"
        )

        print(
            f"    IoU       : "
            f"{segmentation['iou_mean'] * 100:.2f} "
            f"± "
            f"{segmentation['iou_std'] * 100:.2f} %"
        )

        print(
            f"    HD95      : "
            f"{segmentation['hd95_mean']:.2f} "
            f"± "
            f"{segmentation['hd95_std']:.2f} mm"
        )

        print(
            f"    ASD       : "
            f"{segmentation['asd_mean']:.2f} "
            f"± "
            f"{segmentation['asd_std']:.2f} mm"
        )

        print(
            f"    Width MAE : "
            f"{segmentation['width_ae_mean']:.2f} "
            f"± "
            f"{segmentation['width_ae_std']:.2f} mm"
        )

        print(
            f"    GFLOPs    : "
            f"{result['gflops']:.2f}"
        )

        print(
            f"    Params    : "
            f"{result['params'] / 1e6:.2f} M"
        )

        print(
            f"    FPS       : "
            f"{result['timing']['fps']:.1f}"
        )

    print(
        "\n"
        + "=" * 72
    )

    print(
        "  Evaluation complete. "
        f"Results in: {results_dir}"
    )

    print(
        "=" * 72
    )

    for output_file in sorted(
        results_dir.glob("*")
    ):
        print(
            f"    {output_file.name}"
        )


if __name__ == "__main__":
    main()

Ignoring unknown arguments: ['-f', '/root/.local/share/jupyter/runtime/kernel-b0b8627d-2cd2-4987-8ef6-f35768175bf0.json']
  MVSegNet — SEGMENTATION-ONLY EVALUATION
  Device         : cuda
  Fallback px/mm : 0.152
  Data dir       : fetal_vm_segmentation/data/preprocessed
  Models dir     : fetal_vm_segmentation/models
  Results dir    : fetal_vm_segmentation/results
  Runs evaluated : 3
[test] Loaded 55 image-mask pairs

  Test set: 55 samples

  ── MVSEGNET across 3 runs ──
    [run 0] loading mvsegnet_run0_best.pth
[MVSegNet] Encoder channels: 16, 24, 48, 96, 576
[mvsegnet] Parameters: 1,665,164  |  Size: 6.4 MB
    [run 0] using threshold=0.30


    [run 0] dice=0.8049  hd95=1.26 mm
    [run 1] loading mvsegnet_run1_best.pth
[MVSegNet] Encoder channels: 16, 24, 48, 96, 576
[mvsegnet] Parameters: 1,665,164  |  Size: 6.4 MB
    [run 1] using threshold=0.30


    [run 1] dice=0.8089  hd95=1.27 mm
    [run 2] loading mvsegnet_run2_best.pth
[MVSegNet] Encoder channels: 16, 24, 48, 96, 576
[mvsegnet] Parameters: 1,665,164  |  Size: 6.4 MB
    [run 2] using threshold=0.50


    [run 2] dice=0.8083  hd95=1.71 mm
    Computing GFLOPs...
    GFLOPs=2.55  Params=1.67M  Size=6.4 MB

  TABLE 1 — Main Comparison
   Model Params (M) Size (MB) GFLOPs     Dice (%)      IoU (%) Precision (%)   Recall (%)   HD95 (mm)    ASD (mm) Width MAE (mm)   FPS
MVSEGNET       1.67       6.4   2.55 80.74 ± 0.17 68.47 ± 0.23  79.80 ± 0.34 86.37 ± 0.24 1.41 ± 0.21 0.51 ± 0.02    1.20 ± 0.17 179.2

  ✓ table1_comparison.csv

  TABLE 2 — Ablation Study
                     Variant MSAM ABRB Edge Loss Dice (%)  IoU (%) HD95 (mm) ASD (mm)
Baseline (MobileNetV3 U-Net)    ✗    ✗         ✗ — fill — — fill —  — fill — — fill —
                      + MSAM    ✓    ✗         ✗ — fill — — fill —  — fill — — fill —
                      + ABRB    ✓    ✓         ✗ — fill — — fill —  — fill — — fill —
 Full MVSegNet (+ Edge Loss)    ✓    ✓         ✓ — fill — — fill —  — fill — — fill —

  ✓ table2_ablation.csv

  Generating Figures...
    ✓ mvsegnet_qualitative.png
    ✓ mvsegnet_worst_cases.png

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = Path('/content/drive/MyDrive/fetsam_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:


from dataclasses import dataclass
from collections import OrderedDict


@dataclass(frozen=True)
class AblationCfg:
    """Independent flags for MVSegNet components and supervision losses."""

    use_attention_gates: bool = False
    use_msam: bool = False
    use_film: bool = False
    use_abrb: bool = False
    use_edge_loss: bool = False
    use_deep_sup: bool = False


ABLATION_VARIANTS = OrderedDict([
    (
        "V1_baseline",
        AblationCfg(
            use_attention_gates=False,
            use_msam=False,
            use_film=False,
            use_abrb=False,
            use_edge_loss=False,
            use_deep_sup=False,
        ),
    ),
    (
        "V2_+attn_gates",
        AblationCfg(
            use_attention_gates=True,
            use_msam=False,
            use_film=False,
            use_abrb=False,
            use_edge_loss=False,
            use_deep_sup=False,
        ),
    ),
    (
        "V3_+msam",
        AblationCfg(
            use_attention_gates=True,
            use_msam=True,
            use_film=False,
            use_abrb=False,
            use_edge_loss=False,
            use_deep_sup=False,
        ),
    ),
    (
        "V4_+film",
        AblationCfg(
            use_attention_gates=True,
            use_msam=True,
            use_film=True,
            use_abrb=False,
            use_edge_loss=False,
            use_deep_sup=False,
        ),
    ),
    (
        "V5_+abrb",
        AblationCfg(
            use_attention_gates=True,
            use_msam=True,
            use_film=True,
            use_abrb=True,
            use_edge_loss=False,
            use_deep_sup=False,
        ),
    ),
    (
        "V6_+edge_loss",
        AblationCfg(
            use_attention_gates=True,
            use_msam=True,
            use_film=True,
            use_abrb=True,
            use_edge_loss=True,
            use_deep_sup=False,
        ),
    ),
    (
        "V7_full",
        AblationCfg(
            use_attention_gates=True,
            use_msam=True,
            use_film=True,
            use_abrb=True,
            use_edge_loss=True,
            use_deep_sup=True,
        ),
    ),
])


VARIANT_LABELS = {
    "V1_baseline": "V1 — Baseline (MobileNetV3 U-Net)",
    "V2_+attn_gates": "V2 — + Attention Gates",
    "V3_+msam": "V3 — + MSAM",
    "V4_+film": "V4 — + FiLM (GA Conditioning)",
    "V5_+abrb": "V5 — + ABRB",
    "V6_+edge_loss": "V6 — + Edge Supervision",
    "V7_full": "V7 — Full MVSegNet (+ Deep Supervision)",
}


print("✓ Ablation configurations defined.")
print("\nVariant order:")

for name, cfg in ABLATION_VARIANTS.items():
    print(f"  {VARIANT_LABELS[name]}")
    print(
        f"      attn={cfg.use_attention_gates}  "
        f"msam={cfg.use_msam}  "
        f"film={cfg.use_film}  "
        f"abrb={cfg.use_abrb}  "
        f"edge_loss={cfg.use_edge_loss}  "
        f"deep_sup={cfg.use_deep_sup}"
    )

✓ Ablation configurations defined.

Variant order:
  V1 — Baseline (MobileNetV3 U-Net)
      attn=False  msam=False  film=False  abrb=False  edge_loss=False  deep_sup=False
  V2 — + Attention Gates
      attn=True  msam=False  film=False  abrb=False  edge_loss=False  deep_sup=False
  V3 — + MSAM
      attn=True  msam=True  film=False  abrb=False  edge_loss=False  deep_sup=False
  V4 — + FiLM (GA Conditioning)
      attn=True  msam=True  film=True  abrb=False  edge_loss=False  deep_sup=False
  V5 — + ABRB
      attn=True  msam=True  film=True  abrb=True  edge_loss=False  deep_sup=False
  V6 — + Edge Supervision
      attn=True  msam=True  film=True  abrb=True  edge_loss=True  deep_sup=False
  V7 — Full MVSegNet (+ Deep Supervision)
      attn=True  msam=True  film=True  abrb=True  edge_loss=True  deep_sup=True


In [ ]:


import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


class MVSegNetAblation(nn.Module):


    def __init__(
        self,
        cfg: AblationCfg,
        num_classes: int = 1,
        num_severity_classes: int = 4,
    ):
        super().__init__()

        self.cfg = cfg

        # Kept only so older build_model calls do not fail.
        _ = num_severity_classes

        # Edge supervision requires an edge-producing ABRB branch.
        if cfg.use_edge_loss and not cfg.use_abrb:
            raise ValueError(
                "Invalid ablation configuration: "
                "use_edge_loss=True requires use_abrb=True."
            )

        # ── Encoder ───────────────────────────────────────────
        try:
            mobilenet = models.mobilenet_v3_small(
                weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
            )
        except Exception:
            mobilenet = models.mobilenet_v3_small(pretrained=True)

        self.encoder0 = mobilenet.features[0:2]
        self.encoder1 = mobilenet.features[2:4]
        self.encoder2 = mobilenet.features[4:9]
        self.encoder3 = mobilenet.features[9:12]
        self.encoder4 = mobilenet.features[12:]


        dummy = torch.zeros(1, 3, 224, 224)

        was_training = mobilenet.training
        mobilenet.eval()

        with torch.no_grad():
            e0 = self.encoder0(dummy)
            e1 = self.encoder1(e0)
            e2 = self.encoder2(e1)
            e3 = self.encoder3(e2)
            e4 = self.encoder4(e3)

        mobilenet.train(was_training)

        self.c0 = e0.shape[1]
        self.c1 = e1.shape[1]
        self.c2 = e2.shape[1]
        self.c3 = e3.shape[1]
        self.c4 = e4.shape[1]

        self.decoder1 = DecoderBlock(
            self.c4,
            self.c3,
            self.c3,
            dropout=0.20,
        )
        self.decoder2 = DecoderBlock(
            self.c3,
            self.c2,
            self.c2,
            dropout=0.15,
        )
        self.decoder3 = DecoderBlock(
            self.c2,
            self.c1,
            self.c1,
            dropout=0.10,
        )
        self.decoder4 = DecoderBlock(
            self.c1,
            self.c0,
            self.c0,
            dropout=0.00,
        )


        self.seg_head = nn.Sequential(
            nn.Conv2d(
                self.c0,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                64,
                32,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, num_classes, kernel_size=1),
        )

        # ── Optional MSAM modules ──────────────────────────────
        if cfg.use_msam:
            self.msam = MultiScaleVentricleAttention(self.c4)
            self.msam_d1 = MultiScaleVentricleAttention(self.c3)
            self.msam_d2 = MultiScaleVentricleAttention(self.c2)

        # ── Optional FiLM conditioning ─────────────────────────
        if cfg.use_film:
            self.film = FiLMLayer(
                feature_channels=self.c4,
                condition_dim=1,
            )

        # ── Optional attention gates ───────────────────────────
        if cfg.use_attention_gates:
            self.ag1 = AttentionGate(
                F_g=self.c4,
                F_l=self.c3,
                F_int=self.c3 // 2,
            )
            self.ag2 = AttentionGate(
                F_g=self.c3,
                F_l=self.c2,
                F_int=self.c2 // 2,
            )
            self.ag3 = AttentionGate(
                F_g=self.c2,
                F_l=self.c1,
                F_int=max(self.c1 // 2, 4),
            )
            self.ag4 = AttentionGate(
                F_g=self.c1,
                F_l=self.c0,
                F_int=max(self.c0 // 2, 4),
            )

        # ── Optional ABRB ──────────────────────────────────────
        if cfg.use_abrb:
            self.abrb_d2 = AdaptiveBoundaryRefinement(self.c2)

        # ── Optional auxiliary heads ───────────────────────────
        if cfg.use_deep_sup:
            self.aux_head_d3 = nn.Conv2d(
                self.c1,
                num_classes,
                kernel_size=1,
            )
            self.aux_head_d2 = nn.Conv2d(
                self.c2,
                num_classes,
                kernel_size=1,
            )

    def forward(
        self,
        x: torch.Tensor,
        ga_weeks: torch.Tensor = None,
    ) -> dict:

        input_size = x.shape[-2:]

        # ── Encoder ───────────────────────────────────────────
        e0 = self.encoder0(x)
        e1 = self.encoder1(e0)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)

        # ── Bottleneck MSAM ───────────────────────────────────
        if self.cfg.use_msam:
            e4 = self.msam(e4)

        # ── FiLM conditioning ─────────────────────────────────
        if self.cfg.use_film:
            if ga_weeks is None:
                raise ValueError(
                    "ga_weeks must be provided when FiLM is enabled."
                )

            # Allow either shape (B,) or (B, 1).
            if ga_weeks.ndim == 1:
                ga_weeks = ga_weeks.unsqueeze(1)

            expected_shape = (x.shape[0], 1)

            if tuple(ga_weeks.shape) != expected_shape:
                raise ValueError(
                    "ga_weeks must have shape "
                    f"{expected_shape}, but received "
                    f"{tuple(ga_weeks.shape)}."
                )

            ga_weeks = ga_weeks.to(
                device=e4.device,
                dtype=e4.dtype,
            )

            e4 = self.film(e4, ga_weeks)

        # ── Decoder stage 1 ───────────────────────────────────
        if self.cfg.use_attention_gates:
            e3_skip = self.ag1(e4, e3)
        else:
            e3_skip = e3

        d1 = self.decoder1(e4, e3_skip)

        if self.cfg.use_msam:
            d1 = self.msam_d1(d1)

        # ── Decoder stage 2 ───────────────────────────────────
        if self.cfg.use_attention_gates:
            e2_skip = self.ag2(d1, e2)
        else:
            e2_skip = e2

        d2 = self.decoder2(d1, e2_skip)

        if self.cfg.use_msam:
            d2 = self.msam_d2(d2)

        # ── ABRB ──────────────────────────────────────────────
        if self.cfg.use_abrb:
            d2, edge_logits = self.abrb_d2(d2)
        else:
            edge_logits = d2.new_zeros(
                (
                    d2.shape[0],
                    1,
                    d2.shape[2],
                    d2.shape[3],
                )
            )

        # ── Decoder stage 3 ───────────────────────────────────
        if self.cfg.use_attention_gates:
            e1_skip = self.ag3(d2, e1)
        else:
            e1_skip = e1

        d3 = self.decoder3(d2, e1_skip)

        # ── Decoder stage 4 ───────────────────────────────────
        if self.cfg.use_attention_gates:
            e0_skip = self.ag4(d3, e0)
        else:
            e0_skip = e0

        d4 = self.decoder4(d3, e0_skip)

        # ── Main segmentation output ──────────────────────────
        seg_out = F.interpolate(
            self.seg_head(d4),
            size=input_size,
            mode="bilinear",
            align_corners=False,
        )

        # ── Edge output ────────────────────────────────────────
        edge_out = F.interpolate(
            edge_logits,
            size=input_size,
            mode="bilinear",
            align_corners=False,
        )

        # ── Auxiliary outputs ─────────────────────────────────
        if self.cfg.use_deep_sup:
            aux_d3 = F.interpolate(
                self.aux_head_d3(d3),
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )

            aux_d2 = F.interpolate(
                self.aux_head_d2(d2),
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )
        else:
            aux_d3 = torch.zeros_like(seg_out)
            aux_d2 = torch.zeros_like(seg_out)

        return {
            "seg": seg_out,
            "edge": edge_out,
            "aux": [aux_d3, aux_d2],
        }


print("✓ MVSegNetAblation class defined.")

✓ MVSegNetAblation class defined.


In [ ]:
for variant_name, cfg in ABLATION_VARIANTS.items():
    torch.manual_seed(42)

    model = MVSegNetAblation(
        cfg=cfg,
        num_classes=1,
    )

    x = torch.randn(2, 3, 256, 256)

    ga = (
        torch.rand(2, 1)
        if cfg.use_film
        else None
    )

    with torch.no_grad():
        outputs = model(x, ga)

    assert set(outputs.keys()) == {"seg", "edge", "aux"}
    assert outputs["seg"].shape == (2, 1, 256, 256)
    assert outputs["edge"].shape == (2, 1, 256, 256)
    assert len(outputs["aux"]) == 2
    assert outputs["aux"][0].shape == (2, 1, 256, 256)
    assert outputs["aux"][1].shape == (2, 1, 256, 256)

    print(
        f"{variant_name}: passed | "
        f"seg={tuple(outputs['seg'].shape)} | "
        f"edge={tuple(outputs['edge'].shape)}"
    )

V1_baseline: passed | seg=(2, 1, 256, 256) | edge=(2, 1, 256, 256)
V2_+attn_gates: passed | seg=(2, 1, 256, 256) | edge=(2, 1, 256, 256)
V3_+msam: passed | seg=(2, 1, 256, 256) | edge=(2, 1, 256, 256)
V4_+film: passed | seg=(2, 1, 256, 256) | edge=(2, 1, 256, 256)
V5_+abrb: passed | seg=(2, 1, 256, 256) | edge=(2, 1, 256, 256)
V6_+edge_loss: passed | seg=(2, 1, 256, 256) | edge=(2, 1, 256, 256)
V7_full: passed | seg=(2, 1, 256, 256) | edge=(2, 1, 256, 256)


In [ ]:



def get_ablation_criterion(cfg: AblationCfg) -> CombinedLossLite:


    if cfg.use_edge_loss and not cfg.use_abrb:
        raise ValueError(
            "Invalid configuration: edge supervision requires ABRB."
        )

    return CombinedLossLite(
        edge_start_epoch=5,
        boundary_start_epoch=10,

        # Use the original full-model weights for fair comparison.
        edge_weight=0.03 if cfg.use_edge_loss else 0.0,
        aux_weight=0.10 if cfg.use_deep_sup else 0.0,

        # Kept identical for all variants.
        boundary_weight=0.01,
    )


print("✓ get_ablation_criterion() defined.")

print("\nLoss weights per variant:")

for name, cfg in ABLATION_VARIANTS.items():
    criterion = get_ablation_criterion(cfg)

    print(
        f"  {name:20s}  "
        f"edge_weight={criterion.edge_weight:.3f}  "
        f"aux_weight={criterion.aux_weight:.3f}  "
        f"boundary_weight={criterion.boundary_weight:.3f}"
    )

✓ get_ablation_criterion() defined.

Loss weights per variant:
  V1_baseline           edge_weight=0.000  aux_weight=0.000  boundary_weight=0.010
  V2_+attn_gates        edge_weight=0.000  aux_weight=0.000  boundary_weight=0.010
  V3_+msam              edge_weight=0.000  aux_weight=0.000  boundary_weight=0.010
  V4_+film              edge_weight=0.000  aux_weight=0.000  boundary_weight=0.010
  V5_+abrb              edge_weight=0.000  aux_weight=0.000  boundary_weight=0.010
  V6_+edge_loss         edge_weight=0.030  aux_weight=0.000  boundary_weight=0.010
  V7_full               edge_weight=0.030  aux_weight=0.100  boundary_weight=0.010


In [ ]:


import json
import math
import random
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import OneCycleLR


def seed_worker(worker_id):

    worker_seed = torch.initial_seed() % (2**32)

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def train_ablation_run(
    variant_name,
    cfg,
    run_id,
    seed,
    experiment_cfg,
):


    tag = f"ablation_{variant_name}_run{run_id}"

    print(f"\n{'=' * 72}")
    print(f"  {tag} | seed={seed}")
    print(
        f"  attn={cfg.use_attention_gates}  "
        f"msam={cfg.use_msam}  "
        f"film={cfg.use_film}  "
        f"abrb={cfg.use_abrb}  "
        f"edge_loss={cfg.use_edge_loss}  "
        f"deep_sup={cfg.use_deep_sup}"
    )
    print(f"{'=' * 72}")


    set_seed(seed)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print(f"  Device: {device}")

    # ── Directories ──────────────────────────────────────────
    base_dir = Path(experiment_cfg["base_dir"])

    data_dir = base_dir / "data" / "preprocessed"
    models_dir = base_dir / "models" / "ablation"
    results_dir = base_dir / "results" / "ablation"

    models_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    if not data_dir.exists():
        raise FileNotFoundError(
            f"Preprocessed dataset was not found at: {data_dir}"
        )

    checkpoint_path = models_dir / f"{tag}_best.pth"
    history_path = results_dir / f"{tag}_history.json"

    # ── Datasets ─────────────────────────────────────────────
    train_transform, val_transform = get_transforms()

    train_dataset = FetalVMDataset(
        data_dir,
        "train",
        train_transform,
    )

    val_dataset = FetalVMDataset(
        data_dir,
        "val",
        val_transform,
    )

    # Generator controls shuffled sample order.
    generator = torch.Generator()
    generator.manual_seed(seed)

    pin_memory = device.type == "cuda"

    train_loader = DataLoader(
        train_dataset,
        batch_size=experiment_cfg["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=pin_memory,
        worker_init_fn=seed_worker,
        generator=generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=experiment_cfg["batch_size"],
        shuffle=False,
        num_workers=2,
        pin_memory=pin_memory,
        worker_init_fn=seed_worker,
    )

    if len(train_loader) == 0:
        raise ValueError("The training DataLoader contains no batches.")

    if len(val_loader) == 0:
        raise ValueError("The validation DataLoader contains no batches.")

    print(
        f"  Train: {len(train_dataset)} samples | "
        f"Val: {len(val_dataset)} samples"
    )

    # ── Model and criterion ──────────────────────────────────
    model = MVSegNetAblation(
        cfg=cfg,
        num_classes=1,
    ).to(device)

    criterion = get_ablation_criterion(cfg).to(device)

    print(
        f"  Loss weights: "
        f"edge={criterion.edge_weight:.3f}, "
        f"aux={criterion.aux_weight:.3f}, "
        f"boundary={criterion.boundary_weight:.3f}"
    )

    # ── Optimizer ────────────────────────────────────────────
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
    )

    # OneCycleLR must be stepped once after every optimizer step.
    scheduler = OneCycleLR(
        optimizer,
        max_lr=1e-3,
        epochs=experiment_cfg["epochs"],
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        anneal_strategy="cos",
        div_factor=25,
        final_div_factor=1e4,
    )

    scaler = (
        torch.cuda.amp.GradScaler()
        if device.type == "cuda"
        else None
    )

    # Use -inf so the first finite validation Dice is always saved.
    best_dice = float("-inf")
    best_epoch = -1

    history = {
        "train_loss": [],
        "train_dice": [],
        "train_iou": [],
        "val_loss": [],
        "val_dice": [],
        "val_iou": [],
        "lr": [],
    }

    # ── Training loop ────────────────────────────────────────
    for epoch in range(experiment_cfg["epochs"]):

        epoch_number = epoch + 1
        starting_lr = optimizer.param_groups[0]["lr"]

        print(
            f"\n  Epoch {epoch_number}/"
            f"{experiment_cfg['epochs']} "
            f"[lr={starting_lr:.2e}]"
        )

        # IMPORTANT:
        # train_epoch must pass epoch into CombinedLossLite.
        train_stats = train_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            scheduler=scheduler,
            epoch=epoch,
        )

        train_loss = float(train_stats["loss"])
        train_dice = float(train_stats["dice"])
        train_iou = float(train_stats["iou"])


        val_stats = validate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            epoch=epoch,
        )

        if isinstance(val_stats, dict):
            val_loss = float(val_stats["loss"])
            val_dice = float(val_stats["dice"])
            val_iou = float(val_stats["iou"])
        else:
            val_loss = float(val_stats[0])
            val_dice = float(val_stats[1])
            val_iou = float(val_stats[2])

        ending_lr = float(optimizer.param_groups[0]["lr"])

        history["train_loss"].append(train_loss)
        history["train_dice"].append(train_dice)
        history["train_iou"].append(train_iou)

        history["val_loss"].append(val_loss)
        history["val_dice"].append(val_dice)
        history["val_iou"].append(val_iou)

        # This records the LR after all batches in the epoch.
        history["lr"].append(ending_lr)

        print(
            f"  Train  "
            f"loss={train_loss:.4f}  "
            f"dice={train_dice:.4f}  "
            f"iou={train_iou:.4f}"
        )

        print(
            f"  Val    "
            f"loss={val_loss:.4f}  "
            f"dice={val_dice:.4f}  "
            f"iou={val_iou:.4f}"
        )

        if not math.isfinite(val_dice):
            print(
                "  Warning: validation Dice is not finite; "
                "checkpoint was not updated."
            )
            continue

        # ── Save best checkpoint ─────────────────────────────
        if val_dice > best_dice:
            best_dice = val_dice
            best_epoch = epoch

            checkpoint = {
                "epoch": epoch,
                "best_epoch": best_epoch,
                "variant_name": variant_name,
                "cfg": asdict(cfg),
                "run_id": run_id,
                "seed": seed,
                "best_dice": best_dice,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": (
                    scaler.state_dict()
                    if scaler is not None
                    else None
                ),
                "history": history,
            }

            torch.save(checkpoint, checkpoint_path)

            print(
                f"  ✓ Best checkpoint saved "
                f"(dice={best_dice:.4f})"
            )

        with history_path.open("w", encoding="utf-8") as file:
            json.dump(history, file, indent=2)

    if best_epoch < 0:
        raise RuntimeError(
            f"No valid checkpoint was produced for {tag}."
        )

    print(
        f"\n  Run complete. "
        f"Best validation Dice={best_dice:.4f} "
        f"at epoch {best_epoch + 1}."
    )

    return float(best_dice)


print("✓ train_ablation_run() defined.")

✓ train_ablation_run() defined.


In [ ]:


import gc
import json
from pathlib import Path

import numpy as np
import torch


NUM_RUNS = 3

if isinstance(SEEDS, dict):
    seed_values = list(SEEDS.values())
else:
    seed_values = list(SEEDS)

if len(seed_values) < NUM_RUNS:
    raise ValueError(
        f"At least {NUM_RUNS} seeds are required, "
        f"but only {len(seed_values)} were provided."
    )


# Run every configured variant.
variants_to_run = list(ABLATION_VARIANTS.keys())


unknown_variants = [
    name
    for name in variants_to_run
    if name not in ABLATION_VARIANTS
]

if unknown_variants:
    raise ValueError(
        f"Unknown ablation variants: {unknown_variants}"
    )


base_dir = Path(EXPERIMENT_CFG["base_dir"])
models_dir = base_dir / "models" / "ablation"
results_dir = base_dir / "results" / "ablation"

models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

summary_path = results_dir / "ablation_training_summary.json"

abl_train_results = {}


for variant_index, variant_name in enumerate(
    variants_to_run,
    start=1,
):
    cfg = ABLATION_VARIANTS[variant_name]
    run_dices = {}

    for run_id, seed in enumerate(seed_values[:NUM_RUNS]):
        seed = int(seed)

        tag = f"ablation_{variant_name}_run{run_id}"
        checkpoint_path = models_dir / f"{tag}_best.pth"

        # ── Resume completed runs ─────────────────────────────
        if checkpoint_path.exists():
            try:
                checkpoint = torch.load(
                    checkpoint_path,
                    map_location="cpu",
                )

                if "best_dice" not in checkpoint:
                    raise KeyError(
                        "Checkpoint does not contain 'best_dice'."
                    )

                saved_variant = checkpoint.get("variant_name")
                saved_seed = checkpoint.get("seed")

                if saved_variant != variant_name:
                    raise ValueError(
                        "Checkpoint variant does not match "
                        f"the requested variant: {saved_variant}"
                    )

                if saved_seed != seed:
                    raise ValueError(
                        "Checkpoint seed does not match "
                        f"the requested seed: {saved_seed}"
                    )

                best_dice = float(checkpoint["best_dice"])

                print(
                    f"\n  Run {run_id} | seed={seed}: "
                    f"existing checkpoint found"
                )
                print(
                    f"  Skipping training. "
                    f"Saved best Dice={best_dice:.4f}"
                )

            except Exception as error:
                print(
                    f"\n  Existing checkpoint could not be used: "
                    f"{error}"
                )
                print("  This run will be trained again.")

                best_dice = train_ablation_run(
                    variant_name=variant_name,
                    cfg=cfg,
                    run_id=run_id,
                    seed=seed,
                    experiment_cfg=EXPERIMENT_CFG,
                )

        else:
            best_dice = train_ablation_run(
                variant_name=variant_name,
                cfg=cfg,
                run_id=run_id,
                seed=seed,
                experiment_cfg=EXPERIMENT_CFG,
            )

        run_dices[f"run{run_id}"] = {
            "seed": int(seed),
            "best_dice": float(best_dice),
        }

        # Release references and unused cached GPU memory between runs.
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Variant-level statistics ─────────────────────────────
    dice_values = [
        run_info["best_dice"]
        for run_info in run_dices.values()
    ]

    mean_dice = float(np.mean(dice_values))

    # Sample standard deviation is appropriate when the seeds are
    # treated as repeated experimental samples.
    std_dice = (
        float(np.std(dice_values, ddof=1))
        if len(dice_values) > 1
        else 0.0
    )

    print(f"\n  {'─' * 58}")
    print(f"  {VARIANT_LABELS[variant_name]}")

    for run_name, run_info in run_dices.items():
        print(
            f"    {run_name} "
            f"(seed={run_info['seed']}): "
            f"{run_info['best_dice']:.4f}"
        )

    print(
        f"  Mean ± sample SD: "
        f"{mean_dice:.4f} ± {std_dice:.4f}"
    )

    abl_train_results[variant_name] = {
        "label": VARIANT_LABELS[variant_name],
        "config": {
            "use_attention_gates": cfg.use_attention_gates,
            "use_msam": cfg.use_msam,
            "use_film": cfg.use_film,
            "use_abrb": cfg.use_abrb,
            "use_edge_loss": cfg.use_edge_loss,
            "use_deep_sup": cfg.use_deep_sup,
        },
        "runs": run_dices,
        "mean_best_val_dice": mean_dice,
        "std_best_val_dice": std_dice,
        "num_runs": len(dice_values),
    }

    # Save after every variant so completed results survive interruption.
    with summary_path.open("w", encoding="utf-8") as file:
        json.dump(
            abl_train_results,
            file,
            indent=2,
        )


print("\n✓ All requested ablation training runs are complete.")
print(f"✓ Summary saved to: {summary_path}")


  ablation_V1_baseline_run0 | seed=42
  attn=False  msam=False  film=False  abrb=False  edge_loss=False  deep_sup=False
  Device: cuda
[train] Loaded 235 image-mask pairs
[val] Loaded 57 image-mask pairs
  Train: 235 samples | Val: 57 samples
  Loss weights: edge=0.000, aux=0.000, boundary=0.010

  Epoch 1/50 [lr=4.00e-05]


  Train  loss=0.7946  dice=0.0161  iou=0.0081
  Val    loss=0.7807  dice=0.0168  iou=0.0085
  ✓ Best checkpoint saved (dice=0.0168)

  Epoch 2/50 [lr=1.33e-04]


  Train  loss=0.7558  dice=0.0161  iou=0.0082
  Val    loss=0.7042  dice=0.0102  iou=0.0053

  Epoch 3/50 [lr=3.76e-04]


  Train  loss=0.6985  dice=0.0036  iou=0.0018
  Val    loss=0.6419  dice=0.0000  iou=0.0000

  Epoch 4/50 [lr=6.74e-04]


  Train  loss=0.6414  dice=0.0846  iou=0.0533
  Val    loss=0.6084  dice=0.3368  iou=0.2230
  ✓ Best checkpoint saved (dice=0.3368)

  Epoch 5/50 [lr=9.13e-04]


...

  Train  loss=0.1081  dice=0.8293  iou=0.7199
  Val    loss=0.1217  dice=0.8038  iou=0.6863

  Epoch 47/50 [lr=1.91e-05]


  Train  loss=0.1103  dice=0.8244  iou=0.7117
  Val    loss=0.1216  dice=0.8033  iou=0.6856

  Epoch 48/50 [lr=1.07e-05]


  Train  loss=0.1117  dice=0.8223  iou=0.7115
  Val    loss=0.1215  dice=0.8036  iou=0.6862

  Epoch 49/50 [lr=4.71e-06]


  Train  loss=0.1108  dice=0.8243  iou=0.7138
  Val    loss=0.1214  dice=0.8045  iou=0.6872

  Epoch 50/50 [lr=1.14e-06]


  Train  loss=0.1073  dice=0.8298  iou=0.7189
  Val    loss=0.1219  dice=0.8030  iou=0.6852

  Run complete. Best validation Dice=0.8077 at epoch 28.

  ──────────────────────────────────────────────────────────
  V7 — Full MVSegNet (+ Deep Supervision)
    run0 (seed=42): 0.7719
    run1 (seed=43): 0.7988
    run2 (seed=44): 0.8077
  Mean ± sample SD: 0.7928 ± 0.0187

✓ All requested ablation training runs are complete.
✓ Summary saved to: fetal_vm_segmentation/results/ablation/ablation_training_summary.json


In [ ]:
@torch.no_grad()
def evaluate_ablation_checkpoint(
    model: nn.Module,
    loader: DataLoader,
    device,
    px_mm: float,
    threshold: float = 0.5,
    max_visualization_samples: int = 12,
) -> dict:
    """
    Evaluate one ablation checkpoint using image, mask, and GA values.
    """

    model.eval()

    all_metrics = []
    inference_times = []
    visualization_samples = []

    for batch in tqdm(
        loader,
        desc="    Evaluating",
        leave=False,
    ):
        images, masks, ga_weeks = prepare_batch(
            batch,
            device,
        )

        if masks.ndim == 3:
            masks = masks.unsqueeze(1)

        # Explicit check for FiLM variants.
        if (
            hasattr(model, "cfg")
            and model.cfg.use_film
            and ga_weeks is None
        ):
            raise ValueError(
                "GA values are missing from the test batch, "
                "but FiLM is enabled."
            )

        if ga_weeks is not None:
            ga_weeks = ga_weeks.to(
                device=device,
                dtype=images.dtype,
                non_blocking=True,
            )

            if ga_weeks.ndim == 1:
                ga_weeks = ga_weeks.unsqueeze(1)

        if device.type == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        outputs = model(
            images,
            ga_weeks,
        )

        seg_logits = outputs["seg"]

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start_time
        seconds_per_image = elapsed / images.size(0)

        inference_times.extend(
            [seconds_per_image] * images.size(0)
        )

        seg_probabilities = (
            torch.sigmoid(seg_logits)
            .detach()
            .cpu()
            .numpy()
        )

        masks_np = (
            masks.detach()
            .cpu()
            .numpy()
        )

        images_cpu = (
            images.detach()
            .cpu()
        )

        for sample_index in range(images.size(0)):
            sample_metrics = compute_all_seg_metrics(
                              seg_probabilities[sample_index, 0],
                              masks_np[sample_index, 0],
                              px_mm,
                              threshold,
                          )

            all_metrics.append(sample_metrics)

            if (
                len(visualization_samples)
                < max_visualization_samples
            ):
                visualization_samples.append({
                    "image": images_cpu[sample_index],
                    "mask_gt": masks_np[
                        sample_index,
                        0,
                    ],
                    "mask_pred": (
                        seg_probabilities[
                            sample_index,
                            0,
                        ] > threshold
                    ).astype(np.float32),
                })

    def nanmean_std(metric_name):
        values = np.asarray(
            [
                metric[metric_name]
                for metric in all_metrics
            ],
            dtype=np.float64,
        )

        valid_values = values[
            np.isfinite(values)
        ]

        if valid_values.size == 0:
            return float("nan"), float("nan")

        return (
            float(valid_values.mean()),
            float(valid_values.std()),
        )

    segmentation_summary = {}

    metric_names = [
        "dice",
        "iou",
        "precision",
        "recall",
        "hd95",
        "asd",
        "width_ae",
    ]

    for metric_name in metric_names:
        mean_value, std_value = nanmean_std(
            metric_name
        )

        segmentation_summary[
            f"{metric_name}_mean"
        ] = mean_value

        segmentation_summary[
            f"{metric_name}_std"
        ] = std_value

    if inference_times:
        mean_ms = (
            float(np.mean(inference_times))
            * 1000.0
        )

        std_ms = (
            float(np.std(inference_times))
            * 1000.0
        )
    else:
        mean_ms = float("nan")
        std_ms = float("nan")

    fps = (
        1000.0 / mean_ms
        if mean_ms > 0
        else 0.0
    )

    return {
        "seg": segmentation_summary,
        "timing": {
            "mean_ms": mean_ms,
            "std_ms": std_ms,
            "fps": fps,
        },
        "samples": visualization_samples,
        "per_sample": all_metrics,
        "threshold": float(threshold),
        "px_mm": float(px_mm),
    }

In [ ]:


import gc
from dataclasses import asdict
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader


RUN_IDS = [0, 1, 2]


class AblationFlopsWrapper(nn.Module):

    def __init__(
        self,
        model: nn.Module,
        use_film: bool,
    ):
        super().__init__()

        self.model = model
        self.use_film = use_film

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        ga_weeks = None

        if self.use_film:
            ga_weeks = x.new_zeros(
                (x.shape[0], 1)
            )

        outputs = self.model(
            x,
            ga_weeks,
        )

        return outputs["seg"]


def evaluate_ablation_variant(
    variant_name,
    cfg,
    loader,
    models_dir,
    device,
    px_mm,
    input_size=(1, 3, 512, 512),
):


    print(
        f"\n  ── "
        f"{VARIANT_LABELS.get(variant_name, variant_name)} "
        f"──"
    )

    run_results = []
    last_model = None

    expected_cfg = asdict(cfg)

    for run_id in RUN_IDS:
        tag = f"ablation_{variant_name}_run{run_id}"
        checkpoint_path = (
            models_dir / f"{tag}_best.pth"
        )

        if not checkpoint_path.exists():
            print(
                f"    [run {run_id}] checkpoint not found "
                f"→ skipping"
            )
            continue

        print(
            f"    [run {run_id}] "
            f"loading {checkpoint_path.name}"
        )

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
        )

        # Ensure the checkpoint belongs to the expected experiment.
        saved_variant = checkpoint.get("variant_name")

        if (
            saved_variant is not None
            and saved_variant != variant_name
        ):
            raise ValueError(
                f"Checkpoint variant mismatch for {tag}: "
                f"saved={saved_variant}, "
                f"expected={variant_name}"
            )

        saved_cfg = checkpoint.get("cfg")

        if (
            saved_cfg is not None
            and saved_cfg != expected_cfg
        ):
            raise ValueError(
                f"Checkpoint configuration mismatch for {tag}.\n"
                f"Saved:    {saved_cfg}\n"
                f"Expected: {expected_cfg}"
            )

        model = MVSegNetAblation(
            cfg=cfg,
            num_classes=1,
        )

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        model = model.to(device).float()
        model.eval()

        # evaluate_checkpoint must call:
        #   images, masks, ga_weeks = prepare_batch(batch, device)
        #   outputs = model(images, ga_weeks)
        result = evaluate_ablation_checkpoint(
            model=model,
            loader=loader,
            device=device,
            px_mm=px_mm,
        )

        result["epoch"] = int(
            checkpoint.get("epoch", -1)
        )

        result["seed"] = int(
            checkpoint.get("seed", -1)
        )

        run_results.append(result)
        last_model = model

        segmentation = result["seg"]

        print(
            f"    [run {run_id}]  "
            f"dice={segmentation['dice_mean']:.4f}  "
            f"iou={segmentation['iou_mean']:.4f}  "
            f"hd95={segmentation['hd95_mean']:.2f}mm  "
            f"asd={segmentation['asd_mean']:.2f}mm"
        )

    if not run_results:
        print("    No checkpoints found.")
        return None

    aggregate = aggregate_runs(
        run_results
    )

    # ── Complexity metrics ───────────────────────────────────
    print("    Computing GFLOPs …")

    flops_model = AblationFlopsWrapper(
        model=last_model,
        use_film=cfg.use_film,
    ).to(device)

    flops_model.eval()

    gflops = compute_gflops(
        flops_model,
        input_size=input_size,
        device=str(device),
    )

    # Count parameters on the original model, not the wrapper.
    aggregate["gflops"] = float(gflops)
    aggregate["params"] = int(
        count_parameters(last_model)
    )
    aggregate["size_mb"] = float(
        model_size_mb(last_model)
    )

    print(
        f"    GFLOPs={aggregate['gflops']:.2f}  "
        f"Params={aggregate['params'] / 1e6:.2f}M  "
        f"Size={aggregate['size_mb']:.1f}MB"
    )

    del flops_model
    del last_model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return aggregate


# ── Paths and device ───────────────────────────────────────────────

BASE = Path(
    EXPERIMENT_CFG["base_dir"]
)

DATA_DIR = (
    BASE / "data" / "preprocessed"
)

MODELS_DIR = (
    BASE / "models" / "ablation"
)

RESULTS_DIR = (
    BASE / "results" / "ablation"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

PX_MM = 0.5

_, test_transform = get_transforms()

test_dataset = FetalVMDataset(
    DATA_DIR,
    "test",
    test_transform,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=(DEVICE.type == "cuda"),
)

print(
    f"Test set: {len(test_dataset)} samples"
)



sample_batch = next(
    iter(test_loader)
)

sample_images, sample_masks, sample_ga = prepare_batch(
    sample_batch,
    DEVICE,
)

print(
    f"Test batch shapes: "
    f"images={tuple(sample_images.shape)}, "
    f"masks={tuple(sample_masks.shape)}, "
    f"ga={None if sample_ga is None else tuple(sample_ga.shape)}"
)

if any(
    cfg.use_film
    for cfg in ABLATION_VARIANTS.values()
):
    if sample_ga is None:
        raise ValueError(
            "The test dataset does not provide gestational-age values, "
            "but some ablation variants use FiLM."
        )

    if sample_ga.ndim == 1:
        sample_ga = sample_ga.unsqueeze(1)

    if sample_ga.shape[1] != 1:
        raise ValueError(
            "Gestational-age values must have shape (B, 1), "
            f"but received {tuple(sample_ga.shape)}."
        )

del sample_batch
del sample_images
del sample_masks
del sample_ga

if torch.cuda.is_available():
    torch.cuda.empty_cache()




all_abl_results = {}

for variant_name, cfg in ABLATION_VARIANTS.items():
    result = evaluate_ablation_variant(
        variant_name=variant_name,
        cfg=cfg,
        loader=test_loader,
        models_dir=MODELS_DIR,
        device=DEVICE,
        px_mm=PX_MM,
        input_size=(1, 3, 512, 512),
    )

    all_abl_results[variant_name] = result

for variant_name, result in all_abl_results.items():
    if result is None:
        print(
            f"{variant_name}: "
            f"None (missing checkpoint)"
        )
        continue

    segmentation = result["seg"]

    print(f"\n{variant_name}")
    print(
        "  dice_mean = "
        f"{segmentation.get('dice_mean', 'KEY MISSING')}"
    )
    print(
        "  dice_std  = "
        f"{segmentation.get('dice_std', 'KEY MISSING')}"
    )
    print(
        "  iou_mean  = "
        f"{segmentation.get('iou_mean', 'KEY MISSING')}"
    )
    print(
        "  All keys  = "
        f"{list(segmentation.keys())}"
    )


print(
    "\n✓ All ablation evaluation complete."
)

[test] Loaded 55 image-mask pairs
Test set: 55 samples
Test batch shapes: images=(8, 3, 512, 512), masks=(8, 1, 512, 512), ga=(8, 1)

  ── V1 — Baseline (MobileNetV3 U-Net) ──
    [run 0] loading ablation_V1_baseline_run0_best.pth


    [run 0]  dice=0.7977  iou=0.6797  hd95=4.29mm  asd=1.65mm
    [run 1] loading ablation_V1_baseline_run1_best.pth


    [run 1]  dice=0.8160  iou=0.6971  hd95=3.81mm  asd=1.54mm
    [run 2] loading ablation_V1_baseline_run2_best.pth


    [run 2]  dice=0.8036  iou=0.6829  hd95=5.84mm  asd=1.80mm
    Computing GFLOPs …
    GFLOPs=2.52  Params=1.63M  Size=6.2MB

  ── V2 — + Attention Gates ──
    [run 0] loading ablation_V2_+attn_gates_run0_best.pth


    [run 0]  dice=0.8250  iou=0.7088  hd95=3.56mm  asd=1.46mm
    [run 1] loading ablation_V2_+attn_gates_run1_best.pth


...

    [run 1]  dice=0.8110  iou=0.6898  hd95=4.36mm  asd=1.62mm
    [run 2] loading ablation_V6_+edge_loss_run2_best.pth


    [run 2]  dice=0.7981  iou=0.6813  hd95=3.90mm  asd=1.58mm
    Computing GFLOPs …
    GFLOPs=2.76  Params=2.15M  Size=8.2MB

  ── V7 — Full MVSegNet (+ Deep Supervision) ──
    [run 0] loading ablation_V7_full_run0_best.pth


    [run 0]  dice=0.7871  iou=0.6595  hd95=7.73mm  asd=2.26mm
    [run 1] loading ablation_V7_full_run1_best.pth


    [run 1]  dice=0.8007  iou=0.6849  hd95=3.98mm  asd=1.55mm
    [run 2] loading ablation_V7_full_run2_best.pth


    [run 2]  dice=0.8002  iou=0.6832  hd95=4.05mm  asd=1.56mm
    Computing GFLOPs …
    GFLOPs=2.76  Params=2.15M  Size=8.2MB

V1_baseline
  dice_mean = 0.8057552546601404
  dice_std  = 0.007623692896516332
  iou_mean  = 0.6865415670034108
  All keys  = ['dice_mean', 'dice_std', 'iou_mean', 'iou_std', 'precision_mean', 'precision_std', 'recall_mean', 'recall_std', 'hd95_mean', 'hd95_std', 'asd_mean', 'asd_std', 'width_ae_mean', 'width_ae_std']

V2_+attn_gates
  dice_mean = 0.813796549132376
  dice_std  = 0.008631888896996906
  iou_mean  = 0.6944430530071258
  All keys  = ['dice_mean', 'dice_std', 'iou_mean', 'iou_std', 'precision_mean', 'precision_std', 'recall_mean', 'recall_std', 'hd95_mean', 'hd95_std', 'asd_mean', 'asd_std', 'width_ae_mean', 'width_ae_std']

V3_+msam
  dice_mean = 0.8104586787296064
  dice_std  = 0.0025798718763331693
  iou_mean  = 0.6897918804125353
  All keys  = ['dice_mean', 'dice_std', 'iou_mean', 'iou_std', 'precision_mean', 'precision_std', 'recall_mean', 'r

In [ ]:


import pandas as pd
import numpy as np

def fmt(mean, std, pct=False, dec=2):

    scale = 100 if pct else 1
    if np.isnan(mean):
        return 'N/A'
    return f"{mean*scale:.{dec}f} ± {std*scale:.{dec}f}"

def build_ablation_table(all_abl_results):


    rows = []

    for variant_name, cfg in ABLATION_VARIANTS.items():
        label = VARIANT_LABELS.get(
            variant_name,
            variant_name,
        )

        result = all_abl_results.get(variant_name)

        base_row = {
            "Variant": label,
            "Attn Gates": (
                "✓" if cfg.use_attention_gates else "✗"
            ),
            "MSAM": (
                "✓" if cfg.use_msam else "✗"
            ),
            "FiLM": (
                "✓" if cfg.use_film else "✗"
            ),
            "ABRB": (
                "✓" if cfg.use_abrb else "✗"
            ),
            "Edge Loss": (
                "✓" if cfg.use_edge_loss else "✗"
            ),
            "Deep Sup": (
                "✓" if cfg.use_deep_sup else "✗"
            ),
        }

        if result is None:
            rows.append({
                **base_row,
                "Params (M)": "—",
                "GFLOPs": "—",
                "Dice (%)": "— missing —",
                "IoU (%)": "— missing —",
                "HD95 (mm)": "— missing —",
                "ASD (mm)": "— missing —",
            })
            continue

        seg = result["seg"]

        rows.append({
            **base_row,
            "Params (M)": f"{result['params'] / 1e6:.2f}",
            "GFLOPs": f"{result['gflops']:.2f}",
            "Dice (%)": fmt(
                seg.get("dice_mean"),
                seg.get("dice_std"),
                pct=True,
            ),
            "IoU (%)": fmt(
                seg.get("iou_mean"),
                seg.get("iou_std"),
                pct=True,
            ),
            "HD95 (mm)": fmt(
                seg.get("hd95_mean"),
                seg.get("hd95_std"),
            ),
            "ASD (mm)": fmt(
                seg.get("asd_mean"),
                seg.get("asd_std"),
            ),
        })

    return pd.DataFrame(rows)

print("=" * 100)
print("  TABLE 2 — MVSegNet Ablation Study")
print("=" * 100)
abl_df = build_ablation_table(all_abl_results)
print(abl_df.to_string(index=False))

# Save CSV
csv_path = RESULTS_DIR / 'table2_ablation.csv'
abl_df.to_csv(csv_path, index=False)
print(f"\n✓ Saved: {csv_path}")


print("\n" + "=" * 68)
print("  DELTA TABLE — Marginal Dice gain per component")
print("=" * 68)
prev_dice = None
for variant_name in ABLATION_VARIANTS:
    r     = all_abl_results.get(variant_name)
    label = VARIANT_LABELS[variant_name]
    if r is None:
        print(f"  {label:52s}  —  missing")
        continue
    curr = r['seg']['dice_mean'] * 100
    if prev_dice is not None:
        delta = curr - prev_dice
        sign  = '+' if delta >= 0 else ''
        print(f"  {label:52s}  Dice={curr:.2f}%  (Δ {sign}{delta:.2f}%)")
    else:
        print(f"  {label:52s}  Dice={curr:.2f}%  (baseline)")
    prev_dice = curr


serialisable = {}
for vname, r in all_abl_results.items():
    if r is None:
        serialisable[vname] = None
        continue
    serialisable[vname] = {
        'segmentation': r['seg'],
        'gflops'      : r['gflops'],
        'params'      : r['params'],
        'size_mb'     : r['size_mb'],
    }
with open(RESULTS_DIR / 'ablation_results.json', 'w') as f:
    json.dump(serialisable, f, indent=2, default=str)
print(f"\n✓ Full results saved: {RESULTS_DIR / 'ablation_results.json'}")

  TABLE 2 — MVSegNet Ablation Study
                                Variant Attn Gates MSAM FiLM ABRB Edge Loss Deep Sup Params (M) GFLOPs     Dice (%)      IoU (%)   HD95 (mm)    ASD (mm)
      V1 — Baseline (MobileNetV3 U-Net)          ✗    ✗    ✗    ✗         ✗        ✗       1.63   2.52 80.58 ± 0.76 68.65 ± 0.76 4.65 ± 0.87 1.66 ± 0.11
                 V2 — + Attention Gates          ✓    ✗    ✗    ✗         ✗        ✗       1.66   2.55 81.38 ± 0.86 69.44 ± 1.10 4.42 ± 0.69 1.61 ± 0.12
                            V3 — + MSAM          ✓    ✓    ✗    ✗         ✗        ✗       2.15   2.76 81.05 ± 0.26 68.98 ± 0.26 4.98 ± 0.93 1.73 ± 0.16
          V4 — + FiLM (GA Conditioning)          ✓    ✓    ✓    ✗         ✗        ✗       2.15   2.76 79.95 ± 0.37 67.67 ± 0.46 4.96 ± 0.66 1.74 ± 0.09
                            V5 — + ABRB          ✓    ✓    ✓    ✓         ✗        ✗       2.15   2.76 80.93 ± 0.43 68.78 ± 0.57 4.05 ± 0.13 1.60 ± 0.06
                V6 — + Edge Supervision       